# **Initial Steps and Data Preparation**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colors
from statsmodels.stats.outliers_influence import variance_inflation_factor

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')

from google.colab import drive
drive.mount('')

In [ ]:
DATA_PATH = ''

df = pd.read_csv(DATA_PATH)

# binder system label (FA-only / GGBFS-only / blend) used for stratified reporting later
df['system'] = np.where((df['FA (kg/m3)'] > 0) & (df['GGBFS (kg/m3)'] > 0), 'blend',
                        np.where(df['FA (kg/m3)'] > 0, 'FA', 'GGBFS'))

print(df.shape[0], 'records,', df.shape[1], 'columns')
print('mixes:', df['Idx_Sample'].nunique(), '| studies:', df['Ref.'].nunique())
print(df.groupby('system').agg(rows=('Idx_Sample', 'size'),
                               mixes=('Idx_Sample', 'nunique'),
                               refs=('Ref.', 'nunique')))
df.head()

# **Feature Construction**

In [ ]:
# ---- oxide masses contributed by the binder (kg/m3) ----
df['Total binder (kg/m3)'] = df['FA (kg/m3)'] + df['GGBFS (kg/m3)']

df['Binder SiO2 (kg/m3)']  = df['Total binder (kg/m3)'] * df['SiO2']  / 100
df['Binder Al2O3 (kg/m3)'] = df['Total binder (kg/m3)'] * df['Al2O3'] / 100
df['Binder CaO (kg/m3)']   = df['Total binder (kg/m3)'] * df['CaO']   / 100
df['Binder MgO (kg/m3)']   = df['Total binder (kg/m3)'] * df['MgO']   / 100
df['Binder Fe2O3 (kg/m3)'] = df['Total binder (kg/m3)'] * df['Fe2O3'] / 100
df['Binder Na2O (kg/m3)']  = df['Total binder (kg/m3)'] * df['Na2O']  / 100

# ---- oxide masses contributed by the activator (kg/m3) ----
# sodium silicate supplies SiO2 and Na2O on a dry basis
# NaOH supplies Na2O only: 2 NaOH -> Na2O + H2O, factor = 61.979 / (2 x 39.997)
NAOH_TO_NA2O = 61.979 / (2 * 39.997)

df['Activator SiO2 (kg/m3)'] = df['SiO2 (Dry)']
df['Activator Na2O (kg/m3)'] = df['Na2O (Dry)'] + df['NaOH (Dry)'] * NAOH_TO_NA2O

# ---- system totals (binder + activator); the activator supplies no Al2O3, CaO, MgO or Fe2O3 ----
df['Total SiO2 (kg/m3)']  = df['Binder SiO2 (kg/m3)'] + df['Activator SiO2 (kg/m3)']
df['Total Na2O (kg/m3)']  = df['Binder Na2O (kg/m3)'] + df['Activator Na2O (kg/m3)']
df['Total Al2O3 (kg/m3)'] = df['Binder Al2O3 (kg/m3)']
df['Total CaO (kg/m3)']   = df['Binder CaO (kg/m3)']
df['Total MgO (kg/m3)']   = df['Binder MgO (kg/m3)']
df['Total Fe2O3 (kg/m3)'] = df['Binder Fe2O3 (kg/m3)']

chemistry_cols = [
    'Total binder (kg/m3)',
    'Activator SiO2 (kg/m3)', 'Activator Na2O (kg/m3)',
    'Total SiO2 (kg/m3)', 'Total Al2O3 (kg/m3)', 'Total CaO (kg/m3)',
    'Total MgO (kg/m3)', 'Total Fe2O3 (kg/m3)', 'Total Na2O (kg/m3)',
]

print('missing values:', int(df[chemistry_cols].isna().sum().sum()))
df[chemistry_cols].head()

In [ ]:
# molar masses (g/mol)
M = {'SiO2': 60.084, 'Al2O3': 101.961, 'CaO': 56.077,
     'MgO': 40.304, 'Fe2O3': 159.687, 'Na2O': 61.979, 'H2O': 18.015}

# system oxide moles (binder + activator)
sio2_mol  = df['Total SiO2 (kg/m3)']  / M['SiO2']
al2o3_mol = df['Total Al2O3 (kg/m3)'] / M['Al2O3']
cao_mol   = df['Total CaO (kg/m3)']   / M['CaO']
mgo_mol   = df['Total MgO (kg/m3)']   / M['MgO']
fe2o3_mol = df['Total Fe2O3 (kg/m3)'] / M['Fe2O3']
na2o_mol  = df['Total Na2O (kg/m3)']  / M['Na2O']

# activator-only moles, for the activator modulus and H2O/Na2O
act_sio2_mol = df['Activator SiO2 (kg/m3)'] / M['SiO2']
act_na2o_mol = df['Activator Na2O (kg/m3)'] / M['Na2O']

water = df['Total water (in solutions + additional) (kg in 1m3 mix)']
water_mol = water / M['H2O']

def safe_ratio(num, den):
    return np.divide(num, den,
                     out=np.full(len(df), np.nan, dtype=float),
                     where=den > 0)

# five molar ratios on the system basis
df['SiO2_Al2O3_molar']  = safe_ratio(sio2_mol, al2o3_mol)
df['Na2O_Al2O3_molar']  = safe_ratio(na2o_mol, al2o3_mol)
df['CaO_SiO2_molar']    = safe_ratio(cao_mol, sio2_mol)
df['MgO_Al2O3_molar']   = safe_ratio(mgo_mol, al2o3_mol)
df['Fe2O3_Al2O3_molar'] = safe_ratio(fe2o3_mol, al2o3_mol)

# activator modulus and mix-design water ratios
df['Ms_activator']       = safe_ratio(act_sio2_mol, act_na2o_mol)
df['H2O_Na2O_molar']     = safe_ratio(water_mol, act_na2o_mol)
df['Water_binder_ratio'] = safe_ratio(water, df['Total binder (kg/m3)'])

ratio_cols = ['SiO2_Al2O3_molar', 'Na2O_Al2O3_molar', 'CaO_SiO2_molar',
              'MgO_Al2O3_molar', 'Fe2O3_Al2O3_molar',
              'Ms_activator', 'H2O_Na2O_molar', 'Water_binder_ratio']

print('missing values per feature:')
print(df[ratio_cols].isna().sum())
df[ratio_cols].describe().T.round(3)

In [ ]:
# model inputs
INPUTS = [
    'Total binder (kg/m3)',
    'Coarse aggregate (kg/m3)',
    'Fine aggregate (kg in 1m3 mix)',
    'Concentration (M) NaOH',
    'Superplasticizer (kg in 1m3 mix)',
    'Water_binder_ratio',
    'Initial curing time (day)',
    'Initial curing temp (C)',
    'Age_days',
    'CaO_SiO2_molar',
    'SiO2_Al2O3_molar',
    'Na2O_Al2O3_molar',
    'Fe2O3_Al2O3_molar',
    'MgO_Al2O3_molar',
    'Ms_activator',
    'H2O_Na2O_molar',
]

TARGET = 'converted CS_Mpa'

X = df[INPUTS].apply(pd.to_numeric, errors='coerce')
y = pd.to_numeric(df[TARGET], errors='coerce')

print(len(INPUTS), 'input columns')
print('missing cells in X:', int(X.isna().sum().sum()))

In [ ]:
summary = pd.concat([X, y.rename('CS (MPa)')], axis=1).describe().T
summary['skew'] = pd.concat([X, y.rename('CS (MPa)')], axis=1).skew()
summary = summary[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'skew']]
summary.round(3)

In [ ]:
vif_X = X.assign(const=1.0)
vif = pd.DataFrame({
    'feature': INPUTS,
    'VIF': [variance_inflation_factor(vif_X.values, i) for i in range(len(INPUTS))]
}).sort_values('VIF', ascending=False).reset_index(drop=True)
vif.round(2)

# **Correlation Matrix**

In [ ]:
from google.colab import files

ID_MAP = {
    'Total binder (kg/m3)': 'Binder',
    'Coarse aggregate (kg/m3)': r'$\mathrm{C}_{\mathrm{Agg}}$',
    'Fine aggregate (kg in 1m3 mix)': r'$\mathrm{F}_{\mathrm{Agg}}$',
    'Concentration (M) NaOH': 'M',
    'Superplasticizer (kg in 1m3 mix)': 'SP',
    'Water_binder_ratio': 'W/B',
    'Initial curing time (day)': r'$\mathrm{C_{time}}$',
    'Initial curing temp (C)': r'$\mathrm{C_{temp}}$',
    'Age_days': 'Age',
    'CaO_SiO2_molar': 'Ca/Si',
    'SiO2_Al2O3_molar': 'Si/Al',
    'Na2O_Al2O3_molar': 'Na/Al',
    'Fe2O3_Al2O3_molar': 'Fe/Al',
    'MgO_Al2O3_molar': 'Mg/Al',
    'Ms_activator': r'$\mathrm{M_s}$',
    'H2O_Na2O_molar': r'$\mathrm{H_2O/Na_2O}$',
    'converted CS_Mpa': 'CS',
}

CMAP = colors.LinearSegmentedColormap.from_list(
    'teal_white_orange',
    [(0.00, '#01665E'), (0.25, '#5AB4AC'), (0.50, '#F7F7F7'),
     (0.75, '#D8B365'), (1.00, '#8C510A')], N=256)

corr_cols = INPUTS + [TARGET]
labels = [ID_MAP[c] for c in corr_cols]
corr = df[corr_cols].apply(pd.to_numeric, errors='coerce').corr().values
n = len(corr_cols)

fig, ax = plt.subplots(figsize=(15, 13))
im = ax.imshow(corr, cmap=CMAP, vmin=-1, vmax=1, aspect='equal',
               interpolation='none')
ax.grid(False)

ax.set_xticks(np.arange(n), labels, rotation=45, ha='right',
              rotation_mode='anchor', fontsize=16)
ax.set_yticks(np.arange(n), labels, fontsize=16)
ax.tick_params(length=0, pad=8)

for s in ['top', 'right']:
    ax.spines[s].set_visible(False)
for s in ['left', 'bottom']:
    ax.spines[s].set(visible=True, linewidth=2.5, color='black')

for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{corr[i, j]:.2f}', ha='center', va='center',
                fontsize=15, color='white' if abs(corr[i, j]) >= 0.55 else '#202020')

cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.035, shrink=0.88)
cbar.set_label('Pearson correlation coefficient, r', fontsize=16, labelpad=14)
cbar.set_ticks(np.linspace(-1, 1, 9))
cbar.ax.tick_params(labelsize=14)

plt.tight_layout()
fig.savefig('Correlation_Matrix.png', dpi=600, bbox_inches='tight', facecolor='white')
plt.show()
files.download('Correlation_Matrix.png')

# **Model Training and Validation**

In [ ]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
N_FOLDS = 5

# ---- outer split: 80% development / 20% held-out test ----
dev_idx, test_idx = train_test_split(
    np.arange(len(X)), test_size=0.20, random_state=SEED, shuffle=True)

X_dev,  y_dev  = X.iloc[dev_idx],  y.iloc[dev_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

# ---- inner CV: 5-fold on the development set, folds fixed so every model sees the same splits ----
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
cv_splits = list(kf.split(X_dev))

# ---- scaler fitted on development data only ----
scaler = StandardScaler().fit(X_dev)
X_dev_scaled  = pd.DataFrame(scaler.transform(X_dev),  columns=X.columns, index=X_dev.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)


def metrics(y_true, y_pred):
    return {'MAE':  mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'R2':   r2_score(y_true, y_pred)}


def evaluate(build_model, X_dev=X_dev, y_dev=y_dev, X_test=X_test, y_test=y_test,
             cv_splits=cv_splits):
    """CV on the development set, then refit on all of it and score the held-out test set."""
    rows = []
    for k, (tr, te) in enumerate(cv_splits, start=1):
        m = build_model()
        m.fit(X_dev.iloc[tr], y_dev.iloc[tr])
        tr_m = metrics(y_dev.iloc[tr], m.predict(X_dev.iloc[tr]))
        te_m = metrics(y_dev.iloc[te], m.predict(X_dev.iloc[te]))
        rows.append({'fold': k,
                     'train_MAE': tr_m['MAE'], 'train_RMSE': tr_m['RMSE'], 'train_R2': tr_m['R2'],
                     'val_MAE':   te_m['MAE'], 'val_RMSE':   te_m['RMSE'], 'val_R2':   te_m['R2']})

    res = pd.DataFrame(rows).set_index('fold')
    res.loc['CV mean'] = res.mean()
    res.loc['CV std']  = res.iloc[:N_FOLDS].std()

    final = build_model()
    final.fit(X_dev, y_dev)
    ho = metrics(y_test, final.predict(X_test))
    res.loc['Holdout'] = [np.nan] * 3 + [ho['MAE'], ho['RMSE'], ho['R2']]

    return res.round(4), final


print('development:', X_dev.shape[0], 'rows')
print('held-out   :', X_test.shape[0], 'rows')
print('fold sizes :', [len(te) for _, te in cv_splits])

# **ML Base Models**

## **Random Forest**

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def report(res, name):
    """Print per-fold, CV summary, and held-out results."""
    print(f"===== {name} =====\n")
    for k, (tr, te) in enumerate(cv_splits, start=1):
        r = res.loc[k]
        print(f"----- Fold {k}/{N_FOLDS} -----")
        print(f"Windows: Train ({len(tr)}, {X.shape[1]}) | Val ({len(te)}, {X.shape[1]})")
        print(f"TRAIN  MAE={r['train_MAE']:.4f} | RMSE={r['train_RMSE']:.4f} | R²={r['train_R2']:.4f}")
        print(f"  VAL  MAE={r['val_MAE']:.4f} | RMSE={r['val_RMSE']:.4f} | R²={r['val_R2']:.4f}\n")

    cv, sd, ho = res.loc['CV mean'], res.loc['CV std'], res.loc['Holdout']
    print(f"{name} — SUMMARY")
    print(f"TRAIN  MAE={cv['train_MAE']:.4f}±{sd['train_MAE']:.4f} | "
          f"RMSE={cv['train_RMSE']:.4f}±{sd['train_RMSE']:.4f} | "
          f"R²={cv['train_R2']:.4f}±{sd['train_R2']:.4f}")
    print(f"  VAL  MAE={cv['val_MAE']:.4f}±{sd['val_MAE']:.4f} | "
          f"RMSE={cv['val_RMSE']:.4f}±{sd['val_RMSE']:.4f} | "
          f"R²={cv['val_R2']:.4f}±{sd['val_R2']:.4f}")
    print(f" TEST  MAE={ho['val_MAE']:.4f} | RMSE={ho['val_RMSE']:.4f} | R²={ho['val_R2']:.4f}"
          f"   ({len(y_test)} rows)")


def build_rf():
    return RandomForestRegressor(random_state=SEED)

res_rf, rf_base = evaluate(build_rf)
report(res_rf, 'Random Forest')

## **XGBoost**

In [ ]:
from xgboost import XGBRegressor

def build_xgb():
    return XGBRegressor(random_state=SEED)

res_xgb, xgb_base = evaluate(build_xgb)
report(res_xgb, 'XGBoost')

## **SVR**

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor

# SVR is scale-sensitive -> standardize inputs and target inside each fold (fit on train only)
def build_svr():
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), SVR(kernel='rbf')),
        transformer=StandardScaler())

res_svr, svr_base = evaluate(build_svr)
report(res_svr, 'SVR')

## **ANN**

In [ ]:
from sklearn.neural_network import MLPRegressor

# ANN is scale-sensitive -> standardize inputs and target inside each fold (fit on train only)
def build_ann():
    net = MLPRegressor(activation='relu', solver='adam',
                       max_iter=2000, random_state=SEED)
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), net),
        transformer=StandardScaler())

res_ann, ann_base = evaluate(build_ann)
report(res_ann, 'ANN')

## **CatBoost**

In [ ]:
!pip install catboost -q

In [ ]:
from catboost import CatBoostRegressor

def build_cat():
    return CatBoostRegressor(random_state=SEED, verbose=0)

res_cat, cat_base = evaluate(build_cat)
report(res_cat, 'CatBoost')

# **Grey Wolf Optimizer**

In [ ]:
def grey_wolf_optimizer(objective, bounds, n_wolves=10, n_iter=30, seed=SEED):
    """Minimize `objective` over `bounds` (list of (low, high) per dimension)."""
    rng = np.random.default_rng(seed)
    bounds = np.asarray(bounds, dtype=float)
    lb, ub = bounds[:, 0], bounds[:, 1]
    dim = len(bounds)

    wolves  = rng.uniform(lb, ub, size=(n_wolves, dim))
    fitness = np.array([objective(w) for w in wolves])

    idx = np.argsort(fitness)
    alpha, beta, delta = wolves[idx[0]].copy(), wolves[idx[1]].copy(), wolves[idx[2]].copy()
    f_alpha, f_beta, f_delta = fitness[idx[0]], fitness[idx[1]], fitness[idx[2]]

    history = [f_alpha]
    for t in range(n_iter):
        a = 2 - 2 * t / n_iter
        for i in range(n_wolves):
            move = np.zeros(dim)
            for leader in (alpha, beta, delta):
                A = 2 * a * rng.random(dim) - a
                C = 2 * rng.random(dim)
                D = np.abs(C * leader - wolves[i])
                move += leader - A * D
            wolves[i] = np.clip(move / 3, lb, ub)

        fitness = np.array([objective(w) for w in wolves])
        for i in range(n_wolves):
            f = fitness[i]
            if f < f_alpha:
                delta, f_delta = beta, f_beta
                beta,  f_beta  = alpha, f_alpha
                alpha, f_alpha = wolves[i].copy(), f
            elif f < f_beta:
                delta, f_delta = beta, f_beta
                beta,  f_beta  = wolves[i].copy(), f
            elif f < f_delta:
                delta, f_delta = wolves[i].copy(), f
        history.append(f_alpha)

    return alpha, f_alpha, history

from sklearn.model_selection import cross_val_score

# sanity check: minimize the sphere function, true optimum = 0
sphere = lambda x: np.sum(x ** 2)
best_x, best_f, hist = grey_wolf_optimizer(sphere, bounds=[(-10, 10)] * 5, n_wolves=10, n_iter=30)
print("best f found:", round(best_f, 6))
print("best x:", np.round(best_x, 4))

## **RF_GWO_tunning**

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# GWO search space for RF
rf_bounds = [(100, 800),    # n_estimators
             (3, 30),       # max_depth
             (0.3, 1.0),    # max_features
             (1, 20),       # min_samples_leaf
             (2, 20)]       # min_samples_split

def rf_cv_error(params):
    model = RandomForestRegressor(
        n_estimators=int(round(params[0])),
        max_depth=int(round(params[1])),
        max_features=params[2],
        min_samples_leaf=int(round(params[3])),
        min_samples_split=int(round(params[4])),
        random_state=SEED, n_jobs=1)
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

rf_best, rf_err, rf_hist = grey_wolf_optimizer(
    rf_cv_error, bounds=rf_bounds, n_wolves=10, n_iter=30)

rf_n_est     = int(round(rf_best[0]))
rf_depth     = int(round(rf_best[1]))
rf_max_feat  = rf_best[2]
rf_min_leaf  = int(round(rf_best[3]))
rf_min_split = int(round(rf_best[4]))

print(f"best RF:  n_estimators = {rf_n_est}   max_depth = {rf_depth}   "
      f"max_features = {rf_max_feat:.3f}   min_samples_leaf = {rf_min_leaf}   "
      f"min_samples_split = {rf_min_split}")
print(f"best CV R2 during search = {1 - rf_err:.4f}")

In [ ]:
def build_rf_tuned():
    return RandomForestRegressor(n_estimators=rf_n_est, max_depth=rf_depth,
                                 max_features=rf_max_feat,
                                 min_samples_leaf=rf_min_leaf,
                                 min_samples_split=rf_min_split,
                                 random_state=SEED)

res_rf_gwo, rf_gwo_final = evaluate(build_rf_tuned)
report(res_rf_gwo, 'Random Forest-GWO')

**Save**

In [ ]:
import os
import joblib
import sklearn
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

rf_full = build_rf_tuned()
rf_full.fit(X, y)

rf_shap_bundle = {
    "model_name": "Random Forest",
    "optimization_method": "Grey Wolf Optimizer",

    # rf_gwo_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. rf_full is refit on all rows.
    "model": rf_gwo_final,
    "model_full_data": rf_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": rf_gwo_final.predict(X_test),
    "y_dev_pred": rf_gwo_final.predict(X_dev),

    # Deterministic background sample for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "n_estimators": rf_n_est,
        "max_depth": rf_depth,
        "max_features": rf_max_feat,
        "min_samples_leaf": rf_min_leaf,
        "min_samples_split": rf_min_split,
        "random_state": SEED,
    },

    # Optimizer information
    "search_bounds": rf_bounds,
    "best_search_position": np.asarray(rf_best),
    "best_search_error": rf_err,
    "best_search_cv_r2": 1 - rf_err,
    "optimization_history": rf_hist,

    # Fold-level performance
    "cv_results": res_rf_gwo.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "saved_at": datetime.now().isoformat(),
}

RF_SAVE_PATH = os.path.join(SAVE_DIR, "Random_Forest_GWO_Bundle.joblib")
joblib.dump(rf_shap_bundle, RF_SAVE_PATH, compress=3)

chk = joblib.load(RF_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("Random Forest bundle saved successfully.")
print("File:", RF_SAVE_PATH)
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_rf_gwo.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(RF_SAVE_PATH) / (1024**2), 2), "MB")

## **XGB_GWO_tunning**

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score

# GWO search space for XGB
xgb_bounds = [(100, 800),     # n_estimators (rounded)
              (2, 12),        # max_depth (rounded)
              (-2.5, -0.5),   # log10 learning_rate ≈ [0.003, 0.316]
              (0.5, 1.0),     # subsample
              (0.5, 1.0),     # colsample_bytree
              (1, 20),        # min_child_weight (rounded)
              (-1.0, 2.0),    # log10 reg_lambda ≈ [0.1, 100]
              (0.0, 5.0)]     # gamma

def xgb_cv_error(params):
    model = XGBRegressor(
        n_estimators=int(round(params[0])),
        max_depth=int(round(params[1])),
        learning_rate=10.0 ** params[2],
        subsample=params[3],
        colsample_bytree=params[4],
        min_child_weight=int(round(params[5])),
        reg_lambda=10.0 ** params[6],
        gamma=params[7],
        random_state=SEED, n_jobs=1)
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

xgb_best, xgb_err, xgb_hist = grey_wolf_optimizer(
    xgb_cv_error, bounds=xgb_bounds, n_wolves=10, n_iter=30)

xgb_n_est      = int(round(xgb_best[0]))
xgb_depth      = int(round(xgb_best[1]))
xgb_lr         = 10.0 ** xgb_best[2]
xgb_subsample  = xgb_best[3]
xgb_colsample  = xgb_best[4]
xgb_min_child  = int(round(xgb_best[5]))
xgb_reg_lambda = 10.0 ** xgb_best[6]
xgb_gamma      = xgb_best[7]

print(f"best XGB:  n_estimators = {xgb_n_est}   max_depth = {xgb_depth}   "
      f"learning_rate = {xgb_lr:.4f}   subsample = {xgb_subsample:.3f}   "
      f"colsample = {xgb_colsample:.3f}   min_child_weight = {xgb_min_child}   "
      f"reg_lambda = {xgb_reg_lambda:.3f}   gamma = {xgb_gamma:.3f}")
print(f"best CV R2 during search = {1 - xgb_err:.4f}")

In [ ]:
def build_xgb_tuned():
    return XGBRegressor(n_estimators=xgb_n_est, max_depth=xgb_depth,
                        learning_rate=xgb_lr, subsample=xgb_subsample,
                        colsample_bytree=xgb_colsample, min_child_weight=xgb_min_child,
                        reg_lambda=xgb_reg_lambda, gamma=xgb_gamma,
                        random_state=SEED)

res_xgb_gwo, xgb_gwo_final = evaluate(build_xgb_tuned)
report(res_xgb_gwo, 'XGBoost-GWO')

**Save**

In [ ]:
import os
import joblib
import sklearn
import xgboost
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

# Model refit on all data, for deployment and as the MOO surrogate.
xgb_full = build_xgb_tuned()
xgb_full.fit(X, y)

xgb_shap_bundle = {
    "model_name": "XGBoost",
    "optimization_method": "Grey Wolf Optimizer",

    # xgb_gwo_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. xgb_full is refit on all rows.
    "model": xgb_gwo_final,
    "model_full_data": xgb_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": xgb_gwo_final.predict(X_test),
    "y_dev_pred": xgb_gwo_final.predict(X_dev),

    # Deterministic background sample for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "n_estimators": xgb_n_est,
        "max_depth": xgb_depth,
        "learning_rate": xgb_lr,
        "subsample": xgb_subsample,
        "colsample_bytree": xgb_colsample,
        "min_child_weight": xgb_min_child,
        "reg_lambda": xgb_reg_lambda,
        "gamma": xgb_gamma,
        "random_state": SEED,
    },

    # Optimizer information
    "search_bounds": xgb_bounds,
    "best_search_position": np.asarray(xgb_best),
    "best_search_error": xgb_err,
    "best_search_cv_r2": 1 - xgb_err,
    "optimization_history": xgb_hist,

    # Fold-level performance
    "cv_results": res_xgb_gwo.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "xgboost_version": xgboost.__version__,
    "saved_at": datetime.now().isoformat(),
}

XGB_SAVE_PATH = os.path.join(SAVE_DIR, "XGBoost_GWO_Bundle.joblib")
joblib.dump(xgb_shap_bundle, XGB_SAVE_PATH, compress=3)

# Verify the file loads and reproduces the reported held-out score.
chk = joblib.load(XGB_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("XGBoost bundle saved successfully.")
print("File:", XGB_SAVE_PATH)
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_xgb_gwo.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(XGB_SAVE_PATH) / (1024**2), 2), "MB")

## **SVR_GWO_tunning**

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_val_score

# GWO search space for SVR (log10 scale for C, gamma, epsilon)
svr_bounds = [(-2, 4),     # log10 C       -> [0.01, 10000]
              (-5, 1),     # log10 gamma   -> [1e-5, 10]
              (-3, 0)]     # log10 epsilon -> [0.001, 1]

def svr_cv_error(params):
    model = TransformedTargetRegressor(
        regressor=make_pipeline(
            StandardScaler(),
            SVR(kernel='rbf',
                C=10.0 ** params[0],
                gamma=10.0 ** params[1],
                epsilon=10.0 ** params[2])),
        transformer=StandardScaler())
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

svr_best, svr_err, svr_hist = grey_wolf_optimizer(
    svr_cv_error, bounds=svr_bounds, n_wolves=10, n_iter=30)

svr_C       = 10.0 ** svr_best[0]
svr_gamma   = 10.0 ** svr_best[1]
svr_epsilon = 10.0 ** svr_best[2]

print(f"best SVR:  C = {svr_C:.3f}   gamma = {svr_gamma:.4f}   epsilon = {svr_epsilon:.4f}")
print(f"best CV R2 during search = {1 - svr_err:.4f}")

In [ ]:
def build_svr_tuned():
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(),
                                SVR(kernel='rbf', C=svr_C, gamma=svr_gamma,
                                    epsilon=svr_epsilon)),
        transformer=StandardScaler())

res_svr_gwo, svr_gwo_final = evaluate(build_svr_tuned)
report(res_svr_gwo, 'SVR-GWO')

**Save**

In [ ]:
import os
import joblib
import sklearn
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

svr_full = build_svr_tuned()
svr_full.fit(X, y)

svr_shap_bundle = {
    "model_name": "SVR",
    "optimization_method": "Grey Wolf Optimizer",

    # svr_gwo_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. svr_full is refit on all rows.
    "model": svr_gwo_final,
    "model_full_data": svr_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": svr_gwo_final.predict(X_test),
    "y_dev_pred": svr_gwo_final.predict(X_dev),

    # Deterministic background sample, required for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "C": svr_C,
        "gamma": svr_gamma,
        "epsilon": svr_epsilon,
        "kernel": "rbf",
    },

    # Optimizer information
    "search_bounds": svr_bounds,
    "best_search_position": np.asarray(svr_best),
    "best_search_error": svr_err,
    "best_search_cv_r2": 1 - svr_err,
    "optimization_history": svr_hist,

    # Fold-level performance
    "cv_results": res_svr_gwo.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "saved_at": datetime.now().isoformat(),
}

SVR_SAVE_PATH = os.path.join(SAVE_DIR, "SVR_GWO_Bundle.joblib")
joblib.dump(svr_shap_bundle, SVR_SAVE_PATH, compress=3)

chk = joblib.load(SVR_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("SVR bundle saved successfully.")
print("File:", SVR_SAVE_PATH)
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Background records:", chk["X_background"].shape[0])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_svr_gwo.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(SVR_SAVE_PATH) / (1024**2), 2), "MB")

## **ANN_GWO_tunning**

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import cross_val_score

ann_bounds = [(16, 160),     # first hidden width
              (8, 96),       # second hidden width
              (-4, 0.5),     # log10 alpha -> ~[1e-4, 3]
              (-3.5, -1.2)]  # log10 learning_rate_init

def ann_cv_error(params):
    net = MLPRegressor(
        hidden_layer_sizes=(int(round(params[0])), int(round(params[1]))),
        activation='relu', solver='adam',
        alpha=10.0 ** params[2],
        learning_rate_init=10.0 ** params[3],
        max_iter=2000, random_state=SEED)
    model = TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), net),
        transformer=StandardScaler())
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring='r2', n_jobs=-1).mean()

ann_best, ann_err, ann_hist = grey_wolf_optimizer(
    ann_cv_error, bounds=ann_bounds, n_wolves=10, n_iter=30)

ann_l1, ann_l2 = int(round(ann_best[0])), int(round(ann_best[1]))
ann_alpha = 10.0 ** ann_best[2]
ann_lr    = 10.0 ** ann_best[3]

print(f'best ANN:  layers = ({ann_l1}, {ann_l2})   alpha = {ann_alpha:.5f}   lr = {ann_lr:.5f}')
print(f'best CV R2 during search = {1 - ann_err:.4f}')

In [ ]:
def build_ann_tuned():
    net = MLPRegressor(hidden_layer_sizes=(ann_l1, ann_l2), activation='relu',
                       solver='adam', alpha=ann_alpha, learning_rate_init=ann_lr,
                       max_iter=2000, random_state=SEED)
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), net),
        transformer=StandardScaler())

res_ann_gwo, ann_gwo_final = evaluate(build_ann_tuned)
report(res_ann_gwo, 'ANN-GWO')

**Save**

In [ ]:
import os
import joblib
import sklearn
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

ann_full = build_ann_tuned()
ann_full.fit(X, y)

ann_shap_bundle = {
    "model_name": "ANN",
    "optimization_method": "Grey Wolf Optimizer",

    # ann_gwo_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. ann_full is refit on all rows.
    "model": ann_gwo_final,
    "model_full_data": ann_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": ann_gwo_final.predict(X_test),
    "y_dev_pred": ann_gwo_final.predict(X_dev),

    # Deterministic background sample, required for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "hidden_layer_sizes": (ann_l1, ann_l2),
        "activation": "relu",
        "solver": "adam",
        "alpha": ann_alpha,
        "learning_rate_init": ann_lr,
        "max_iter": 3000,
        "random_state": SEED,
    },

    # Optimizer information
    "search_bounds": ann_bounds,
    "best_search_position": np.asarray(ann_best),
    "best_search_error": ann_err,
    "best_search_cv_r2": 1 - ann_err,
    "optimization_history": ann_hist,

    # Fold-level performance
    "cv_results": res_ann_gwo.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "saved_at": datetime.now().isoformat(),
}

ANN_SAVE_PATH = os.path.join(SAVE_DIR, "ANN_GWO_Bundle.joblib")
joblib.dump(ann_shap_bundle, ANN_SAVE_PATH, compress=3)

chk = joblib.load(ANN_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("ANN bundle saved successfully.")
print("File:", ANN_SAVE_PATH)
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Background records:", chk["X_background"].shape[0])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_ann_gwo.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(ANN_SAVE_PATH) / (1024**2), 2), "MB")

## **CatBoost_GWO_tunning**

In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_val_score

cat_bounds = [(200, 1600),    # iterations
              (3, 8),         # depth
              (-2.5, -0.5),   # log10 learning_rate -> ~[0.003, 0.316]
              (-0.5, 2.5)]    # log10 l2_leaf_reg   -> ~[0.316, 316]

def cat_cv_error(params):
    model = CatBoostRegressor(
        iterations=int(round(params[0])),
        depth=int(round(params[1])),
        learning_rate=float(10.0 ** params[2]),   # float() -> avoids CatBoost clone error
        l2_leaf_reg=float(10.0 ** params[3]),
        random_state=SEED, verbose=0)             # all 12 threads
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2").mean()   # folds sequential

cat_best, cat_err, cat_hist = grey_wolf_optimizer(
    cat_cv_error, bounds=cat_bounds, n_wolves=10, n_iter=30)

cat_iter  = int(round(cat_best[0]))
cat_depth = int(round(cat_best[1]))
cat_lr    = float(10.0 ** cat_best[2])
cat_l2    = float(10.0 ** cat_best[3])

print(f"best CatBoost:  iterations = {cat_iter}   depth = {cat_depth}   "
      f"learning_rate = {cat_lr:.4f}   l2_leaf_reg = {cat_l2:.3f}")
print(f"best CV R2 during search = {1 - cat_err:.4f}")

In [ ]:
def build_cat_tuned():
    return CatBoostRegressor(iterations=cat_iter, depth=cat_depth,
                             learning_rate=cat_lr, l2_leaf_reg=cat_l2,
                             random_state=SEED, verbose=0)

res_cat_gwo, cat_gwo_final = evaluate(build_cat_tuned)
report(res_cat_gwo, 'CatBoost-GWO')

In [ ]:
import os
import joblib
import sklearn
import catboost
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

cat_full = build_cat_tuned()
cat_full.fit(X, y)

cat_shap_bundle = {
    "model_name": "CatBoost",
    "optimization_method": "Grey Wolf Optimizer",

    # cat_gwo_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. cat_full is refit on all rows.
    "model": cat_gwo_final,
    "model_full_data": cat_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": cat_gwo_final.predict(X_test),
    "y_dev_pred": cat_gwo_final.predict(X_dev),

    # Deterministic background sample, available if needed during SHAP analysis
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "iterations": cat_iter,
        "depth": cat_depth,
        "learning_rate": cat_lr,
        "l2_leaf_reg": cat_l2,
        "random_state": SEED,
        "verbose": 0,
    },

    # Optimizer information
    "search_bounds": cat_bounds,
    "best_search_position": np.asarray(cat_best),
    "best_search_error": cat_err,
    "best_search_cv_r2": 1 - cat_err,
    "optimization_history": np.asarray(cat_hist),
    "optimizer_settings": {"n_wolves": 10, "n_iterations": 30, "seed": SEED},

    # Fold-level performance
    "cv_results": res_cat_gwo.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "catboost_version": catboost.__version__,
    "saved_at": datetime.now().isoformat(),
}

CAT_SAVE_PATH = os.path.join(SAVE_DIR, "CatBoost_GWO_Bundle.joblib")
joblib.dump(cat_shap_bundle, CAT_SAVE_PATH, compress=3)

chk = joblib.load(CAT_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("CatBoost-GWO bundle saved successfully.")
print("File:", CAT_SAVE_PATH)
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Background records:", chk["X_background"].shape[0])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_cat_gwo.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(CAT_SAVE_PATH) / (1024**2), 2), "MB")

# **Galactic Field Optimization**

In [ ]:
import math
import numpy as np
from dataclasses import dataclass
from typing import Optional, Tuple

def clamp(X, lb, ub):
    return np.minimum(np.maximum(X, lb), ub)

def safe_norm(X, axis=-1, keepdims=False, eps=1e-12):
    return np.linalg.norm(X, axis=axis, keepdims=keepdims) + eps

def levy_flight(n, d, beta=1.5):
    sigma = (
        math.gamma(1 + beta) * np.sin(np.pi * beta / 2)
        / (math.gamma((1 + beta) / 2) * beta * 2 ** ((beta - 1) / 2))
    ) ** (1 / beta)
    u = sigma * np.random.randn(n, d)
    v = np.random.randn(n, d)
    return u / (np.abs(v) ** (1 / beta) + 1e-12)

@dataclass
class GFOParams:
    n_agents: int = 40
    iters: int = 300
    seed: int = 42

    alpha_mass: float = 2.5
    G0: float = 1.3
    gamma_G: float = 1.8
    kappa0: float = 0.10
    gamma_kappa: float = 1.0
    p: float = 2.0
    q: float = 4.0

    R0_max_frac: float = 0.35
    R0_min_frac: float = 0.05
    lam0: float = 0.02
    lam_max: float = 3.2
    delta0: float = 0.90
    gamma_delta: float = 1.8

    smax_frac: float = 0.25
    beta_step: float = 1.8
    step_noise_sigma: float = 0.08

    stagnation_window: int = 12
    boost_iters: int = 8
    boost_G_factor: float = 1.35
    boost_delta_factor: float = 1.10

    k_neighbors: Optional[int] = None
    jump_prob0: float = 0.10
    jump_phase_cutoff: float = 0.55
    protected_elites: int = 5
    restart_frac: float = 0.10

    local_search_start: float = 0.70
    local_sigma0: float = 0.05
    leader_weights: Tuple[float, float, float] = (0.5, 0.3, 0.2)

    levy_prob0: float = 0.18
    levy_phase_cutoff: float = 0.65
    levy_scale: float = 0.12
    elite_refine_count: int = 3
    refine_trials: int = 2

def gfo_schedule(t, T, scale, P):
    frac = t / max(T - 1, 1)
    return {
        "frac": frac,
        "G": P.G0 * np.exp(-P.gamma_G * frac),
        "kappa": P.kappa0 * np.exp(-P.gamma_kappa * frac),
        "R0": scale * (P.R0_max_frac * (1 - frac) + P.R0_min_frac),
        "lam": P.lam0 + (P.lam_max - P.lam0) * frac**3,
        "delta": P.delta0 * np.exp(-P.gamma_delta * frac),
        "s_base": P.smax_frac * scale * (1 - frac) ** (0.65 * P.beta_step),
        "jump_prob": P.jump_prob0 * (1 - frac) if frac < P.jump_phase_cutoff else 0.0,
        "levy_prob": P.levy_prob0 * (1 - frac) if frac < P.levy_phase_cutoff else 0.0,
        "step_noise": P.step_noise_sigma * (1 - frac),
        "local_sigma": P.local_sigma0 * (1 - frac) ** 1.5,
    }

def fitness_to_mass(f, alpha):
    finite = np.isfinite(f)

    if not np.any(finite):
        return np.full(len(f), 1 / len(f))

    safe_f = f.copy()
    safe_f[~finite] = np.max(safe_f[finite]) + 1.0
    fbest, fworst = np.min(safe_f), np.max(safe_f)

    if fworst - fbest < 1e-12:
        return np.full(len(f), 1 / len(f))

    masses = np.exp(-alpha * (safe_f - fbest) / (fworst - fbest))
    return masses / np.sum(masses)

def pick_neighbors(X, k):
    distances = np.sum((X[:, None, :] - X[None, :, :]) ** 2, axis=-1)
    np.fill_diagonal(distances, np.inf)
    return np.argsort(distances, axis=1)[:, :k]

def leader_center(X, f, weights):
    idx = np.argsort(f)[:3]
    return weights[0] * X[idx[0]] + weights[1] * X[idx[1]] + weights[2] * X[idx[2]]

def coordinate_scale(X, elite_idx):
    scale = np.std(X[elite_idx], axis=0) + 1e-12
    return scale / np.mean(scale)

def compute_field(X, masses, center, schedule, P):
    n, d = X.shape
    field = np.zeros((n, d))

    neighbors = (
        pick_neighbors(X, P.k_neighbors)
        if P.k_neighbors is not None and P.k_neighbors < n
        else None
    )

    if neighbors is None:
        displacement = X[None, :, :] - X[:, None, :]
        distance = safe_norm(displacement, axis=2, keepdims=True)
        distance_scalar = distance[..., 0]
        direction = displacement / distance
        mass_product = masses[:, None] * masses[None, :]

        attraction = (
            schedule["G"] * mass_product / (distance_scalar**P.p + 1e-12)
        )[:, :, None] * direction

        repulsion = np.zeros_like(attraction)
        repulsion_mask = (distance_scalar < schedule["R0"]) & (~np.eye(n, dtype=bool))
        repulsion_factor = (
            -schedule["kappa"] * mass_product / (distance_scalar**P.q + 1e-12)
        )
        repulsion[repulsion_mask] = (
            repulsion_factor[:, :, None] * direction
        )[repulsion_mask]

        field = np.sum(attraction + repulsion, axis=1)

    else:
        for i, neighbor_ids in enumerate(neighbors):
            displacement = X[neighbor_ids] - X[i]
            distance = safe_norm(displacement, axis=1, keepdims=True)
            distance_scalar = distance[:, 0]
            direction = displacement / distance
            mass_product = masses[i] * masses[neighbor_ids]

            attraction = (
                schedule["G"] * mass_product / (distance_scalar**P.p + 1e-12)
            )[:, None] * direction

            repulsion = np.zeros_like(attraction)
            repulsion_mask = distance_scalar < schedule["R0"]
            repulsion[repulsion_mask] = (
                -schedule["kappa"] * mass_product[repulsion_mask]
                / (distance_scalar[repulsion_mask]**P.q + 1e-12)
            )[:, None] * direction[repulsion_mask]

            field[i] = np.sum(attraction + repulsion, axis=0)

    return field + schedule["lam"] * (center - X)

def orbital_update(X, field, schedule, lb, ub, coord_scale):
    n, d = X.shape

    field_direction = field / safe_norm(field, axis=1, keepdims=True)
    random_direction = np.random.randn(n, d)
    random_direction /= safe_norm(random_direction, axis=1, keepdims=True)

    direction = (
        (1 - schedule["delta"]) * field_direction
        + schedule["delta"] * random_direction
    )
    direction /= safe_norm(direction, axis=1, keepdims=True)

    step = schedule["s_base"] * (
        1 + schedule["step_noise"] * np.random.randn(n)
    )
    X_new = X + np.clip(step, 0, None)[:, None] * direction * coord_scale

    return clamp(X_new, lb, ub)

class GFO:
    def __init__(self, func, lb, ub, params=None, verbose=False):
        self.func = func
        self.lb = np.asarray(lb, dtype=float)
        self.ub = np.asarray(ub, dtype=float)
        self.p = GFOParams() if params is None else params
        self.verbose = verbose

        if self.lb.shape != self.ub.shape:
            raise ValueError("Lower and upper bounds must have identical shapes.")

        self.d = len(self.lb)
        self.scale = float(np.mean(self.ub - self.lb))

    def fit(self):
        P, n, T = self.p, self.p.n_agents, self.p.iters
        np.random.seed(P.seed)

        X = self.lb + (self.ub - self.lb) * np.random.rand(n, self.d)
        fitness = np.array([self.func(x) for x in X], dtype=float)

        best_idx = int(np.argmin(fitness))
        best_x = X[best_idx].copy()
        best_f = float(fitness[best_idx])

        history = [best_f]
        stagnation = 0
        boost_left = 0

        for t in range(T):
            schedule = gfo_schedule(t, T, self.scale, P)

            if boost_left > 0:
                schedule["G"] *= P.boost_G_factor
                schedule["delta"] = min(
                    1.0, schedule["delta"] * P.boost_delta_factor
                )
                boost_left -= 1

            elite_ids = np.argsort(fitness)[:min(P.protected_elites, n)]
            coord_scale = coordinate_scale(X, elite_ids)
            center = leader_center(X, fitness, P.leader_weights)
            masses = fitness_to_mass(fitness, P.alpha_mass)
            field = compute_field(X, masses, center, schedule, P)

            X_new = orbital_update(
                X, field, schedule, self.lb, self.ub, coord_scale
            )

            # Elite encircling
            A = (
                2 * (1 - schedule["frac"]) * np.random.rand(n, self.d)
                - (1 - schedule["frac"])
            )
            C = 2 * np.random.rand(n, self.d)
            X_encircle = best_x - A * np.abs(C * best_x - X)

            if np.random.rand() >= 1 - schedule["frac"]**0.5:
                X_new = X_encircle

            # Late-stage exploitation
            if schedule["frac"] > 0.60:
                sigma = 0.20 * (1 - schedule["frac"]**0.5)
                mask = np.random.rand(n) < 0.50
                X_exploit = (
                    X + sigma * (best_x - X)
                    + sigma * (self.ub - self.lb) * np.random.randn(n, self.d)
                )
                X_new[mask] = X_exploit[mask]

            # Early search around the best agent
            if schedule["frac"] < 0.25:
                mask = np.random.rand(n) < 0.70
                X_strong = (
                    best_x
                    + 0.30 * (self.ub - self.lb) * np.random.randn(n, self.d)
                )
                X_new[mask] = X_strong[mask]

            # Random jumps
            jump_mask = np.random.rand(n) < schedule["jump_prob"]
            jump_mask[elite_ids] = False

            if np.any(jump_mask):
                count = int(np.sum(jump_mask))
                X_new[jump_mask] = (
                    self.lb
                    + (self.ub - self.lb) * np.random.rand(count, self.d)
                )

            # Levy-flight diversification
            levy_mask = np.random.rand(n) < schedule["levy_prob"]
            levy_mask[elite_ids] = False

            if np.any(levy_mask):
                count = int(np.sum(levy_mask))
                levy_scale = 8.0 * P.levy_scale * (1 + schedule["frac"])
                X_new[levy_mask] = (
                    center
                    + levy_scale * levy_flight(count, self.d)
                    * (self.ub - self.lb)
                )

            # Greedy replacement
            X_new = clamp(X_new, self.lb, self.ub)
            new_fitness = np.array([self.func(x) for x in X_new], dtype=float)
            improved = new_fitness < fitness

            X[improved] = X_new[improved]
            fitness[improved] = new_fitness[improved]

            current_idx = int(np.argmin(fitness))
            current_f = float(fitness[current_idx])

            if current_f < best_f - 1e-12:
                best_x = X[current_idx].copy()
                best_f = current_f
                stagnation = 0
            else:
                stagnation += 1

            # Local elite refinement
            if schedule["frac"] > P.local_search_start:
                refine_ids = np.argsort(fitness)[:min(P.elite_refine_count, n)]

                for idx in refine_ids:
                    local_x = X[idx].copy()
                    local_f = float(fitness[idx])

                    for _ in range(P.refine_trials):
                        candidate = (
                            local_x
                            + schedule["local_sigma"] * coord_scale
                            * (self.ub - self.lb) * np.random.randn(self.d)
                        )
                        candidate = clamp(candidate, self.lb, self.ub)
                        candidate_f = float(self.func(candidate))

                        if candidate_f < local_f:
                            local_x, local_f = candidate, candidate_f

                    X[idx], fitness[idx] = local_x, local_f

                    if local_f < best_f:
                        best_x, best_f = local_x.copy(), local_f
                        stagnation = 0

            # Adaptive restart after stagnation
            if stagnation >= P.stagnation_window:
                stagnation, boost_left = 0, P.boost_iters
                restart_frac = min(
                    0.50, P.restart_frac + 0.30 * schedule["frac"]
                )
                count = max(1, int(restart_frac * n))
                worst_ids = np.argsort(fitness)[-count:]

                X[worst_ids] = (
                    self.lb
                    + (self.ub - self.lb) * np.random.rand(count, self.d)
                )
                fitness[worst_ids] = np.array(
                    [self.func(x) for x in X[worst_ids]], dtype=float
                )

                current_idx = int(np.argmin(fitness))
                if float(fitness[current_idx]) < best_f:
                    best_x = X[current_idx].copy()
                    best_f = float(fitness[current_idx])

            history.append(best_f)

            if self.verbose and (t + 1) % max(1, T // 10) == 0:
                print(
                    f"Iteration {t + 1:4d}/{T} | "
                    f"Best objective = {best_f:.6g}"
                )

        return best_x, best_f, {"history_best": np.asarray(history)}

def galactic_field_optimizer(objective, bounds, n_agents=10, n_iter=30,
                             seed=None, verbose=False):
    bounds = np.asarray(bounds, dtype=float)

    if bounds.ndim != 2 or bounds.shape[1] != 2:
        raise ValueError("bounds must contain (lower, upper) pairs.")
    if np.any(~np.isfinite(bounds)) or np.any(bounds[:, 0] >= bounds[:, 1]):
        raise ValueError("Every bound must be finite and satisfy lower < upper.")
    if n_agents < 5 or n_iter < 1:
        raise ValueError("n_agents must be >= 5 and n_iter must be >= 1.")

    seed = int(globals().get("SEED", 42) if seed is None else seed)

    def checked_objective(x):
        value = float(objective(np.asarray(x, dtype=float)))
        return value if np.isfinite(value) else np.inf

    params = GFOParams(
        n_agents=int(n_agents),
        iters=int(n_iter),
        seed=seed
    )

    best_x, best_f, info = GFO(
        checked_objective,
        bounds[:, 0],
        bounds[:, 1],
        params=params,
        verbose=verbose
    ).fit()

    return best_x, best_f, info["history_best"]

# Automatic reproducibility and validity check
def _check_gfo():
    random_state = np.random.get_state()

    try:
        sphere = lambda x: float(np.sum(np.asarray(x)**2))
        test_bounds = [(-5, 5)] * 4

        x1, f1, h1 = galactic_field_optimizer(
            sphere, test_bounds, n_agents=8, n_iter=10, seed=123
        )
        x2, f2, h2 = galactic_field_optimizer(
            sphere, test_bounds, n_agents=8, n_iter=10, seed=123
        )

        assert np.allclose(x1, x2)
        assert np.isclose(f1, f2)
        assert np.allclose(h1, h2)
        assert len(h1) == 11
        assert np.all(np.diff(h1) <= 1e-12)
        assert np.all((x1 >= -5) & (x1 <= 5))
        assert np.isfinite(f1)

    finally:
        np.random.set_state(random_state)

    print("GFO implementation check passed.")

_check_gfo()

## **RF_GFO_tunning**

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to RF-GWO
rf_gfo_bounds = [(100, 800),    # n_estimators
                 (3, 30),       # max_depth
                 (0.3, 1.0),    # max_features
                 (1, 20),       # min_samples_leaf
                 (2, 20)]       # min_samples_split

def rf_gfo_cv_error(params):
    model = RandomForestRegressor(
        n_estimators=int(round(params[0])),
        max_depth=int(round(params[1])),
        max_features=float(params[2]),
        min_samples_leaf=int(round(params[3])),
        min_samples_split=int(round(params[4])),
        random_state=SEED, n_jobs=1)
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

rf_gfo_best, rf_gfo_err, rf_gfo_hist = galactic_field_optimizer(
    rf_gfo_cv_error,
    bounds=rf_gfo_bounds,
    n_agents=10,
    n_iter=30,
    seed=SEED
)

rf_gfo_n_est     = int(round(rf_gfo_best[0]))
rf_gfo_depth     = int(round(rf_gfo_best[1]))
rf_gfo_max_feat  = float(rf_gfo_best[2])
rf_gfo_min_leaf  = int(round(rf_gfo_best[3]))
rf_gfo_min_split = int(round(rf_gfo_best[4]))

print(f"best RF-GFO:  n_estimators = {rf_gfo_n_est}   max_depth = {rf_gfo_depth}   "
      f"max_features = {rf_gfo_max_feat:.3f}   min_samples_leaf = {rf_gfo_min_leaf}   "
      f"min_samples_split = {rf_gfo_min_split}")
print(f"best CV R2 during search = {1 - rf_gfo_err:.4f}")

In [ ]:
def build_rf_gfo_tuned():
    return RandomForestRegressor(
        n_estimators=rf_gfo_n_est,
        max_depth=rf_gfo_depth,
        max_features=rf_gfo_max_feat,
        min_samples_leaf=rf_gfo_min_leaf,
        min_samples_split=rf_gfo_min_split,
        random_state=SEED)

res_rf_gfo, rf_gfo_final = evaluate(build_rf_gfo_tuned)
report(res_rf_gfo, 'RF-GFO')

In [ ]:
import os
import joblib
import sklearn
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

rf_gfo_full = build_rf_gfo_tuned()
rf_gfo_full.fit(X, y)

rf_gfo_shap_bundle = {
    "model_name": "Random Forest",
    "optimization_method": "Galactic Field Optimization",

    # rf_gfo_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. rf_gfo_full is refit on all rows.
    "model": rf_gfo_final,
    "model_full_data": rf_gfo_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": rf_gfo_final.predict(X_test),
    "y_dev_pred": rf_gfo_final.predict(X_dev),

    # Deterministic background sample for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "n_estimators": rf_gfo_n_est,
        "max_depth": rf_gfo_depth,
        "max_features": rf_gfo_max_feat,
        "min_samples_leaf": rf_gfo_min_leaf,
        "min_samples_split": rf_gfo_min_split,
        "random_state": SEED,
    },

    # Optimizer information
    "search_bounds": rf_gfo_bounds,
    "best_search_position": np.asarray(rf_gfo_best),
    "best_search_error": rf_gfo_err,
    "best_search_cv_r2": 1 - rf_gfo_err,
    "optimization_history": np.asarray(rf_gfo_hist),
    "optimizer_settings": {"n_agents": 10, "n_iterations": 30, "seed": SEED},

    # Fold-level performance
    "cv_results": res_rf_gfo.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "saved_at": datetime.now().isoformat(),
}

RF_GFO_SAVE_PATH = os.path.join(SAVE_DIR, "RF_GFO_Bundle.joblib")
joblib.dump(rf_gfo_shap_bundle, RF_GFO_SAVE_PATH, compress=3)

chk = joblib.load(RF_GFO_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("RF-GFO bundle saved successfully.")
print("File:", RF_GFO_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_rf_gfo.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(RF_GFO_SAVE_PATH) / (1024**2), 2), "MB")

## **XGB_GFO_tunning**

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to XGBoost-GWO
xgb_gfo_bounds = [(100, 800),     # n_estimators
                  (2, 12),        # max_depth
                  (-2.5, -0.5),   # log10 learning_rate
                  (0.5, 1.0),     # subsample
                  (0.5, 1.0),     # colsample_bytree
                  (1, 20),        # min_child_weight
                  (-1.0, 2.0),    # log10 reg_lambda
                  (0.0, 5.0)]     # gamma

def xgb_gfo_cv_error(params):
    model = XGBRegressor(
        n_estimators=int(round(params[0])),
        max_depth=int(round(params[1])),
        learning_rate=float(10.0 ** params[2]),
        subsample=float(params[3]),
        colsample_bytree=float(params[4]),
        min_child_weight=int(round(params[5])),
        reg_lambda=float(10.0 ** params[6]),
        gamma=float(params[7]),
        random_state=SEED, n_jobs=1)
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

xgb_gfo_best, xgb_gfo_err, xgb_gfo_hist = galactic_field_optimizer(
    xgb_gfo_cv_error, bounds=xgb_gfo_bounds, n_agents=10, n_iter=30, seed=SEED)

xgb_gfo_n_est      = int(round(xgb_gfo_best[0]))
xgb_gfo_depth      = int(round(xgb_gfo_best[1]))
xgb_gfo_lr         = float(10.0 ** xgb_gfo_best[2])
xgb_gfo_subsample  = float(xgb_gfo_best[3])
xgb_gfo_colsample  = float(xgb_gfo_best[4])
xgb_gfo_min_child  = int(round(xgb_gfo_best[5]))
xgb_gfo_reg_lambda = float(10.0 ** xgb_gfo_best[6])
xgb_gfo_gamma      = float(xgb_gfo_best[7])

print(f"best XGBoost-GFO:  n_estimators = {xgb_gfo_n_est}   max_depth = {xgb_gfo_depth}   "
      f"learning_rate = {xgb_gfo_lr:.4f}   subsample = {xgb_gfo_subsample:.3f}   "
      f"colsample = {xgb_gfo_colsample:.3f}   min_child_weight = {xgb_gfo_min_child}   "
      f"reg_lambda = {xgb_gfo_reg_lambda:.3f}   gamma = {xgb_gfo_gamma:.3f}")
print(f"best CV R2 during search = {1 - xgb_gfo_err:.4f}")

In [ ]:
def build_xgb_gfo_tuned():
    return XGBRegressor(n_estimators=xgb_gfo_n_est, max_depth=xgb_gfo_depth,
                        learning_rate=xgb_gfo_lr, subsample=xgb_gfo_subsample,
                        colsample_bytree=xgb_gfo_colsample,
                        min_child_weight=xgb_gfo_min_child,
                        reg_lambda=xgb_gfo_reg_lambda, gamma=xgb_gfo_gamma,
                        random_state=SEED)

res_xgb_gfo, xgb_gfo_final = evaluate(build_xgb_gfo_tuned)
report(res_xgb_gfo, 'XGBoost-GFO')

In [ ]:
import os
import joblib
import sklearn
import xgboost
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

xgb_gfo_full = build_xgb_gfo_tuned()
xgb_gfo_full.fit(X, y)

xgb_gfo_shap_bundle = {
    "model_name": "XGBoost",
    "optimization_method": "Galactic Field Optimization",

    # xgb_gfo_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. xgb_gfo_full is refit on all rows.
    "model": xgb_gfo_final,
    "model_full_data": xgb_gfo_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": xgb_gfo_final.predict(X_test),
    "y_dev_pred": xgb_gfo_final.predict(X_dev),

    # Deterministic background sample for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "n_estimators": xgb_gfo_n_est,
        "max_depth": xgb_gfo_depth,
        "learning_rate": xgb_gfo_lr,
        "subsample": xgb_gfo_subsample,
        "colsample_bytree": xgb_gfo_colsample,
        "min_child_weight": xgb_gfo_min_child,
        "reg_lambda": xgb_gfo_reg_lambda,
        "gamma": xgb_gfo_gamma,
        "random_state": SEED,
    },

    # Optimizer information
    "search_bounds": xgb_gfo_bounds,
    "best_search_position": np.asarray(xgb_gfo_best),
    "best_search_error": xgb_gfo_err,
    "best_search_cv_r2": 1 - xgb_gfo_err,
    "optimization_history": np.asarray(xgb_gfo_hist),
    "optimizer_settings": {"n_agents": 10, "n_iterations": 30, "seed": SEED},

    # Fold-level performance
    "cv_results": res_xgb_gfo.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "xgboost_version": xgboost.__version__,
    "saved_at": datetime.now().isoformat(),
}

XGB_GFO_SAVE_PATH = os.path.join(SAVE_DIR, "XGBoost_GFO_Bundle.joblib")
joblib.dump(xgb_gfo_shap_bundle, XGB_GFO_SAVE_PATH, compress=3)

chk = joblib.load(XGB_GFO_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("XGBoost-GFO bundle saved successfully.")
print("File:", XGB_GFO_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_xgb_gfo.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(XGB_GFO_SAVE_PATH) / (1024**2), 2), "MB")

## **SVR_GFO_tunning**

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to SVR-GWO
svr_gfo_bounds = [(-2, 4),    # log10 C       -> [0.01, 10000]
                  (-5, 1),    # log10 gamma   -> [1e-5, 10]
                  (-3, 0)]    # log10 epsilon -> [0.001, 1]

def svr_gfo_cv_error(params):
    model = TransformedTargetRegressor(
        regressor=make_pipeline(
            StandardScaler(),
            SVR(kernel='rbf',
                C=float(10.0 ** params[0]),
                gamma=float(10.0 ** params[1]),
                epsilon=float(10.0 ** params[2]))),
        transformer=StandardScaler())
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

svr_gfo_best, svr_gfo_err, svr_gfo_hist = galactic_field_optimizer(
    svr_gfo_cv_error, bounds=svr_gfo_bounds, n_agents=10, n_iter=30, seed=SEED)

svr_gfo_C       = float(10.0 ** svr_gfo_best[0])
svr_gfo_gamma   = float(10.0 ** svr_gfo_best[1])
svr_gfo_epsilon = float(10.0 ** svr_gfo_best[2])

print(f"best SVR-GFO:  C = {svr_gfo_C:.3f}   gamma = {svr_gfo_gamma:.4f}   "
      f"epsilon = {svr_gfo_epsilon:.4f}")
print(f"best CV R2 during search = {1 - svr_gfo_err:.4f}")

In [ ]:
def build_svr_gfo_tuned():
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(),
                                SVR(kernel='rbf', C=svr_gfo_C, gamma=svr_gfo_gamma,
                                    epsilon=svr_gfo_epsilon)),
        transformer=StandardScaler())

res_svr_gfo, svr_gfo_final = evaluate(build_svr_gfo_tuned)
report(res_svr_gfo, 'SVR-GFO')

In [ ]:
import os
import joblib
import sklearn
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

svr_gfo_full = build_svr_gfo_tuned()
svr_gfo_full.fit(X, y)

svr_gfo_shap_bundle = {
    "model_name": "SVR",
    "optimization_method": "Galactic Field Optimization",

    # svr_gfo_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. svr_gfo_full is refit on all rows.
    "model": svr_gfo_final,
    "model_full_data": svr_gfo_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": svr_gfo_final.predict(X_test),
    "y_dev_pred": svr_gfo_final.predict(X_dev),

    # Deterministic background sample, required for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "C": svr_gfo_C,
        "gamma": svr_gfo_gamma,
        "epsilon": svr_gfo_epsilon,
        "kernel": "rbf",
    },

    # Optimizer information
    "search_bounds": svr_gfo_bounds,
    "best_search_position": np.asarray(svr_gfo_best),
    "best_search_error": svr_gfo_err,
    "best_search_cv_r2": 1 - svr_gfo_err,
    "optimization_history": np.asarray(svr_gfo_hist),
    "optimizer_settings": {"n_agents": 10, "n_iterations": 30, "seed": SEED},

    # Fold-level performance
    "cv_results": res_svr_gfo.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "saved_at": datetime.now().isoformat(),
}

SVR_GFO_SAVE_PATH = os.path.join(SAVE_DIR, "SVR_GFO_Bundle.joblib")
joblib.dump(svr_gfo_shap_bundle, SVR_GFO_SAVE_PATH, compress=3)

chk = joblib.load(SVR_GFO_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("SVR-GFO bundle saved successfully.")
print("File:", SVR_GFO_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Background records:", chk["X_background"].shape[0])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_svr_gfo.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(SVR_GFO_SAVE_PATH) / (1024**2), 2), "MB")

## **ANN_GFO_tunning**

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to ANN-GWO
ann_gfo_bounds = [(16, 160),     # first hidden width
                  (8, 96),       # second hidden width
                  (-4, 0.5),     # log10 alpha -> ~[1e-4, 3]
                  (-3.5, -1.2)]  # log10 learning_rate_init

def ann_gfo_cv_error(params):
    net = MLPRegressor(
        hidden_layer_sizes=(int(round(params[0])), int(round(params[1]))),
        activation="relu", solver="adam",
        alpha=float(10.0 ** params[2]),
        learning_rate_init=float(10.0 ** params[3]),
        max_iter=2000, random_state=SEED)
    model = TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), net),
        transformer=StandardScaler())
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

ann_gfo_best, ann_gfo_err, ann_gfo_hist = galactic_field_optimizer(
    ann_gfo_cv_error, bounds=ann_gfo_bounds, n_agents=10, n_iter=30, seed=SEED)

ann_gfo_l1, ann_gfo_l2 = int(round(ann_gfo_best[0])), int(round(ann_gfo_best[1]))
ann_gfo_alpha = float(10.0 ** ann_gfo_best[2])
ann_gfo_lr    = float(10.0 ** ann_gfo_best[3])

print(f"best ANN-GFO:  layers = ({ann_gfo_l1}, {ann_gfo_l2})   "
      f"alpha = {ann_gfo_alpha:.5f}   lr = {ann_gfo_lr:.5f}")
print(f"best CV R2 during search = {1 - ann_gfo_err:.4f}")

In [ ]:
def build_ann_gfo_tuned():
    net = MLPRegressor(hidden_layer_sizes=(ann_gfo_l1, ann_gfo_l2), activation="relu",
                       solver="adam", alpha=ann_gfo_alpha,
                       learning_rate_init=ann_gfo_lr, max_iter=2000,
                       random_state=SEED)
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), net),
        transformer=StandardScaler())

res_ann_gfo, ann_gfo_final = evaluate(build_ann_gfo_tuned)
report(res_ann_gfo, 'ANN-GFO')

In [ ]:
import os
import joblib
import sklearn
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

ann_gfo_full = build_ann_gfo_tuned()
ann_gfo_full.fit(X, y)

ann_gfo_shap_bundle = {
    "model_name": "ANN",
    "optimization_method": "Galactic Field Optimization",

    # ann_gfo_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. ann_gfo_full is refit on all rows.
    "model": ann_gfo_final,
    "model_full_data": ann_gfo_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": ann_gfo_final.predict(X_test),
    "y_dev_pred": ann_gfo_final.predict(X_dev),

    # Deterministic background sample, required for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "hidden_layer_sizes": (ann_gfo_l1, ann_gfo_l2),
        "activation": "relu",
        "solver": "adam",
        "alpha": ann_gfo_alpha,
        "learning_rate_init": ann_gfo_lr,
        "max_iter": 2000,
        "random_state": SEED,
    },

    # Optimizer information
    "search_bounds": ann_gfo_bounds,
    "best_search_position": np.asarray(ann_gfo_best),
    "best_search_error": ann_gfo_err,
    "best_search_cv_r2": 1 - ann_gfo_err,
    "optimization_history": np.asarray(ann_gfo_hist),
    "optimizer_settings": {"n_agents": 10, "n_iterations": 30, "seed": SEED},

    # Fold-level performance
    "cv_results": res_ann_gfo.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "saved_at": datetime.now().isoformat(),
}

ANN_GFO_SAVE_PATH = os.path.join(SAVE_DIR, "ANN_GFO_Bundle.joblib")
joblib.dump(ann_gfo_shap_bundle, ANN_GFO_SAVE_PATH, compress=3)

chk = joblib.load(ANN_GFO_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("ANN-GFO bundle saved successfully.")
print("File:", ANN_GFO_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Background records:", chk["X_background"].shape[0])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_ann_gfo.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(ANN_GFO_SAVE_PATH) / (1024**2), 2), "MB")

## **CatBoost_GFO_tunning**

In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to CatBoost-GWO
cat_gfo_bounds = [(200, 1600),   # iterations
                  (3, 8),        # depth
                  (-2.5, -0.5),  # log10 learning_rate -> ~[0.003, 0.316]
                  (-0.5, 2.5)]   # log10 l2_leaf_reg   -> ~[0.316, 316]

def cat_gfo_cv_error(params):
    model = CatBoostRegressor(
        iterations=int(round(params[0])),
        depth=int(round(params[1])),
        learning_rate=float(10.0 ** params[2]),
        l2_leaf_reg=float(10.0 ** params[3]),
        random_state=SEED, verbose=0)             # all 12 threads
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2").mean()   # folds sequential

cat_gfo_best, cat_gfo_err, cat_gfo_hist = galactic_field_optimizer(
    cat_gfo_cv_error, bounds=cat_gfo_bounds, n_agents=10, n_iter=30, seed=SEED)

cat_gfo_iter  = int(round(cat_gfo_best[0]))
cat_gfo_depth = int(round(cat_gfo_best[1]))
cat_gfo_lr    = float(10.0 ** cat_gfo_best[2])
cat_gfo_l2    = float(10.0 ** cat_gfo_best[3])

print(f"best CatBoost-GFO:  iterations = {cat_gfo_iter}   depth = {cat_gfo_depth}   "
      f"learning_rate = {cat_gfo_lr:.4f}   l2_leaf_reg = {cat_gfo_l2:.3f}")
print(f"best CV R2 during search = {1 - cat_gfo_err:.4f}")

In [ ]:
def build_cat_gfo_tuned():
    return CatBoostRegressor(iterations=cat_gfo_iter, depth=cat_gfo_depth,
                             learning_rate=cat_gfo_lr, l2_leaf_reg=cat_gfo_l2,
                             random_state=SEED, verbose=0)

res_cat_gfo, cat_gfo_final = evaluate(build_cat_gfo_tuned)
report(res_cat_gfo, 'CatBoost-GFO')

In [ ]:
import os
import joblib
import sklearn
import catboost
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

cat_gfo_full = build_cat_gfo_tuned()
cat_gfo_full.fit(X, y)

cat_gfo_shap_bundle = {
    "model_name": "CatBoost",
    "optimization_method": "Galactic Field Optimization",

    # cat_gfo_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. cat_gfo_full is refit on all rows.
    "model": cat_gfo_final,
    "model_full_data": cat_gfo_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": cat_gfo_final.predict(X_test),
    "y_dev_pred": cat_gfo_final.predict(X_dev),

    # Deterministic background sample, available if needed during SHAP analysis
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "iterations": cat_gfo_iter,
        "depth": cat_gfo_depth,
        "learning_rate": cat_gfo_lr,
        "l2_leaf_reg": cat_gfo_l2,
        "random_state": SEED,
        "verbose": 0,
    },

    # Optimizer information
    "search_bounds": cat_gfo_bounds,
    "best_search_position": np.asarray(cat_gfo_best),
    "best_search_error": cat_gfo_err,
    "best_search_cv_r2": 1 - cat_gfo_err,
    "optimization_history": np.asarray(cat_gfo_hist),
    "optimizer_settings": {"n_agents": 10, "n_iterations": 30, "seed": SEED},

    # Fold-level performance
    "cv_results": res_cat_gfo.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "catboost_version": catboost.__version__,
    "saved_at": datetime.now().isoformat(),
}

CAT_GFO_SAVE_PATH = os.path.join(SAVE_DIR, "CatBoost_GFO_Bundle.joblib")
joblib.dump(cat_gfo_shap_bundle, CAT_GFO_SAVE_PATH, compress=3)

chk = joblib.load(CAT_GFO_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("CatBoost-GFO bundle saved successfully.")
print("File:", CAT_GFO_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Background records:", chk["X_background"].shape[0])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_cat_gfo.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(CAT_GFO_SAVE_PATH) / (1024**2), 2), "MB")

# **Whale Optimization Algorithm**

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score

def whale_optimization_algorithm(objective, bounds, n_whales=10, n_iter=30,
                                 seed=None, b=1.0, verbose=False):
    """Minimize `objective` over `bounds` using the Whale Optimization Algorithm."""
    seed = int(globals().get("SEED", 42) if seed is None else seed)
    rng = np.random.default_rng(seed)

    bounds = np.asarray(bounds, dtype=float)
    if bounds.ndim != 2 or bounds.shape[1] != 2:
        raise ValueError("bounds must contain (lower, upper) pairs.")
    if np.any(~np.isfinite(bounds)) or np.any(bounds[:, 0] >= bounds[:, 1]):
        raise ValueError("Every bound must be finite and satisfy lower < upper.")
    if n_whales < 5 or n_iter < 1:
        raise ValueError("n_whales must be >= 5 and n_iter must be >= 1.")

    lb, ub = bounds[:, 0], bounds[:, 1]
    dim = len(bounds)

    def checked_objective(x):
        try:
            value = float(objective(np.asarray(x, dtype=float)))
            return value if np.isfinite(value) else np.inf
        except Exception:
            return np.inf

    whales = rng.uniform(lb, ub, size=(int(n_whales), dim))
    fitness = np.array([checked_objective(w) for w in whales], dtype=float)

    best_idx = int(np.argmin(fitness))
    best_x = whales[best_idx].copy()
    best_f = float(fitness[best_idx])
    history = [best_f]

    for t in range(int(n_iter)):
        a = 2.0 - 2.0 * t / n_iter

        for i in range(int(n_whales)):
            r1, r2 = rng.random(), rng.random()
            A = 2.0 * a * r1 - a
            C = 2.0 * r2
            p = rng.random()
            l = rng.uniform(-1.0, 1.0)

            if p < 0.5:
                if abs(A) < 1.0:
                    D = np.abs(C * best_x - whales[i])
                    new_pos = best_x - A * D
                else:
                    rand_idx = rng.integers(int(n_whales))
                    X_rand = whales[rand_idx].copy()
                    D = np.abs(C * X_rand - whales[i])
                    new_pos = X_rand - A * D
            else:
                D = np.abs(best_x - whales[i])
                new_pos = D * np.exp(b * l) * np.cos(2.0 * np.pi * l) + best_x

            whales[i] = np.clip(new_pos, lb, ub)

        fitness = np.array([checked_objective(w) for w in whales], dtype=float)
        current_idx = int(np.argmin(fitness))
        current_f = float(fitness[current_idx])

        if current_f < best_f:
            best_x = whales[current_idx].copy()
            best_f = current_f

        history.append(best_f)

        if verbose and (t + 1) % max(1, n_iter // 10) == 0:
            print(f"Iteration {t + 1:4d}/{n_iter} | Best objective = {best_f:.6g}")

    return best_x, best_f, history

# Automatic reproducibility and validity check
def _check_woa():
    sphere = lambda x: float(np.sum(np.asarray(x) ** 2))
    test_bounds = [(-5, 5)] * 4

    x1, f1, h1 = whale_optimization_algorithm(
        sphere, test_bounds, n_whales=8, n_iter=10, seed=123
    )
    x2, f2, h2 = whale_optimization_algorithm(
        sphere, test_bounds, n_whales=8, n_iter=10, seed=123
    )

    assert np.allclose(x1, x2)
    assert np.isclose(f1, f2)
    assert np.allclose(h1, h2)
    assert len(h1) == 11
    assert np.all(np.diff(h1) <= 1e-12)
    assert np.all((x1 >= -5) & (x1 <= 5))
    assert np.isfinite(f1)

    print("WOA implementation check passed.")
    print("best f found:", round(f1, 6))
    print("best x:", np.round(x1, 4))

_check_woa()

## **RF_WOA_tunning**

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to RF-GWO and RF-GFO
rf_woa_bounds = [(100, 800),    # n_estimators
                 (3, 30),       # max_depth
                 (0.3, 1.0),    # max_features
                 (1, 20),       # min_samples_leaf
                 (2, 20)]       # min_samples_split

def rf_woa_cv_error(params):
    model = RandomForestRegressor(
        n_estimators=int(round(params[0])),
        max_depth=int(round(params[1])),
        max_features=float(params[2]),
        min_samples_leaf=int(round(params[3])),
        min_samples_split=int(round(params[4])),
        random_state=SEED, n_jobs=1)
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

rf_woa_best, rf_woa_err, rf_woa_hist = whale_optimization_algorithm(
    rf_woa_cv_error,
    bounds=rf_woa_bounds,
    n_whales=10,
    n_iter=30,
    seed=SEED
)

rf_woa_n_est     = int(round(rf_woa_best[0]))
rf_woa_depth     = int(round(rf_woa_best[1]))
rf_woa_max_feat  = float(rf_woa_best[2])
rf_woa_min_leaf  = int(round(rf_woa_best[3]))
rf_woa_min_split = int(round(rf_woa_best[4]))

print(f"best RF-WOA:  n_estimators = {rf_woa_n_est}   max_depth = {rf_woa_depth}   "
      f"max_features = {rf_woa_max_feat:.3f}   min_samples_leaf = {rf_woa_min_leaf}   "
      f"min_samples_split = {rf_woa_min_split}")
print(f"best CV R2 during search = {1 - rf_woa_err:.4f}")

In [ ]:
def build_rf_woa_tuned():
    return RandomForestRegressor(
        n_estimators=rf_woa_n_est,
        max_depth=rf_woa_depth,
        max_features=rf_woa_max_feat,
        min_samples_leaf=rf_woa_min_leaf,
        min_samples_split=rf_woa_min_split,
        random_state=SEED)

res_rf_woa, rf_woa_final = evaluate(build_rf_woa_tuned)
report(res_rf_woa, 'RF-WOA')

In [ ]:
import os
import joblib
import sklearn
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

rf_woa_full = build_rf_woa_tuned()
rf_woa_full.fit(X, y)

rf_woa_shap_bundle = {
    "model_name": "Random Forest",
    "optimization_method": "Whale Optimization Algorithm",

    # rf_woa_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. rf_woa_full is refit on all rows.
    "model": rf_woa_final,
    "model_full_data": rf_woa_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": rf_woa_final.predict(X_test),
    "y_dev_pred": rf_woa_final.predict(X_dev),

    # Deterministic background sample for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "n_estimators": rf_woa_n_est,
        "max_depth": rf_woa_depth,
        "max_features": rf_woa_max_feat,
        "min_samples_leaf": rf_woa_min_leaf,
        "min_samples_split": rf_woa_min_split,
        "random_state": SEED,
    },

    # Optimizer information
    "search_bounds": rf_woa_bounds,
    "best_search_position": np.asarray(rf_woa_best),
    "best_search_error": rf_woa_err,
    "best_search_cv_r2": 1 - rf_woa_err,
    "optimization_history": np.asarray(rf_woa_hist),
    "optimizer_settings": {"n_whales": 10, "n_iterations": 30, "seed": SEED},

    # Fold-level performance
    "cv_results": res_rf_woa.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "saved_at": datetime.now().isoformat(),
}

RF_WOA_SAVE_PATH = os.path.join(SAVE_DIR, "RF_WOA_Bundle.joblib")
joblib.dump(rf_woa_shap_bundle, RF_WOA_SAVE_PATH, compress=3)

chk = joblib.load(RF_WOA_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("RF-WOA bundle saved successfully.")
print("File:", RF_WOA_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_rf_woa.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(RF_WOA_SAVE_PATH) / (1024**2), 2), "MB")

## **XGB_WOA_tunning**

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to XGBoost-GWO and XGBoost-GFO
xgb_woa_bounds = [(100, 800),     # n_estimators
                  (2, 12),        # max_depth
                  (-2.5, -0.5),   # log10 learning_rate
                  (0.5, 1.0),     # subsample
                  (0.5, 1.0),     # colsample_bytree
                  (1, 20),        # min_child_weight
                  (-1.0, 2.0),    # log10 reg_lambda
                  (0.0, 5.0)]     # gamma

def xgb_woa_cv_error(params):
    model = XGBRegressor(
        n_estimators=int(round(params[0])),
        max_depth=int(round(params[1])),
        learning_rate=float(10.0 ** params[2]),
        subsample=float(params[3]),
        colsample_bytree=float(params[4]),
        min_child_weight=int(round(params[5])),
        reg_lambda=float(10.0 ** params[6]),
        gamma=float(params[7]),
        random_state=SEED, n_jobs=1)
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

xgb_woa_best, xgb_woa_err, xgb_woa_hist = whale_optimization_algorithm(
    xgb_woa_cv_error, bounds=xgb_woa_bounds, n_whales=10, n_iter=30, seed=SEED)

xgb_woa_n_est      = int(round(xgb_woa_best[0]))
xgb_woa_depth      = int(round(xgb_woa_best[1]))
xgb_woa_lr         = float(10.0 ** xgb_woa_best[2])
xgb_woa_subsample  = float(xgb_woa_best[3])
xgb_woa_colsample  = float(xgb_woa_best[4])
xgb_woa_min_child  = int(round(xgb_woa_best[5]))
xgb_woa_reg_lambda = float(10.0 ** xgb_woa_best[6])
xgb_woa_gamma      = float(xgb_woa_best[7])

print(f"best XGBoost-WOA:  n_estimators = {xgb_woa_n_est}   max_depth = {xgb_woa_depth}   "
      f"learning_rate = {xgb_woa_lr:.4f}   subsample = {xgb_woa_subsample:.3f}   "
      f"colsample = {xgb_woa_colsample:.3f}   min_child_weight = {xgb_woa_min_child}   "
      f"reg_lambda = {xgb_woa_reg_lambda:.3f}   gamma = {xgb_woa_gamma:.3f}")
print(f"best CV R2 during search = {1 - xgb_woa_err:.4f}")

In [ ]:
def build_xgb_woa_tuned():
    return XGBRegressor(n_estimators=xgb_woa_n_est, max_depth=xgb_woa_depth,
                        learning_rate=xgb_woa_lr, subsample=xgb_woa_subsample,
                        colsample_bytree=xgb_woa_colsample,
                        min_child_weight=xgb_woa_min_child,
                        reg_lambda=xgb_woa_reg_lambda, gamma=xgb_woa_gamma,
                        random_state=SEED)

res_xgb_woa, xgb_woa_final = evaluate(build_xgb_woa_tuned)
report(res_xgb_woa, 'XGBoost-WOA')

In [ ]:
import os
import joblib
import sklearn
import xgboost
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

xgb_woa_full = build_xgb_woa_tuned()
xgb_woa_full.fit(X, y)

xgb_woa_shap_bundle = {
    "model_name": "XGBoost",
    "optimization_method": "Whale Optimization Algorithm",

    # xgb_woa_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. xgb_woa_full is refit on all rows.
    "model": xgb_woa_final,
    "model_full_data": xgb_woa_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": xgb_woa_final.predict(X_test),
    "y_dev_pred": xgb_woa_final.predict(X_dev),

    # Deterministic background sample for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "n_estimators": xgb_woa_n_est,
        "max_depth": xgb_woa_depth,
        "learning_rate": xgb_woa_lr,
        "subsample": xgb_woa_subsample,
        "colsample_bytree": xgb_woa_colsample,
        "min_child_weight": xgb_woa_min_child,
        "reg_lambda": xgb_woa_reg_lambda,
        "gamma": xgb_woa_gamma,
        "random_state": SEED,
    },

    # Optimizer information
    "search_bounds": xgb_woa_bounds,
    "best_search_position": np.asarray(xgb_woa_best),
    "best_search_error": xgb_woa_err,
    "best_search_cv_r2": 1 - xgb_woa_err,
    "optimization_history": np.asarray(xgb_woa_hist),
    "optimizer_settings": {"n_whales": 10, "n_iterations": 30, "seed": SEED},

    # Fold-level performance
    "cv_results": res_xgb_woa.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "xgboost_version": xgboost.__version__,
    "saved_at": datetime.now().isoformat(),
}

XGB_WOA_SAVE_PATH = os.path.join(SAVE_DIR, "XGBoost_WOA_Bundle.joblib")
joblib.dump(xgb_woa_shap_bundle, XGB_WOA_SAVE_PATH, compress=3)

chk = joblib.load(XGB_WOA_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("XGBoost-WOA bundle saved successfully.")
print("File:", XGB_WOA_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_xgb_woa.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(XGB_WOA_SAVE_PATH) / (1024**2), 2), "MB")

## **SVR_WOA_tunning**

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to SVR-GWO and SVR-GFO
svr_woa_bounds = [(-2, 4),    # log10 C       -> [0.01, 10000]
                  (-5, 1),    # log10 gamma   -> [1e-5, 10]
                  (-3, 0)]    # log10 epsilon -> [0.001, 1]

def svr_woa_cv_error(params):
    model = TransformedTargetRegressor(
        regressor=make_pipeline(
            StandardScaler(),
            SVR(kernel='rbf',
                C=float(10.0 ** params[0]),
                gamma=float(10.0 ** params[1]),
                epsilon=float(10.0 ** params[2]))),
        transformer=StandardScaler())
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

svr_woa_best, svr_woa_err, svr_woa_hist = whale_optimization_algorithm(
    svr_woa_cv_error, bounds=svr_woa_bounds, n_whales=10, n_iter=30, seed=SEED)

svr_woa_C       = float(10.0 ** svr_woa_best[0])
svr_woa_gamma   = float(10.0 ** svr_woa_best[1])
svr_woa_epsilon = float(10.0 ** svr_woa_best[2])

print(f"best SVR-WOA:  C = {svr_woa_C:.3f}   gamma = {svr_woa_gamma:.4f}   "
      f"epsilon = {svr_woa_epsilon:.4f}")
print(f"best CV R2 during search = {1 - svr_woa_err:.4f}")

In [ ]:
def build_svr_woa_tuned():
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(),
                                SVR(kernel='rbf', C=svr_woa_C, gamma=svr_woa_gamma,
                                    epsilon=svr_woa_epsilon)),
        transformer=StandardScaler())

res_svr_woa, svr_woa_final = evaluate(build_svr_woa_tuned)
report(res_svr_woa, 'SVR-WOA')

In [ ]:
import os
import joblib
import sklearn
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

svr_woa_full = build_svr_woa_tuned()
svr_woa_full.fit(X, y)

svr_woa_shap_bundle = {
    "model_name": "SVR",
    "optimization_method": "Whale Optimization Algorithm",

    # svr_woa_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. svr_woa_full is refit on all rows.
    "model": svr_woa_final,
    "model_full_data": svr_woa_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": svr_woa_final.predict(X_test),
    "y_dev_pred": svr_woa_final.predict(X_dev),

    # Deterministic background sample, required for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "C": svr_woa_C,
        "gamma": svr_woa_gamma,
        "epsilon": svr_woa_epsilon,
        "kernel": "rbf",
    },

    # Optimizer information
    "search_bounds": svr_woa_bounds,
    "best_search_position": np.asarray(svr_woa_best),
    "best_search_error": svr_woa_err,
    "best_search_cv_r2": 1 - svr_woa_err,
    "optimization_history": np.asarray(svr_woa_hist),
    "optimizer_settings": {"n_whales": 10, "n_iterations": 30, "seed": SEED},

    # Fold-level performance
    "cv_results": res_svr_woa.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "saved_at": datetime.now().isoformat(),
}

SVR_WOA_SAVE_PATH = os.path.join(SAVE_DIR, "SVR_WOA_Bundle.joblib")
joblib.dump(svr_woa_shap_bundle, SVR_WOA_SAVE_PATH, compress=3)

chk = joblib.load(SVR_WOA_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("SVR-WOA bundle saved successfully.")
print("File:", SVR_WOA_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Background records:", chk["X_background"].shape[0])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_svr_woa.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(SVR_WOA_SAVE_PATH) / (1024**2), 2), "MB")

## **ANN_WOA_tunning**

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to ANN-GWO and ANN-GFO
ann_woa_bounds = [(16, 160),     # first hidden width
                  (8, 96),       # second hidden width
                  (-4, 0.5),     # log10 alpha -> ~[1e-4, 3]
                  (-3.5, -1.2)]  # log10 learning_rate_init

def ann_woa_cv_error(params):
    net = MLPRegressor(
        hidden_layer_sizes=(int(round(params[0])), int(round(params[1]))),
        activation="relu", solver="adam",
        alpha=float(10.0 ** params[2]),
        learning_rate_init=float(10.0 ** params[3]),
        max_iter=2000, random_state=SEED)
    model = TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), net),
        transformer=StandardScaler())
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

ann_woa_best, ann_woa_err, ann_woa_hist = whale_optimization_algorithm(
    ann_woa_cv_error, bounds=ann_woa_bounds, n_whales=10, n_iter=30, seed=SEED)

ann_woa_l1, ann_woa_l2 = int(round(ann_woa_best[0])), int(round(ann_woa_best[1]))
ann_woa_alpha = float(10.0 ** ann_woa_best[2])
ann_woa_lr    = float(10.0 ** ann_woa_best[3])

print(f"best ANN-WOA:  layers = ({ann_woa_l1}, {ann_woa_l2})   "
      f"alpha = {ann_woa_alpha:.5f}   lr = {ann_woa_lr:.5f}")
print(f"best CV R2 during search = {1 - ann_woa_err:.4f}")

In [ ]:
def build_ann_woa_tuned():
    net = MLPRegressor(hidden_layer_sizes=(ann_woa_l1, ann_woa_l2), activation="relu",
                       solver="adam", alpha=ann_woa_alpha,
                       learning_rate_init=ann_woa_lr, max_iter=2000,
                       random_state=SEED)
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), net),
        transformer=StandardScaler())

res_ann_woa, ann_woa_final = evaluate(build_ann_woa_tuned)
report(res_ann_woa, 'ANN-WOA')

In [ ]:
import os
import joblib
import sklearn
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

ann_woa_full = build_ann_woa_tuned()
ann_woa_full.fit(X, y)

ann_woa_shap_bundle = {
    "model_name": "ANN",
    "optimization_method": "Whale Optimization Algorithm",

    # ann_woa_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. ann_woa_full is refit on all rows.
    "model": ann_woa_final,
    "model_full_data": ann_woa_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": ann_woa_final.predict(X_test),
    "y_dev_pred": ann_woa_final.predict(X_dev),

    # Deterministic background sample, required for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "hidden_layer_sizes": (ann_woa_l1, ann_woa_l2),
        "activation": "relu",
        "solver": "adam",
        "alpha": ann_woa_alpha,
        "learning_rate_init": ann_woa_lr,
        "max_iter": 2000,
        "random_state": SEED,
    },

    # Optimizer information
    "search_bounds": ann_woa_bounds,
    "best_search_position": np.asarray(ann_woa_best),
    "best_search_error": ann_woa_err,
    "best_search_cv_r2": 1 - ann_woa_err,
    "optimization_history": np.asarray(ann_woa_hist),
    "optimizer_settings": {"n_whales": 10, "n_iterations": 30, "seed": SEED},

    # Fold-level performance
    "cv_results": res_ann_woa.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "saved_at": datetime.now().isoformat(),
}

ANN_WOA_SAVE_PATH = os.path.join(SAVE_DIR, "ANN_WOA_Bundle.joblib")
joblib.dump(ann_woa_shap_bundle, ANN_WOA_SAVE_PATH, compress=3)

chk = joblib.load(ANN_WOA_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("ANN-WOA bundle saved successfully.")
print("File:", ANN_WOA_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Background records:", chk["X_background"].shape[0])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_ann_woa.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(ANN_WOA_SAVE_PATH) / (1024**2), 2), "MB")

## **CatBoost_WOA_tunning**

In [ ]:
import time, subprocess
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_val_score

print(subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode().strip())

def cat_probe(params, task_type):
    model = CatBoostRegressor(
        iterations=int(round(params[0])),
        depth=int(round(params[1])),
        learning_rate=float(10.0 ** params[2]),
        l2_leaf_reg=float(10.0 ** params[3]),
        task_type=task_type, devices='0',
        random_state=SEED, verbose=0)
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits, scoring="r2").mean()

for tt in ['CPU', 'GPU']:
    t0 = time.time(); e = cat_probe(np.array([1600., 8., -1.5, 1.0]), tt)
    worst = time.time() - t0
    t0 = time.time(); cat_probe(np.array([900., 5., -1.5, 1.0]), tt)
    mid = time.time() - t0
    print(f"{tt}: worst(1600,d8) = {worst:.1f} s | mid(900,d5) = {mid:.1f} s | "
          f"CV R2 = {1 - e:.4f}")

In [ ]:
import time
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to CatBoost-GWO and CatBoost-GFO
cat_woa_bounds = [(200, 1600),   # iterations
                  (3, 8),        # depth
                  (-2.5, -0.5),  # log10 learning_rate -> ~[0.003, 0.316]
                  (-0.5, 2.5)]   # log10 l2_leaf_reg   -> ~[0.316, 316]

_n, _t0 = 0, time.time()

def cat_woa_cv_error(params):
    global _n
    model = CatBoostRegressor(
        iterations=int(round(params[0])),
        depth=int(round(params[1])),
        learning_rate=float(10.0 ** params[2]),
        l2_leaf_reg=float(10.0 ** params[3]),
        random_state=SEED, verbose=0)             # all 12 threads
    score = 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                                scoring="r2").mean()   # folds sequential
    _n += 1
    if _n % 10 == 0:
        el = (time.time() - _t0) / 60
        print(f"  {_n:3d}/310   {el:5.1f} min elapsed   ETA {el*310/_n:5.0f} min total",
              flush=True)
    return score

cat_woa_best, cat_woa_err, cat_woa_hist = whale_optimization_algorithm(
    cat_woa_cv_error, bounds=cat_woa_bounds, n_whales=10, n_iter=30, seed=SEED)

cat_woa_iter  = int(round(cat_woa_best[0]))
cat_woa_depth = int(round(cat_woa_best[1]))
cat_woa_lr    = float(10.0 ** cat_woa_best[2])
cat_woa_l2    = float(10.0 ** cat_woa_best[3])

print(f"\nbest CatBoost-WOA:  iterations = {cat_woa_iter}   depth = {cat_woa_depth}   "
      f"learning_rate = {cat_woa_lr:.4f}   l2_leaf_reg = {cat_woa_l2:.3f}")
print(f"best CV R2 during search = {1 - cat_woa_err:.4f}")

In [ ]:
def build_cat_woa_tuned():
    return CatBoostRegressor(iterations=cat_woa_iter, depth=cat_woa_depth,
                             learning_rate=cat_woa_lr, l2_leaf_reg=cat_woa_l2,
                             random_state=SEED, verbose=0)

res_cat_woa, cat_woa_final = evaluate(build_cat_woa_tuned)
report(res_cat_woa, 'CatBoost-WOA')

In [ ]:
import os
import joblib
import sklearn
import catboost
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

cat_woa_full = build_cat_woa_tuned()
cat_woa_full.fit(X, y)

cat_woa_shap_bundle = {
    "model_name": "CatBoost",
    "optimization_method": "Whale Optimization Algorithm",

    # cat_woa_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. cat_woa_full is refit on all rows.
    "model": cat_woa_final,
    "model_full_data": cat_woa_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": cat_woa_final.predict(X_test),
    "y_dev_pred": cat_woa_final.predict(X_dev),

    # Deterministic background sample, available if needed during SHAP analysis
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "iterations": cat_woa_iter,
        "depth": cat_woa_depth,
        "learning_rate": cat_woa_lr,
        "l2_leaf_reg": cat_woa_l2,
        "random_state": SEED,
        "verbose": 0,
    },

    # Optimizer information
    "search_bounds": cat_woa_bounds,
    "best_search_position": np.asarray(cat_woa_best),
    "best_search_error": cat_woa_err,
    "best_search_cv_r2": 1 - cat_woa_err,
    "optimization_history": np.asarray(cat_woa_hist),
    "optimizer_settings": {"n_whales": 10, "n_iterations": 30, "seed": SEED},

    # Fold-level performance
    "cv_results": res_cat_woa.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "catboost_version": catboost.__version__,
    "saved_at": datetime.now().isoformat(),
}

CAT_WOA_SAVE_PATH = os.path.join(SAVE_DIR, "CatBoost_WOA_Bundle.joblib")
joblib.dump(cat_woa_shap_bundle, CAT_WOA_SAVE_PATH, compress=3)

chk = joblib.load(CAT_WOA_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("CatBoost-WOA bundle saved successfully.")
print("File:", CAT_WOA_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Background records:", chk["X_background"].shape[0])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_cat_woa.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(CAT_WOA_SAVE_PATH) / (1024**2), 2), "MB")

# **Differential Evolution Optimization**

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score

def differential_evolution_optimizer(objective, bounds, n_agents=10, n_iter=30,
                                     seed=None, F=0.8, CR=0.9, verbose=False):
    """Minimize `objective` over `bounds` using DE/rand/1/bin."""
    seed = int(globals().get("SEED", 42) if seed is None else seed)
    rng = np.random.default_rng(seed)

    bounds = np.asarray(bounds, dtype=float)
    if bounds.ndim != 2 or bounds.shape[1] != 2:
        raise ValueError("bounds must contain (lower, upper) pairs.")
    if np.any(~np.isfinite(bounds)) or np.any(bounds[:, 0] >= bounds[:, 1]):
        raise ValueError("Every bound must be finite and satisfy lower < upper.")
    if n_agents < 5 or n_iter < 1:
        raise ValueError("n_agents must be >= 5 and n_iter must be >= 1.")
    if not (0 < F <= 2):
        raise ValueError("F must satisfy 0 < F <= 2.")
    if not (0 <= CR <= 1):
        raise ValueError("CR must satisfy 0 <= CR <= 1.")

    lb, ub = bounds[:, 0], bounds[:, 1]
    dim = len(bounds)
    n_agents = int(n_agents)
    n_iter = int(n_iter)

    def checked_objective(x):
        try:
            value = float(objective(np.asarray(x, dtype=float)))
            return value if np.isfinite(value) else np.inf
        except Exception:
            return np.inf

    pop = rng.uniform(lb, ub, size=(n_agents, dim))
    fitness = np.array([checked_objective(x) for x in pop], dtype=float)

    best_idx = int(np.argmin(fitness))
    best_x = pop[best_idx].copy()
    best_f = float(fitness[best_idx])
    history = [best_f]

    for t in range(n_iter):
        for i in range(n_agents):
            candidates = np.delete(np.arange(n_agents), i)
            r1, r2, r3 = rng.choice(candidates, size=3, replace=False)

            mutant = pop[r1] + F * (pop[r2] - pop[r3])
            mutant = np.clip(mutant, lb, ub)

            cross_mask = rng.random(dim) < CR
            cross_mask[rng.integers(dim)] = True
            trial = np.where(cross_mask, mutant, pop[i])

            trial_f = checked_objective(trial)

            if trial_f < fitness[i]:
                pop[i] = trial
                fitness[i] = trial_f

                if trial_f < best_f:
                    best_x = trial.copy()
                    best_f = float(trial_f)

        history.append(best_f)

        if verbose and (t + 1) % max(1, n_iter // 10) == 0:
            print(f"Iteration {t + 1:4d}/{n_iter} | Best objective = {best_f:.6g}")

    return best_x, best_f, history

# Automatic reproducibility and validity check
def _check_de():
    sphere = lambda x: float(np.sum(np.asarray(x) ** 2))
    test_bounds = [(-5, 5)] * 4

    x1, f1, h1 = differential_evolution_optimizer(
        sphere, test_bounds, n_agents=8, n_iter=10, seed=123
    )
    x2, f2, h2 = differential_evolution_optimizer(
        sphere, test_bounds, n_agents=8, n_iter=10, seed=123)

    assert np.allclose(x1, x2)
    assert np.isclose(f1, f2)
    assert np.allclose(h1, h2)
    assert len(h1) == 11
    assert np.all(np.diff(h1) <= 1e-12)
    assert np.all((x1 >= -5) & (x1 <= 5))
    assert np.isfinite(f1)

    print("DE implementation check passed.")
    print("best f found:", round(f1, 6))
    print("best x:", np.round(x1, 4))

_check_de()

## **RF_DE_tunning**

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to RF-GWO, RF-GFO, and RF-WOA
rf_de_bounds = [(100, 800),    # n_estimators
                (3, 30),       # max_depth
                (0.3, 1.0),    # max_features
                (1, 20),       # min_samples_leaf
                (2, 20)]       # min_samples_split

def rf_de_cv_error(params):
    model = RandomForestRegressor(
        n_estimators=int(round(params[0])),
        max_depth=int(round(params[1])),
        max_features=float(params[2]),
        min_samples_leaf=int(round(params[3])),
        min_samples_split=int(round(params[4])),
        random_state=SEED, n_jobs=1)
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

rf_de_best, rf_de_err, rf_de_hist = differential_evolution_optimizer(
    rf_de_cv_error, bounds=rf_de_bounds, n_agents=10, n_iter=30, seed=SEED)

rf_de_n_est     = int(round(rf_de_best[0]))
rf_de_depth     = int(round(rf_de_best[1]))
rf_de_max_feat  = float(rf_de_best[2])
rf_de_min_leaf  = int(round(rf_de_best[3]))
rf_de_min_split = int(round(rf_de_best[4]))

print(f"best RF-DE:  n_estimators = {rf_de_n_est}   max_depth = {rf_de_depth}   "
      f"max_features = {rf_de_max_feat:.3f}   min_samples_leaf = {rf_de_min_leaf}   "
      f"min_samples_split = {rf_de_min_split}")
print(f"best CV R2 during search = {1 - rf_de_err:.4f}")

In [ ]:
def build_rf_de_tuned():
    return RandomForestRegressor(
        n_estimators=rf_de_n_est,
        max_depth=rf_de_depth,
        max_features=rf_de_max_feat,
        min_samples_leaf=rf_de_min_leaf,
        min_samples_split=rf_de_min_split,
        random_state=SEED)

res_rf_de, rf_de_final = evaluate(build_rf_de_tuned)
report(res_rf_de, 'RF-DE')

In [ ]:
import os
import joblib
import sklearn
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

rf_de_full = build_rf_de_tuned()
rf_de_full.fit(X, y)

rf_de_shap_bundle = {
    "model_name": "Random Forest",
    "optimization_method": "Differential Evolution",

    # rf_de_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. rf_de_full is refit on all rows.
    "model": rf_de_final,
    "model_full_data": rf_de_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": rf_de_final.predict(X_test),
    "y_dev_pred": rf_de_final.predict(X_dev),

    # Deterministic background sample for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "n_estimators": rf_de_n_est,
        "max_depth": rf_de_depth,
        "max_features": rf_de_max_feat,
        "min_samples_leaf": rf_de_min_leaf,
        "min_samples_split": rf_de_min_split,
        "random_state": SEED,
    },

    # Optimizer information
    "search_bounds": rf_de_bounds,
    "best_search_position": np.asarray(rf_de_best),
    "best_search_error": rf_de_err,
    "best_search_cv_r2": 1 - rf_de_err,
    "optimization_history": np.asarray(rf_de_hist),
    "optimizer_settings": {"n_agents": 10, "n_iterations": 30, "seed": SEED,
                           "F": 0.8, "CR": 0.9, "strategy": "DE/rand/1/bin"},

    # Fold-level performance
    "cv_results": res_rf_de.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "saved_at": datetime.now().isoformat(),
}

RF_DE_SAVE_PATH = os.path.join(SAVE_DIR, "RF_DE_Bundle.joblib")
joblib.dump(rf_de_shap_bundle, RF_DE_SAVE_PATH, compress=3)

chk = joblib.load(RF_DE_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("RF-DE bundle saved successfully.")
print("File:", RF_DE_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_rf_de.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(RF_DE_SAVE_PATH) / (1024**2), 2), "MB")

## **XGB_DE_tunning**

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to XGBoost-GWO, XGBoost-GFO, and XGBoost-WOA
xgb_de_bounds = [(100, 800),     # n_estimators
                 (2, 12),        # max_depth
                 (-2.5, -0.5),   # log10 learning_rate
                 (0.5, 1.0),     # subsample
                 (0.5, 1.0),     # colsample_bytree
                 (1, 20),        # min_child_weight
                 (-1.0, 2.0),    # log10 reg_lambda
                 (0.0, 5.0)]     # gamma

def xgb_de_cv_error(params):
    model = XGBRegressor(
        n_estimators=int(round(params[0])),
        max_depth=int(round(params[1])),
        learning_rate=float(10.0 ** params[2]),
        subsample=float(params[3]),
        colsample_bytree=float(params[4]),
        min_child_weight=int(round(params[5])),
        reg_lambda=float(10.0 ** params[6]),
        gamma=float(params[7]),
        random_state=SEED, n_jobs=1)
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

xgb_de_best, xgb_de_err, xgb_de_hist = differential_evolution_optimizer(
    xgb_de_cv_error, bounds=xgb_de_bounds, n_agents=10, n_iter=30, seed=SEED)

xgb_de_n_est      = int(round(xgb_de_best[0]))
xgb_de_depth      = int(round(xgb_de_best[1]))
xgb_de_lr         = float(10.0 ** xgb_de_best[2])
xgb_de_subsample  = float(xgb_de_best[3])
xgb_de_colsample  = float(xgb_de_best[4])
xgb_de_min_child  = int(round(xgb_de_best[5]))
xgb_de_reg_lambda = float(10.0 ** xgb_de_best[6])
xgb_de_gamma      = float(xgb_de_best[7])

print(f"best XGBoost-DE:  n_estimators = {xgb_de_n_est}   max_depth = {xgb_de_depth}   "
      f"learning_rate = {xgb_de_lr:.4f}   subsample = {xgb_de_subsample:.3f}   "
      f"colsample = {xgb_de_colsample:.3f}   min_child_weight = {xgb_de_min_child}   "
      f"reg_lambda = {xgb_de_reg_lambda:.3f}   gamma = {xgb_de_gamma:.3f}")
print(f"best CV R2 during search = {1 - xgb_de_err:.4f}")

In [ ]:
def build_xgb_de_tuned():
    return XGBRegressor(n_estimators=xgb_de_n_est, max_depth=xgb_de_depth,
                        learning_rate=xgb_de_lr, subsample=xgb_de_subsample,
                        colsample_bytree=xgb_de_colsample,
                        min_child_weight=xgb_de_min_child,
                        reg_lambda=xgb_de_reg_lambda, gamma=xgb_de_gamma,
                        random_state=SEED)

res_xgb_de, xgb_de_final = evaluate(build_xgb_de_tuned)
report(res_xgb_de, 'XGBoost-DE')

In [ ]:
import os
import joblib
import sklearn
import xgboost
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

xgb_de_full = build_xgb_de_tuned()
xgb_de_full.fit(X, y)

xgb_de_shap_bundle = {
    "model_name": "XGBoost",
    "optimization_method": "Differential Evolution",

    # xgb_de_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. xgb_de_full is refit on all rows.
    "model": xgb_de_final,
    "model_full_data": xgb_de_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": xgb_de_final.predict(X_test),
    "y_dev_pred": xgb_de_final.predict(X_dev),

    # Deterministic background sample for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "n_estimators": xgb_de_n_est,
        "max_depth": xgb_de_depth,
        "learning_rate": xgb_de_lr,
        "subsample": xgb_de_subsample,
        "colsample_bytree": xgb_de_colsample,
        "min_child_weight": xgb_de_min_child,
        "reg_lambda": xgb_de_reg_lambda,
        "gamma": xgb_de_gamma,
        "random_state": SEED,
    },

    # Optimizer information
    "search_bounds": xgb_de_bounds,
    "best_search_position": np.asarray(xgb_de_best),
    "best_search_error": xgb_de_err,
    "best_search_cv_r2": 1 - xgb_de_err,
    "optimization_history": np.asarray(xgb_de_hist),
    "optimizer_settings": {"n_agents": 10, "n_iterations": 30, "seed": SEED,
                           "F": 0.8, "CR": 0.9, "strategy": "DE/rand/1/bin"},

    # Fold-level performance
    "cv_results": res_xgb_de.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "xgboost_version": xgboost.__version__,
    "saved_at": datetime.now().isoformat(),
}

XGB_DE_SAVE_PATH = os.path.join(SAVE_DIR, "XGBoost_DE_Bundle.joblib")
joblib.dump(xgb_de_shap_bundle, XGB_DE_SAVE_PATH, compress=3)

chk = joblib.load(XGB_DE_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("XGBoost-DE bundle saved successfully.")
print("File:", XGB_DE_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_xgb_de.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(XGB_DE_SAVE_PATH) / (1024**2), 2), "MB")

## **SVR_DE_tunning**

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to SVR-GWO, SVR-GFO, and SVR-WOA
svr_de_bounds = [(-2, 4),    # log10 C       -> [0.01, 10000]
                 (-5, 1),    # log10 gamma   -> [1e-5, 10]
                 (-3, 0)]    # log10 epsilon -> [0.001, 1]

def svr_de_cv_error(params):
    model = TransformedTargetRegressor(
        regressor=make_pipeline(
            StandardScaler(),
            SVR(kernel='rbf',
                C=float(10.0 ** params[0]),
                gamma=float(10.0 ** params[1]),
                epsilon=float(10.0 ** params[2]))),
        transformer=StandardScaler())
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

svr_de_best, svr_de_err, svr_de_hist = differential_evolution_optimizer(
    svr_de_cv_error, bounds=svr_de_bounds, n_agents=10, n_iter=30, seed=SEED)

svr_de_C       = float(10.0 ** svr_de_best[0])
svr_de_gamma   = float(10.0 ** svr_de_best[1])
svr_de_epsilon = float(10.0 ** svr_de_best[2])

print(f"best SVR-DE:  C = {svr_de_C:.3f}   gamma = {svr_de_gamma:.4f}   "
      f"epsilon = {svr_de_epsilon:.4f}")
print(f"best CV R2 during search = {1 - svr_de_err:.4f}")

In [ ]:
def build_svr_de_tuned():
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(),
                                SVR(kernel='rbf', C=svr_de_C, gamma=svr_de_gamma,
                                    epsilon=svr_de_epsilon)),
        transformer=StandardScaler())

res_svr_de, svr_de_final = evaluate(build_svr_de_tuned)
report(res_svr_de, 'SVR-DE')

In [ ]:
import os
import joblib
import sklearn
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

svr_de_full = build_svr_de_tuned()
svr_de_full.fit(X, y)

svr_de_shap_bundle = {
    "model_name": "SVR",
    "optimization_method": "Differential Evolution",

    # svr_de_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. svr_de_full is refit on all rows.
    "model": svr_de_final,
    "model_full_data": svr_de_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": svr_de_final.predict(X_test),
    "y_dev_pred": svr_de_final.predict(X_dev),

    # Deterministic background sample, required for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "C": svr_de_C,
        "gamma": svr_de_gamma,
        "epsilon": svr_de_epsilon,
        "kernel": "rbf",
    },

    # Optimizer information
    "search_bounds": svr_de_bounds,
    "best_search_position": np.asarray(svr_de_best),
    "best_search_error": svr_de_err,
    "best_search_cv_r2": 1 - svr_de_err,
    "optimization_history": np.asarray(svr_de_hist),
    "optimizer_settings": {"n_agents": 10, "n_iterations": 30, "seed": SEED,
                           "F": 0.8, "CR": 0.9, "strategy": "DE/rand/1/bin"},

    # Fold-level performance
    "cv_results": res_svr_de.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "saved_at": datetime.now().isoformat(),
}

SVR_DE_SAVE_PATH = os.path.join(SAVE_DIR, "SVR_DE_Bundle.joblib")
joblib.dump(svr_de_shap_bundle, SVR_DE_SAVE_PATH, compress=3)

chk = joblib.load(SVR_DE_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("SVR-DE bundle saved successfully.")
print("File:", SVR_DE_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Background records:", chk["X_background"].shape[0])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_svr_de.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(SVR_DE_SAVE_PATH) / (1024**2), 2), "MB")

## **ANN_DE_tunning**

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to ANN-GWO, ANN-GFO, and ANN-WOA
ann_de_bounds = [(16, 160),     # first hidden width
                 (8, 96),       # second hidden width
                 (-4, 0.5),     # log10 alpha -> ~[1e-4, 3]
                 (-3.5, -1.2)]  # log10 learning_rate_init

def ann_de_cv_error(params):
    net = MLPRegressor(
        hidden_layer_sizes=(int(round(params[0])), int(round(params[1]))),
        activation="relu", solver="adam",
        alpha=float(10.0 ** params[2]),
        learning_rate_init=float(10.0 ** params[3]),
        max_iter=2000, random_state=SEED)
    model = TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), net),
        transformer=StandardScaler())
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

ann_de_best, ann_de_err, ann_de_hist = differential_evolution_optimizer(
    ann_de_cv_error, bounds=ann_de_bounds, n_agents=10, n_iter=30, seed=SEED)

ann_de_l1, ann_de_l2 = int(round(ann_de_best[0])), int(round(ann_de_best[1]))
ann_de_alpha = float(10.0 ** ann_de_best[2])
ann_de_lr    = float(10.0 ** ann_de_best[3])

print(f"best ANN-DE:  layers = ({ann_de_l1}, {ann_de_l2})   "
      f"alpha = {ann_de_alpha:.5f}   lr = {ann_de_lr:.5f}")
print(f"best CV R2 during search = {1 - ann_de_err:.4f}")

In [ ]:
def build_ann_de_tuned():
    net = MLPRegressor(hidden_layer_sizes=(ann_de_l1, ann_de_l2), activation="relu",
                       solver="adam", alpha=ann_de_alpha,
                       learning_rate_init=ann_de_lr, max_iter=2000,
                       random_state=SEED)
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), net),
        transformer=StandardScaler())

res_ann_de, ann_de_final = evaluate(build_ann_de_tuned)
report(res_ann_de, 'ANN-DE')

In [ ]:
import os
import joblib
import sklearn
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

ann_de_full = build_ann_de_tuned()
ann_de_full.fit(X, y)

ann_de_shap_bundle = {
    "model_name": "ANN",
    "optimization_method": "Differential Evolution",

    # ann_de_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. ann_de_full is refit on all rows.
    "model": ann_de_final,
    "model_full_data": ann_de_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": ann_de_final.predict(X_test),
    "y_dev_pred": ann_de_final.predict(X_dev),

    # Deterministic background sample, required for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "hidden_layer_sizes": (ann_de_l1, ann_de_l2),
        "activation": "relu",
        "solver": "adam",
        "alpha": ann_de_alpha,
        "learning_rate_init": ann_de_lr,
        "max_iter": 2000,
        "random_state": SEED,
    },

    # Optimizer information
    "search_bounds": ann_de_bounds,
    "best_search_position": np.asarray(ann_de_best),
    "best_search_error": ann_de_err,
    "best_search_cv_r2": 1 - ann_de_err,
    "optimization_history": np.asarray(ann_de_hist),
    "optimizer_settings": {"n_agents": 10, "n_iterations": 30, "seed": SEED,
                           "F": 0.8, "CR": 0.9, "strategy": "DE/rand/1/bin"},

    # Fold-level performance
    "cv_results": res_ann_de.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "saved_at": datetime.now().isoformat(),
}

ANN_DE_SAVE_PATH = os.path.join(SAVE_DIR, "ANN_DE_Bundle.joblib")
joblib.dump(ann_de_shap_bundle, ANN_DE_SAVE_PATH, compress=3)

chk = joblib.load(ANN_DE_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("ANN-DE bundle saved successfully.")
print("File:", ANN_DE_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Background records:", chk["X_background"].shape[0])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_ann_de.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(ANN_DE_SAVE_PATH) / (1024**2), 2), "MB")

## **CatBoost_DE_tunning**

In [ ]:
import time
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to CatBoost-GWO, CatBoost-GFO, and CatBoost-WOA
cat_de_bounds = [(200, 1600),   # iterations
                 (3, 8),        # depth
                 (-2.5, -0.5),  # log10 learning_rate -> ~[0.003, 0.316]
                 (-0.5, 2.5)]   # log10 l2_leaf_reg   -> ~[0.316, 316]

_n, _t0 = 0, time.time()

def cat_de_cv_error(params):
    global _n
    model = CatBoostRegressor(
        iterations=int(round(params[0])),
        depth=int(round(params[1])),
        learning_rate=float(10.0 ** params[2]),
        l2_leaf_reg=float(10.0 ** params[3]),
        random_state=SEED, verbose=0)             # all 12 threads
    score = 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                                scoring="r2").mean()   # folds sequential
    _n += 1
    if _n % 10 == 0:
        el = (time.time() - _t0) / 60
        print(f"  {_n:3d}/310   {el:5.1f} min elapsed   ETA {el*310/_n:5.0f} min total",
              flush=True)
    return score

cat_de_best, cat_de_err, cat_de_hist = differential_evolution_optimizer(
    cat_de_cv_error, bounds=cat_de_bounds, n_agents=10, n_iter=30, seed=SEED)

cat_de_iter  = int(round(cat_de_best[0]))
cat_de_depth = int(round(cat_de_best[1]))
cat_de_lr    = float(10.0 ** cat_de_best[2])
cat_de_l2    = float(10.0 ** cat_de_best[3])

print(f"\nbest CatBoost-DE:  iterations = {cat_de_iter}   depth = {cat_de_depth}   "
      f"learning_rate = {cat_de_lr:.4f}   l2_leaf_reg = {cat_de_l2:.3f}")
print(f"best CV R2 during search = {1 - cat_de_err:.4f}")

In [ ]:
def build_cat_de_tuned():
    return CatBoostRegressor(iterations=cat_de_iter, depth=cat_de_depth,
                             learning_rate=cat_de_lr, l2_leaf_reg=cat_de_l2,
                             random_state=SEED, verbose=0)

res_cat_de, cat_de_final = evaluate(build_cat_de_tuned)
report(res_cat_de, 'CatBoost-DE')

In [ ]:
import os
import joblib
import sklearn
import catboost
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

cat_de_full = build_cat_de_tuned()
cat_de_full.fit(X, y)

cat_de_shap_bundle = {
    "model_name": "CatBoost",
    "optimization_method": "Differential Evolution",

    # cat_de_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. cat_de_full is refit on all rows.
    "model": cat_de_final,
    "model_full_data": cat_de_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": cat_de_final.predict(X_test),
    "y_dev_pred": cat_de_final.predict(X_dev),

    # Deterministic background sample, available if needed during SHAP analysis
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "iterations": cat_de_iter,
        "depth": cat_de_depth,
        "learning_rate": cat_de_lr,
        "l2_leaf_reg": cat_de_l2,
        "random_state": SEED,
        "verbose": 0,
    },

    # Optimizer information
    "search_bounds": cat_de_bounds,
    "best_search_position": np.asarray(cat_de_best),
    "best_search_error": cat_de_err,
    "best_search_cv_r2": 1 - cat_de_err,
    "optimization_history": np.asarray(cat_de_hist),
    "optimizer_settings": {"n_agents": 10, "n_iterations": 30, "seed": SEED,
                           "F": 0.8, "CR": 0.9, "strategy": "DE/rand/1/bin"},

    # Fold-level performance
    "cv_results": res_cat_de.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "catboost_version": catboost.__version__,
    "saved_at": datetime.now().isoformat(),
}

CAT_DE_SAVE_PATH = os.path.join(SAVE_DIR, "CatBoost_DE_Bundle.joblib")
joblib.dump(cat_de_shap_bundle, CAT_DE_SAVE_PATH, compress=3)

chk = joblib.load(CAT_DE_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("CatBoost-DE bundle saved successfully.")
print("File:", CAT_DE_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Background records:", chk["X_background"].shape[0])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_cat_de.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(CAT_DE_SAVE_PATH) / (1024**2), 2), "MB")

# **Bayesian Optimization**

In [ ]:
# =============================================================================
# Bayesian Optimization (BO)
# Gaussian-process surrogate + Expected Improvement acquisition
# Based on Snoek et al., 2012
# Run once before the model-specific BO tuning cells
# =============================================================================

import warnings
import numpy as np
from scipy.stats import norm
from sklearn.model_selection import cross_val_score
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.exceptions import ConvergenceWarning


def bayesian_optimization(objective, bounds, n_initial=10, n_iter=300,
                          seed=None, n_candidates=2000, xi=0.01, verbose=False):
    """Minimize `objective` over `bounds` using GP-based Bayesian Optimization."""
    seed = int(globals().get("SEED", 42) if seed is None else seed)
    rng = np.random.default_rng(seed)

    bounds = np.asarray(bounds, dtype=float)
    if bounds.ndim != 2 or bounds.shape[1] != 2:
        raise ValueError("bounds must contain (lower, upper) pairs.")
    if np.any(~np.isfinite(bounds)) or np.any(bounds[:, 0] >= bounds[:, 1]):
        raise ValueError("Every bound must be finite and satisfy lower < upper.")
    if n_initial < 3 or n_iter < 1:
        raise ValueError("n_initial must be >= 3 and n_iter must be >= 1.")
    if n_candidates < 100:
        raise ValueError("n_candidates must be >= 100.")
    if xi < 0:
        raise ValueError("xi must be non-negative.")

    lb, ub = bounds[:, 0], bounds[:, 1]
    dim = len(bounds)
    n_initial = int(n_initial)
    n_iter = int(n_iter)
    n_candidates = int(n_candidates)
    bad_fitness = 1e12

    def checked_objective(x):
        try:
            value = float(objective(np.asarray(x, dtype=float)))
            return value if np.isfinite(value) else bad_fitness
        except Exception:
            return bad_fitness

    def to_unit(X):
        return (X - lb) / (ub - lb)

    def from_unit(Z):
        return lb + Z * (ub - lb)

    X = rng.uniform(lb, ub, size=(n_initial, dim))
    y_vals = np.array([checked_objective(x) for x in X], dtype=float)

    best_idx = int(np.argmin(y_vals))
    best_x = X[best_idx].copy()
    best_f = float(y_vals[best_idx])
    history = [best_f]

    for t in range(n_iter):
        X_unit = to_unit(X)

        kernel = (
            ConstantKernel(1.0, (1e-3, 1e3))
            * Matern(length_scale=np.ones(dim), length_scale_bounds=(1e-2, 1e2), nu=2.5)
            + WhiteKernel(noise_level=1e-8, noise_level_bounds=(1e-10, 1e-3))
        )

        gp = GaussianProcessRegressor(
            kernel=kernel,
            normalize_y=True,
            n_restarts_optimizer=3,
            random_state=seed + t
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            gp.fit(X_unit, y_vals)

        Z_cand = rng.random((n_candidates, dim))
        mu, sigma = gp.predict(Z_cand, return_std=True)
        sigma = np.maximum(sigma, 1e-12)

        improvement = best_f - mu - xi
        z = improvement / sigma
        ei = improvement * norm.cdf(z) + sigma * norm.pdf(z)
        ei[sigma <= 1e-12] = 0.0

        x_new = from_unit(Z_cand[int(np.argmax(ei))])
        f_new = checked_objective(x_new)

        X = np.vstack([X, x_new])
        y_vals = np.append(y_vals, f_new)

        if f_new < best_f:
            best_x = x_new.copy()
            best_f = float(f_new)

        history.append(best_f)

        if verbose and (t + 1) % max(1, n_iter // 10) == 0:
            print(f"Iteration {t + 1:4d}/{n_iter} | Best objective = {best_f:.6g}")

    return best_x, best_f, history


# Automatic reproducibility and validity check
def _check_bo():
    sphere = lambda x: float(np.sum(np.asarray(x) ** 2))
    test_bounds = [(-5, 5)] * 4

    x1, f1, h1 = bayesian_optimization(
        sphere, test_bounds, n_initial=8, n_iter=10, seed=123, n_candidates=500
    )
    x2, f2, h2 = bayesian_optimization(
        sphere, test_bounds, n_initial=8, n_iter=10, seed=123, n_candidates=500
    )

    assert np.allclose(x1, x2)
    assert np.isclose(f1, f2)
    assert np.allclose(h1, h2)
    assert len(h1) == 11
    assert np.all(np.diff(h1) <= 1e-12)
    assert np.all((x1 >= -5) & (x1 <= 5))
    assert np.isfinite(f1)

    print("BO implementation check passed.")
    print("best f found:", round(f1, 6))
    print("best x:", np.round(x1, 4))


_check_bo()

## **RF_BO_tunning**

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to RF-GWO, RF-GFO, RF-WOA, and RF-DE
rf_bo_bounds = [(100, 800),    # n_estimators
                (3, 30),       # max_depth
                (0.3, 1.0),    # max_features
                (1, 20),       # min_samples_leaf
                (2, 20)]       # min_samples_split

def rf_bo_cv_error(params):
    model = RandomForestRegressor(
        n_estimators=int(round(params[0])),
        max_depth=int(round(params[1])),
        max_features=float(params[2]),
        min_samples_leaf=int(round(params[3])),
        min_samples_split=int(round(params[4])),
        random_state=SEED, n_jobs=1)
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

rf_bo_best, rf_bo_err, rf_bo_hist = bayesian_optimization(
    rf_bo_cv_error,
    bounds=rf_bo_bounds,
    n_initial=10,
    n_iter=300,
    seed=SEED,
    n_candidates=2000,
    xi=0.01
)

rf_bo_n_est     = int(round(rf_bo_best[0]))
rf_bo_depth     = int(round(rf_bo_best[1]))
rf_bo_max_feat  = float(rf_bo_best[2])
rf_bo_min_leaf  = int(round(rf_bo_best[3]))
rf_bo_min_split = int(round(rf_bo_best[4]))

print(f"best RF-BO:  n_estimators = {rf_bo_n_est}   max_depth = {rf_bo_depth}   "
      f"max_features = {rf_bo_max_feat:.3f}   min_samples_leaf = {rf_bo_min_leaf}   "
      f"min_samples_split = {rf_bo_min_split}")
print(f"best CV R2 during search = {1 - rf_bo_err:.4f}")

In [ ]:
def build_rf_bo_tuned():
    return RandomForestRegressor(
        n_estimators=rf_bo_n_est,
        max_depth=rf_bo_depth,
        max_features=rf_bo_max_feat,
        min_samples_leaf=rf_bo_min_leaf,
        min_samples_split=rf_bo_min_split,
        random_state=SEED)

res_rf_bo, rf_bo_final = evaluate(build_rf_bo_tuned)
report(res_rf_bo, 'RF-BO')

In [ ]:
import os
import joblib
import sklearn
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

rf_bo_full = build_rf_bo_tuned()
rf_bo_full.fit(X, y)

rf_bo_shap_bundle = {
    "model_name": "Random Forest",
    "optimization_method": "Bayesian Optimization",

    # rf_bo_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. rf_bo_full is refit on all rows.
    "model": rf_bo_final,
    "model_full_data": rf_bo_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": rf_bo_final.predict(X_test),
    "y_dev_pred": rf_bo_final.predict(X_dev),

    # Deterministic background sample for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "n_estimators": rf_bo_n_est,
        "max_depth": rf_bo_depth,
        "max_features": rf_bo_max_feat,
        "min_samples_leaf": rf_bo_min_leaf,
        "min_samples_split": rf_bo_min_split,
        "random_state": SEED,
    },

    # Optimizer information
    "search_bounds": rf_bo_bounds,
    "best_search_position": np.asarray(rf_bo_best),
    "best_search_error": rf_bo_err,
    "best_search_cv_r2": 1 - rf_bo_err,
    "optimization_history": np.asarray(rf_bo_hist),
    "optimizer_settings": {
        "n_initial": 10,
        "n_iterations": 300,
        "n_candidates": 2000,
        "xi": 0.01,
        "seed": SEED,
        "surrogate_model": "Gaussian Process",
        "acquisition_function": "Expected Improvement",
        "kernel": "ConstantKernel * Matern(nu=2.5) + WhiteKernel",
    },

    # Fold-level performance
    "cv_results": res_rf_bo.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "saved_at": datetime.now().isoformat(),
}

RF_BO_SAVE_PATH = os.path.join(SAVE_DIR, "RF_BO_Bundle.joblib")
joblib.dump(rf_bo_shap_bundle, RF_BO_SAVE_PATH, compress=3)

chk = joblib.load(RF_BO_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("RF-BO bundle saved successfully.")
print("File:", RF_BO_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_rf_bo.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(RF_BO_SAVE_PATH) / (1024**2), 2), "MB")

## **XGB_BO_tunning**

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to XGBoost-GWO, XGBoost-GFO, XGBoost-WOA, and XGBoost-DE
xgb_bo_bounds = [(100, 800),     # n_estimators
                 (2, 12),        # max_depth
                 (-2.5, -0.5),   # log10 learning_rate
                 (0.5, 1.0),     # subsample
                 (0.5, 1.0),     # colsample_bytree
                 (1, 20),        # min_child_weight
                 (-1.0, 2.0),    # log10 reg_lambda
                 (0.0, 5.0)]     # gamma

def xgb_bo_cv_error(params):
    model = XGBRegressor(
        n_estimators=int(round(params[0])),
        max_depth=int(round(params[1])),
        learning_rate=float(10.0 ** params[2]),
        subsample=float(params[3]),
        colsample_bytree=float(params[4]),
        min_child_weight=int(round(params[5])),
        reg_lambda=float(10.0 ** params[6]),
        gamma=float(params[7]),
        random_state=SEED, n_jobs=1)
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

xgb_bo_best, xgb_bo_err, xgb_bo_hist = bayesian_optimization(
    xgb_bo_cv_error, bounds=xgb_bo_bounds, n_initial=10, n_iter=300,
    seed=SEED, n_candidates=2000, xi=0.01)

xgb_bo_n_est      = int(round(xgb_bo_best[0]))
xgb_bo_depth      = int(round(xgb_bo_best[1]))
xgb_bo_lr         = float(10.0 ** xgb_bo_best[2])
xgb_bo_subsample  = float(xgb_bo_best[3])
xgb_bo_colsample  = float(xgb_bo_best[4])
xgb_bo_min_child  = int(round(xgb_bo_best[5]))
xgb_bo_reg_lambda = float(10.0 ** xgb_bo_best[6])
xgb_bo_gamma      = float(xgb_bo_best[7])

print(f"best XGBoost-BO:  n_estimators = {xgb_bo_n_est}   max_depth = {xgb_bo_depth}   "
      f"learning_rate = {xgb_bo_lr:.4f}   subsample = {xgb_bo_subsample:.3f}   "
      f"colsample = {xgb_bo_colsample:.3f}   min_child_weight = {xgb_bo_min_child}   "
      f"reg_lambda = {xgb_bo_reg_lambda:.3f}   gamma = {xgb_bo_gamma:.3f}")
print(f"best CV R2 during search = {1 - xgb_bo_err:.4f}")

In [ ]:
def build_xgb_bo_tuned():
    return XGBRegressor(n_estimators=xgb_bo_n_est, max_depth=xgb_bo_depth,
                        learning_rate=xgb_bo_lr, subsample=xgb_bo_subsample,
                        colsample_bytree=xgb_bo_colsample,
                        min_child_weight=xgb_bo_min_child,
                        reg_lambda=xgb_bo_reg_lambda, gamma=xgb_bo_gamma,
                        random_state=SEED)

res_xgb_bo, xgb_bo_final = evaluate(build_xgb_bo_tuned)
report(res_xgb_bo, 'XGBoost-BO')

In [ ]:
import os
import joblib
import sklearn
import xgboost
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

xgb_bo_full = build_xgb_bo_tuned()
xgb_bo_full.fit(X, y)

xgb_bo_shap_bundle = {
    "model_name": "XGBoost",
    "optimization_method": "Bayesian Optimization",

    # xgb_bo_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. xgb_bo_full is refit on all rows.
    "model": xgb_bo_final,
    "model_full_data": xgb_bo_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": xgb_bo_final.predict(X_test),
    "y_dev_pred": xgb_bo_final.predict(X_dev),

    # Deterministic background sample for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "n_estimators": xgb_bo_n_est,
        "max_depth": xgb_bo_depth,
        "learning_rate": xgb_bo_lr,
        "subsample": xgb_bo_subsample,
        "colsample_bytree": xgb_bo_colsample,
        "min_child_weight": xgb_bo_min_child,
        "reg_lambda": xgb_bo_reg_lambda,
        "gamma": xgb_bo_gamma,
        "random_state": SEED,
    },

    # Optimizer information
    "search_bounds": xgb_bo_bounds,
    "best_search_position": np.asarray(xgb_bo_best),
    "best_search_error": xgb_bo_err,
    "best_search_cv_r2": 1 - xgb_bo_err,
    "optimization_history": np.asarray(xgb_bo_hist),
    "optimizer_settings": {
        "n_initial": 10,
        "n_iterations": 300,
        "n_candidates": 2000,
        "xi": 0.01,
        "seed": SEED,
        "surrogate_model": "Gaussian Process",
        "acquisition_function": "Expected Improvement",
        "kernel": "ConstantKernel * Matern(nu=2.5) + WhiteKernel",
    },

    # Fold-level performance
    "cv_results": res_xgb_bo.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "xgboost_version": xgboost.__version__,
    "saved_at": datetime.now().isoformat(),
}

XGB_BO_SAVE_PATH = os.path.join(SAVE_DIR, "XGBoost_BO_Bundle.joblib")
joblib.dump(xgb_bo_shap_bundle, XGB_BO_SAVE_PATH, compress=3)

chk = joblib.load(XGB_BO_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("XGBoost-BO bundle saved successfully.")
print("File:", XGB_BO_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_xgb_bo.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(XGB_BO_SAVE_PATH) / (1024**2), 2), "MB")

## **SVR_BO_tunning**

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to SVR-GWO, SVR-GFO, SVR-WOA, and SVR-DE
svr_bo_bounds = [(-2, 4),    # log10 C       -> [0.01, 10000]
                 (-5, 1),    # log10 gamma   -> [1e-5, 10]
                 (-3, 0)]    # log10 epsilon -> [0.001, 1]

def svr_bo_cv_error(params):
    model = TransformedTargetRegressor(
        regressor=make_pipeline(
            StandardScaler(),
            SVR(kernel='rbf',
                C=float(10.0 ** params[0]),
                gamma=float(10.0 ** params[1]),
                epsilon=float(10.0 ** params[2]))),
        transformer=StandardScaler())
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

svr_bo_best, svr_bo_err, svr_bo_hist = bayesian_optimization(
    svr_bo_cv_error, bounds=svr_bo_bounds, n_initial=10, n_iter=300,
    seed=SEED, n_candidates=2000, xi=0.01)

svr_bo_C       = float(10.0 ** svr_bo_best[0])
svr_bo_gamma   = float(10.0 ** svr_bo_best[1])
svr_bo_epsilon = float(10.0 ** svr_bo_best[2])

print(f"best SVR-BO:  C = {svr_bo_C:.3f}   gamma = {svr_bo_gamma:.4f}   "
      f"epsilon = {svr_bo_epsilon:.4f}")
print(f"best CV R2 during search = {1 - svr_bo_err:.4f}")

In [ ]:
def build_svr_bo_tuned():
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(),
                                SVR(kernel='rbf', C=svr_bo_C, gamma=svr_bo_gamma,
                                    epsilon=svr_bo_epsilon)),
        transformer=StandardScaler())

res_svr_bo, svr_bo_final = evaluate(build_svr_bo_tuned)
report(res_svr_bo, 'SVR-BO')

In [ ]:
import os
import joblib
import sklearn
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

svr_bo_full = build_svr_bo_tuned()
svr_bo_full.fit(X, y)

svr_bo_shap_bundle = {
    "model_name": "SVR",
    "optimization_method": "Bayesian Optimization",

    # svr_bo_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. svr_bo_full is refit on all rows.
    "model": svr_bo_final,
    "model_full_data": svr_bo_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": svr_bo_final.predict(X_test),
    "y_dev_pred": svr_bo_final.predict(X_dev),

    # Deterministic background sample, required for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "C": svr_bo_C,
        "gamma": svr_bo_gamma,
        "epsilon": svr_bo_epsilon,
        "kernel": "rbf",
    },

    # Optimizer information
    "search_bounds": svr_bo_bounds,
    "best_search_position": np.asarray(svr_bo_best),
    "best_search_error": svr_bo_err,
    "best_search_cv_r2": 1 - svr_bo_err,
    "optimization_history": np.asarray(svr_bo_hist),
    "optimizer_settings": {
        "n_initial": 10,
        "n_iterations": 300,
        "n_candidates": 2000,
        "xi": 0.01,
        "seed": SEED,
        "surrogate_model": "Gaussian Process",
        "acquisition_function": "Expected Improvement",
        "kernel": "ConstantKernel * Matern(nu=2.5) + WhiteKernel",
    },

    # Fold-level performance
    "cv_results": res_svr_bo.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "saved_at": datetime.now().isoformat(),
}

SVR_BO_SAVE_PATH = os.path.join(SAVE_DIR, "SVR_BO_Bundle.joblib")
joblib.dump(svr_bo_shap_bundle, SVR_BO_SAVE_PATH, compress=3)

chk = joblib.load(SVR_BO_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("SVR-BO bundle saved successfully.")
print("File:", SVR_BO_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Background records:", chk["X_background"].shape[0])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_svr_bo.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(SVR_BO_SAVE_PATH) / (1024**2), 2), "MB")

## **ANN_BO_tunning**

In [ ]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to ANN-GWO, ANN-GFO, ANN-WOA, and ANN-DE
ann_bo_bounds = [(16, 160),     # first hidden width
                 (8, 96),       # second hidden width
                 (-4, 0.5),     # log10 alpha -> ~[1e-4, 3]
                 (-3.5, -1.2)]  # log10 learning_rate_init

def ann_bo_cv_error(params):
    net = MLPRegressor(
        hidden_layer_sizes=(int(round(params[0])), int(round(params[1]))),
        activation="relu", solver="adam",
        alpha=float(10.0 ** params[2]),
        learning_rate_init=float(10.0 ** params[3]),
        max_iter=2000, random_state=SEED)
    model = TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), net),
        transformer=StandardScaler())
    return 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                               scoring="r2", n_jobs=-1).mean()

ann_bo_best, ann_bo_err, ann_bo_hist = bayesian_optimization(
    ann_bo_cv_error, bounds=ann_bo_bounds, n_initial=10, n_iter=300,
    seed=SEED, n_candidates=2000, xi=0.01)

ann_bo_l1, ann_bo_l2 = int(round(ann_bo_best[0])), int(round(ann_bo_best[1]))
ann_bo_alpha = float(10.0 ** ann_bo_best[2])
ann_bo_lr    = float(10.0 ** ann_bo_best[3])

print(f"best ANN-BO:  layers = ({ann_bo_l1}, {ann_bo_l2})   "
      f"alpha = {ann_bo_alpha:.5f}   lr = {ann_bo_lr:.5f}")
print(f"best CV R2 during search = {1 - ann_bo_err:.4f}")

In [ ]:
def build_ann_bo_tuned():
    net = MLPRegressor(hidden_layer_sizes=(ann_bo_l1, ann_bo_l2), activation="relu",
                       solver="adam", alpha=ann_bo_alpha,
                       learning_rate_init=ann_bo_lr, max_iter=2000,
                       random_state=SEED)
    return TransformedTargetRegressor(
        regressor=make_pipeline(StandardScaler(), net),
        transformer=StandardScaler())

res_ann_bo, ann_bo_final = evaluate(build_ann_bo_tuned)
report(res_ann_bo, 'ANN-BO')

In [ ]:
import os
import joblib
import sklearn
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

ann_bo_full = build_ann_bo_tuned()
ann_bo_full.fit(X, y)

ann_bo_shap_bundle = {
    "model_name": "ANN",
    "optimization_method": "Bayesian Optimization",

    # ann_bo_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. ann_bo_full is refit on all rows.
    "model": ann_bo_final,
    "model_full_data": ann_bo_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": ann_bo_final.predict(X_test),
    "y_dev_pred": ann_bo_final.predict(X_dev),

    # Deterministic background sample, required for model-agnostic SHAP
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "hidden_layer_sizes": (ann_bo_l1, ann_bo_l2),
        "activation": "relu",
        "solver": "adam",
        "alpha": ann_bo_alpha,
        "learning_rate_init": ann_bo_lr,
        "max_iter": 2000,
        "random_state": SEED,
    },

    # Optimizer information
    "search_bounds": ann_bo_bounds,
    "best_search_position": np.asarray(ann_bo_best),
    "best_search_error": ann_bo_err,
    "best_search_cv_r2": 1 - ann_bo_err,
    "optimization_history": np.asarray(ann_bo_hist),
    "optimizer_settings": {
        "n_initial": 10,
        "n_iterations": 300,
        "n_candidates": 2000,
        "xi": 0.01,
        "seed": SEED,
        "surrogate_model": "Gaussian Process",
        "acquisition_function": "Expected Improvement",
        "kernel": "ConstantKernel * Matern(nu=2.5) + WhiteKernel",
    },

    # Fold-level performance
    "cv_results": res_ann_bo.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "saved_at": datetime.now().isoformat(),
}

ANN_BO_SAVE_PATH = os.path.join(SAVE_DIR, "ANN_BO_Bundle.joblib")
joblib.dump(ann_bo_shap_bundle, ANN_BO_SAVE_PATH, compress=3)

chk = joblib.load(ANN_BO_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("ANN-BO bundle saved successfully.")
print("File:", ANN_BO_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Background records:", chk["X_background"].shape[0])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_ann_bo.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(ANN_BO_SAVE_PATH) / (1024**2), 2), "MB")

## **CatBoost_BO_tunning**

In [ ]:
import time
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_val_score

# Identical search space to CatBoost-GWO, CatBoost-GFO, CatBoost-WOA, and CatBoost-DE
cat_bo_bounds = [(200, 1600),   # iterations
                 (3, 8),        # depth
                 (-2.5, -0.5),  # log10 learning_rate -> ~[0.003, 0.316]
                 (-0.5, 2.5)]   # log10 l2_leaf_reg   -> ~[0.316, 316]

_n, _t0 = 0, time.time()

def cat_bo_cv_error(params):
    global _n
    model = CatBoostRegressor(
        iterations=int(round(params[0])),
        depth=int(round(params[1])),
        learning_rate=float(10.0 ** params[2]),
        l2_leaf_reg=float(10.0 ** params[3]),
        random_state=SEED, verbose=0)             # all 12 threads
    score = 1 - cross_val_score(model, X_dev, y_dev, cv=cv_splits,
                                scoring="r2").mean()   # folds sequential
    _n += 1
    if _n % 10 == 0:
        el = (time.time() - _t0) / 60
        print(f"  {_n:3d}/310   {el:5.1f} min elapsed   ETA {el*310/_n:5.0f} min total",
              flush=True)
    return score

cat_bo_best, cat_bo_err, cat_bo_hist = bayesian_optimization(
    cat_bo_cv_error, bounds=cat_bo_bounds, n_initial=10, n_iter=300,
    seed=SEED, n_candidates=2000, xi=0.01)

cat_bo_iter  = int(round(cat_bo_best[0]))
cat_bo_depth = int(round(cat_bo_best[1]))
cat_bo_lr    = float(10.0 ** cat_bo_best[2])
cat_bo_l2    = float(10.0 ** cat_bo_best[3])

print(f"\nbest CatBoost-BO:  iterations = {cat_bo_iter}   depth = {cat_bo_depth}   "
      f"learning_rate = {cat_bo_lr:.4f}   l2_leaf_reg = {cat_bo_l2:.3f}")
print(f"best CV R2 during search = {1 - cat_bo_err:.4f}")

In [ ]:
def build_cat_bo_tuned():
    return CatBoostRegressor(iterations=cat_bo_iter, depth=cat_bo_depth,
                             learning_rate=cat_bo_lr, l2_leaf_reg=cat_bo_l2,
                             random_state=SEED, verbose=0)

res_cat_bo, cat_bo_final = evaluate(build_cat_bo_tuned)
report(res_cat_bo, 'CatBoost-BO')

In [ ]:
import os
import joblib
import sklearn
import catboost
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
os.makedirs(SAVE_DIR, exist_ok=True)

cat_bo_full = build_cat_bo_tuned()
cat_bo_full.fit(X, y)

cat_bo_shap_bundle = {
    "model_name": "CatBoost",
    "optimization_method": "Bayesian Optimization",

    # cat_bo_final is fitted on the development set only -> the model whose
    # reported CV and held-out metrics are valid. cat_bo_full is refit on all rows.
    "model": cat_bo_final,
    "model_full_data": cat_bo_full,

    # Data and split definition
    "X": X.copy(),
    "y": y.copy(),
    "feature_names": list(X.columns),
    "target_name": y.name,
    "dev_index": np.asarray(dev_idx),
    "test_index": np.asarray(test_idx),
    "cv_splits": [(np.asarray(tr), np.asarray(te)) for tr, te in cv_splits],
    "cv_method": repr(kf),

    # Held-out predictions, for scatter plots, residuals and conformal calibration
    "y_test": y_test.copy(),
    "y_test_pred": cat_bo_final.predict(X_test),
    "y_dev_pred": cat_bo_final.predict(X_dev),

    # Deterministic background sample, available if needed during SHAP analysis
    "X_background": X.sample(n=min(100, len(X)), random_state=SEED).copy(),

    # Best hyperparameters
    "best_parameters": {
        "iterations": cat_bo_iter,
        "depth": cat_bo_depth,
        "learning_rate": cat_bo_lr,
        "l2_leaf_reg": cat_bo_l2,
        "random_state": SEED,
        "verbose": 0,
    },

    # Optimizer information
    "search_bounds": cat_bo_bounds,
    "best_search_position": np.asarray(cat_bo_best),
    "best_search_error": cat_bo_err,
    "best_search_cv_r2": 1 - cat_bo_err,
    "optimization_history": np.asarray(cat_bo_hist),
    "optimizer_settings": {
        "n_initial": 10,
        "n_iterations": 300,
        "n_candidates": 2000,
        "xi": 0.01,
        "seed": SEED,
        "surrogate_model": "Gaussian Process",
        "acquisition_function": "Expected Improvement",
        "kernel": "ConstantKernel * Matern(nu=2.5) + WhiteKernel",
    },

    # Fold-level performance
    "cv_results": res_cat_bo.copy(),

    # Reproducibility
    "random_seed": SEED,
    "sklearn_version": sklearn.__version__,
    "catboost_version": catboost.__version__,
    "saved_at": datetime.now().isoformat(),
}

CAT_BO_SAVE_PATH = os.path.join(SAVE_DIR, "CatBoost_BO_Bundle.joblib")
joblib.dump(cat_bo_shap_bundle, CAT_BO_SAVE_PATH, compress=3)

chk = joblib.load(CAT_BO_SAVE_PATH)
r2_check = r2_score(chk["y_test"], chk["model"].predict(chk["X"].iloc[chk["test_index"]]))

print("CatBoost-BO bundle saved successfully.")
print("File:", CAT_BO_SAVE_PATH)
print("Optimizer:", chk["optimization_method"])
print("Records:", chk["X"].shape[0], "| Features:", chk["X"].shape[1])
print("Background records:", chk["X_background"].shape[0])
print("Best parameters:", chk["best_parameters"])
print(f"Held-out R2 from bundle: {r2_check:.4f}  "
      f"(reported: {res_cat_bo.loc['Holdout', 'val_R2']:.4f})")
print("File size:", round(os.path.getsize(CAT_BO_SAVE_PATH) / (1024**2), 2), "MB")

# **Performance Metrics and Errors**

## **Summary Table**

In [ ]:
# =============================================================================
# Models Performance — Load saved 25 model-optimizer bundles and build summary table
# =============================================================================

import os
import joblib
import numpy as np
import pandas as pd

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"

bundle_paths = {
    ("RF", "GWO"):       "RF_GWO_Bundle.joblib",
    ("XGBoost", "GWO"):  "XGBoost_GWO_Bundle.joblib",
    ("SVR", "GWO"):      "SVR_GWO_Bundle.joblib",
    ("ANN", "GWO"):      "ANN_GWO_Bundle.joblib",
    ("CatBoost", "GWO"): "CatBoost_GWO_Bundle.joblib",

    ("RF", "GFO"):       "RF_GFO_Bundle.joblib",
    ("XGBoost", "GFO"):  "XGBoost_GFO_Bundle.joblib",
    ("SVR", "GFO"):      "SVR_GFO_Bundle.joblib",
    ("ANN", "GFO"):      "ANN_GFO_Bundle.joblib",
    ("CatBoost", "GFO"): "CatBoost_GFO_Bundle.joblib",

    ("RF", "WOA"):       "RF_WOA_Bundle.joblib",
    ("XGBoost", "WOA"):  "XGBoost_WOA_Bundle.joblib",
    ("SVR", "WOA"):      "SVR_WOA_Bundle.joblib",
    ("ANN", "WOA"):      "ANN_WOA_Bundle.joblib",
    ("CatBoost", "WOA"): "CatBoost_WOA_Bundle.joblib",

    ("RF", "DE"):        "RF_DE_Bundle.joblib",
    ("XGBoost", "DE"):   "XGBoost_DE_Bundle.joblib",
    ("SVR", "DE"):       "SVR_DE_Bundle.joblib",
    ("ANN", "DE"):       "ANN_DE_Bundle.joblib",
    ("CatBoost", "DE"):  "CatBoost_DE_Bundle.joblib",

    ("RF", "BO"):        "RF_BO_Bundle.joblib",
    ("XGBoost", "BO"):   "XGBoost_BO_Bundle.joblib",
    ("SVR", "BO"):       "SVR_BO_Bundle.joblib",
    ("ANN", "BO"):       "ANN_BO_Bundle.joblib",
    ("CatBoost", "BO"):  "CatBoost_BO_Bundle.joblib",
}

required_cols = ["train_MAE", "train_RMSE", "train_R2", "val_MAE", "val_RMSE", "val_R2"]
summary_labels = ["CV mean", "CV std", "Holdout"]

summary_rows = []
fold_rows = []
missing_files = []

for (model_name, optimizer_name), file_name in bundle_paths.items():
    path = os.path.join(SAVE_DIR, file_name)

    if not os.path.exists(path):
        missing_files.append(path)
        continue

    bundle = joblib.load(path)

    if "cv_results" not in bundle:
        raise KeyError(f"`cv_results` not found in bundle: {path}")

    cv = bundle["cv_results"].copy()

    for col in required_cols:
        if col not in cv.columns:
            raise KeyError(f"Column `{col}` not found in cv_results for {model_name}-{optimizer_name}")

    # fold rows only; the last three rows are the CV mean, CV std and held-out summary
    folds = cv.drop(index=[i for i in summary_labels if i in cv.index])[required_cols]

    folds_out = folds.copy()
    folds_out["Model"] = model_name
    folds_out["Optimizer"] = optimizer_name
    folds_out["Model-Optimizer"] = f"{model_name}-{optimizer_name}"
    fold_rows.append(folds_out.reset_index())

    means = folds.mean()
    stds = folds.std()

    # held-out metrics are stored in the val_ columns of the Holdout row
    holdout = cv.loc["Holdout"]

    summary_rows.append({
        "Model": model_name,
        "Optimizer": optimizer_name,
        "Model-Optimizer": f"{model_name}-{optimizer_name}",

        "Train R2 mean": means["train_R2"],
        "Train R2 std": stds["train_R2"],
        "Train RMSE mean": means["train_RMSE"],
        "Train RMSE std": stds["train_RMSE"],
        "Train MAE mean": means["train_MAE"],
        "Train MAE std": stds["train_MAE"],

        "Val R2 mean": means["val_R2"],
        "Val R2 std": stds["val_R2"],
        "Val RMSE mean": means["val_RMSE"],
        "Val RMSE std": stds["val_RMSE"],
        "Val MAE mean": means["val_MAE"],
        "Val MAE std": stds["val_MAE"],

        "Test R2": holdout["val_R2"],
        "Test RMSE": holdout["val_RMSE"],
        "Test MAE": holdout["val_MAE"],
    })

if missing_files:
    print("Missing saved bundles:")
    for p in missing_files:
        print(" -", p)

assert len(summary_rows) > 0, "No saved bundles were loaded."

model_optimizer_summary = pd.DataFrame(summary_rows)
model_optimizer_folds = pd.concat(fold_rows, ignore_index=True)

model_order = ["RF", "XGBoost", "SVR", "ANN", "CatBoost"]
optimizer_order = ["GWO", "GFO", "WOA", "DE", "BO"]

model_optimizer_summary["Model"] = pd.Categorical(
    model_optimizer_summary["Model"], categories=model_order, ordered=True)

model_optimizer_summary["Optimizer"] = pd.Categorical(
    model_optimizer_summary["Optimizer"], categories=optimizer_order, ordered=True)

model_optimizer_summary = (
    model_optimizer_summary
    .sort_values(["Model", "Optimizer"])
    .reset_index(drop=True)
)

# Paper-ready formatted table
model_optimizer_table = model_optimizer_summary.copy()

for label, mean_col, std_col in [
    ("Train R2",  "Train R2 mean",  "Train R2 std"),
    ("Val R2",    "Val R2 mean",    "Val R2 std"),
    ("Val RMSE",  "Val RMSE mean",  "Val RMSE std"),
    ("Val MAE",   "Val MAE mean",   "Val MAE std"),
]:
    model_optimizer_table[label] = (
        model_optimizer_table[mean_col].map(lambda x: f"{x:.4f}") + " ± " +
        model_optimizer_table[std_col].map(lambda x: f"{x:.4f}")
    )

for label in ["Test R2", "Test RMSE", "Test MAE"]:
    model_optimizer_table[label] = model_optimizer_table[label].map(lambda x: f"{x:.4f}")

display_cols = [
    "Model", "Optimizer",
    "Train R2",
    "Val R2", "Val RMSE", "Val MAE",
    "Test R2", "Test RMSE", "Test MAE",
]

model_optimizer_table = model_optimizer_table[display_cols]

display(model_optimizer_table)

print()
print("Loaded model-optimizer combinations:", len(model_optimizer_summary))
print("Train and validation reported as fold mean ± std over 5 folds | Test is the single held-out 20%")

## **Friedman Test**

In [ ]:
# =============================================================================
# Statistical comparison — Friedman + Nemenyi on stored fold-level validation R2
# No refitting; reads the 25 tuned bundles from Drive
# =============================================================================
!pip install -q scikit-posthocs

import os, glob, joblib
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
ALPHA = 0.05

MODEL_LABEL = {"Random Forest": "RF", "XGBoost": "XGB", "SVR": "SVR",
               "ANN": "ANN", "CatBoost": "CatBoost"}
OPT_LABEL = {"Grey Wolf Optimizer": "GWO", "Galactic Field Optimization": "GFO",
             "Whale Optimization Algorithm": "WOA", "Differential Evolution": "DE",
             "Bayesian Optimization": "BO"}
MODEL_ORDER = ["RF", "XGB", "SVR", "ANN", "CatBoost"]
OPT_ORDER = ["GWO", "GFO", "WOA", "DE", "BO"]

# ---- load fold-level validation R2, verifying every bundle shares the same splits ----
scores, ref = {}, None
for path in sorted(glob.glob(os.path.join(SAVE_DIR, "*_Bundle.joblib"))):
    b = joblib.load(path)
    if b.get("model_name") not in MODEL_LABEL:
        continue
    if ref is None:
        ref = (np.asarray(b["dev_index"]), np.asarray(b["test_index"]),
               [(np.asarray(tr), np.asarray(te)) for tr, te in b["cv_splits"]])
    else:
        assert np.array_equal(np.asarray(b["dev_index"]), ref[0]), f"dev split differs: {path}"
        assert np.array_equal(np.asarray(b["test_index"]), ref[1]), f"test split differs: {path}"
        for (tr_r, te_r), (tr_c, te_c) in zip(ref[2], b["cv_splits"]):
            assert np.array_equal(np.asarray(tr_c), tr_r) and \
                   np.array_equal(np.asarray(te_c), te_r), f"CV folds differ: {path}"
    key = (MODEL_LABEL[b["model_name"]], OPT_LABEL[b["optimization_method"]])
    scores[key] = b["cv_results"].loc[range(1, N_FOLDS + 1), "val_R2"].astype(float).to_numpy()

assert len(scores) == 25, f"Expected 25 combinations, found {len(scores)}"
print(f"Loaded {len(scores)} combinations | identical dev/test split and {N_FOLDS} CV folds confirmed\n")

fold_r2 = pd.DataFrame({f"{m}-{o}": v for (m, o), v in scores.items()},
                       index=[f"Fold {k}" for k in range(1, N_FOLDS + 1)])

# ---- per-fold ranks (1 = best) with the average rank as the final column ----
ranks = fold_r2.rank(axis=1, ascending=False).T
ranks["Average rank"] = ranks.mean(axis=1)
ranks["Mean val R2"] = fold_r2.mean().values
ranks = ranks.sort_values("Average rank")

print("Per-fold ranks of the 25 combinations (1 = best in that fold):")
display(ranks.round(2))


# ---- reshape to long form for the grouped tests ----
long = (fold_r2.T.rename_axis("Combination").reset_index()
        .melt(id_vars="Combination", var_name="Fold", value_name="R2"))
long[["Model", "Optimizer"]] = long["Combination"].str.split("-", expand=True)

opt_mat = long.pivot_table(index=["Model", "Fold"], columns="Optimizer", values="R2")[OPT_ORDER]
mod_mat = long.pivot_table(index=["Optimizer", "Fold"], columns="Model", values="R2")[MODEL_ORDER]


def friedman(mat, label, note=""):
    stat, p = friedmanchisquare(*[mat[c].to_numpy() for c in mat.columns])
    print(f"\n{'='*70}\n{label}   ({mat.shape[1]} treatments, {mat.shape[0]} blocks){note}")
    print(f"{'='*70}")
    avg = mat.rank(axis=1, ascending=False).mean().sort_values()
    print(pd.DataFrame({"Average rank": avg.round(3),
                        "Mean val R2": mat.mean()[avg.index].round(4)}).to_string())
    print(f"\nFriedman chi2 = {stat:.4f}   p = {p:.6g}   alpha = {ALPHA}")

    if p < ALPHA:
        print("-> reject H0: at least one differs. Nemenyi post-hoc:")
        nem = sp.posthoc_nemenyi_friedman(mat.to_numpy())
        nem.index = nem.columns = mat.columns
        pairs = [(a, b, nem.loc[a, b]) for i, a in enumerate(mat.columns)
                 for b in mat.columns[i + 1:] if nem.loc[a, b] < ALPHA]
        if pairs:
            display(pd.DataFrame(pairs, columns=["A", "B", "Nemenyi p"])
                    .sort_values("Nemenyi p").round(6).reset_index(drop=True))
        else:
            print("   no individual pair significant at alpha = 0.05")
    else:
        print("-> fail to reject H0: no significant difference; Nemenyi not run.")
    return p


# ---- the two well-powered tests, then the 25-way test with its caveat ----
p_opt = friedman(opt_mat, "OPTIMIZER COMPARISON")
p_mod = friedman(mod_mat, "MODEL FAMILY COMPARISON")
p_all = friedman(fold_r2, "ALL 25 COMBINATIONS",
                 note="\n  CAUTION: 5 blocks against 25 treatments is underpowered;"
                      "\n  the chi-square approximation is unreliable at this ratio.")

## **Mix-based Performance**

In [ ]:
# =============================================================================
# Mixture-system performance on the held-out test set — CatBoost-BO
# =============================================================================

import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from matplotlib.lines import Line2D
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from google.colab import files

BUNDLE_PATH = (
    "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper/"
    "CatBoost_BO_Bundle.joblib"
)

assert os.path.exists(BUNDLE_PATH), f"Bundle not found: {BUNDLE_PATH}"

bundle = joblib.load(BUNDLE_PATH)
ti = bundle["test_index"]

res = pd.DataFrame({
    "system": np.where(df["GGBFS (kg/m3)"].values[ti] == 0, "FA mixes",
                       np.where(df["FA (kg/m3)"].values[ti] == 0, "Slag mixes", "Blend mixes")),
    "y": np.asarray(bundle["y"].iloc[ti]).ravel(),
    "pred": np.asarray(bundle["y_test_pred"]).ravel(),
})

SYSTEMS = ["FA mixes", "Slag mixes", "Blend mixes"]
SYS_COLORS = {"FA mixes": "#D00000", "Slag mixes": "#00802B", "Blend mixes": "#0033CC"}

rows = []
for s in SYSTEMS + ["All mixes"]:
    g = res if s == "All mixes" else res[res["system"] == s]
    rows.append({"System": s, "n": len(g),
                 "R2": r2_score(g["y"], g["pred"]),
                 "RMSE": np.sqrt(mean_squared_error(g["y"], g["pred"])),
                 "MAE": mean_absolute_error(g["y"], g["pred"]),
                 "Bias": (g["pred"] - g["y"]).mean(),
                 "CS mean": g["y"].mean(), "CS std": g["y"].std()})

system_performance = pd.DataFrame(rows).set_index("System")
display(system_performance.round(4))

upper = int(np.ceil(max(res["y"].max(), res["pred"].max()) / 10.0) * 10)
line_x = np.linspace(0, upper, 300)
residual = res["pred"] - res["y"]

fig = plt.figure(figsize=(19.0, 12.0))
gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 0.85], hspace=0.28, wspace=0.20)

ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(line_x, line_x, color="black", lw=1.6, zorder=3, label="Ideal (1:1)")
ax1.plot(line_x, 1.10 * line_x, color="gray", lw=1.1, ls=":", zorder=2, label="±10% error")
ax1.plot(line_x, 0.90 * line_x, color="gray", lw=1.1, ls=":", zorder=2)

for s in SYSTEMS:
    g = res[res["system"] == s]
    ax1.scatter(g["y"], g["pred"], s=44, color=SYS_COLORS[s], alpha=0.75,
                edgecolors="white", linewidths=0.35, zorder=4,
                label=f"{s} (R$^2$ = {r2_score(g['y'], g['pred']):.3f})")

ax1.set_xlim(0, upper)
ax1.set_ylim(0, upper)
ax1.set_xlabel("Experimental CS (MPa)", fontsize=16, labelpad=9)
ax1.set_ylabel("Predicted CS (MPa)", fontsize=16, labelpad=9)
ax1.set_title("(a) Predicted against experimental strength", fontsize=17, fontweight="bold", pad=10)
ax1.xaxis.set_major_locator(MultipleLocator(20))
ax1.xaxis.set_minor_locator(MultipleLocator(10))
ax1.yaxis.set_major_locator(MultipleLocator(20))
ax1.yaxis.set_minor_locator(MultipleLocator(10))
ax1.tick_params(axis="both", which="major", labelsize=13, length=5, width=1.0)
ax1.tick_params(axis="both", which="minor", length=2.5, width=0.8)
ax1.grid(which="major", linestyle="--", linewidth=0.55, alpha=0.20)
ax1.legend(loc="upper left", fontsize=14, frameon=True)

ax2 = fig.add_subplot(gs[0, 1])
ax2.axhline(0, color="black", lw=1.4, zorder=2)

data = [residual[res["system"] == s].values for s in SYSTEMS]
bp = ax2.boxplot(data, positions=np.arange(len(SYSTEMS)), widths=0.45,
                 patch_artist=True, showfliers=False,
                 medianprops=dict(color="black", linewidth=1.8),
                 whiskerprops=dict(color="black", linewidth=1.2),
                 capprops=dict(color="black", linewidth=1.2),
                 boxprops=dict(edgecolor="black", linewidth=1.5), zorder=3)

for box, s in zip(bp["boxes"], SYSTEMS):
    box.set_facecolor(SYS_COLORS[s])
    box.set_alpha(0.75)

rng = np.random.default_rng(SEED)
for i, s in enumerate(SYSTEMS):
    v = residual[res["system"] == s].values
    ax2.scatter(np.full(len(v), i) + rng.normal(0, 0.055, len(v)), v, s=22,
                color=SYS_COLORS[s], alpha=0.60, edgecolors="white",
                linewidths=0.2, zorder=4)

ylim = 30
ax2.set_ylim(-ylim, ylim)

for i, s in enumerate(SYSTEMS):
    g = res[res["system"] == s]
    ax2.text(i, ylim * 0.93,
             f"RMSE {np.sqrt(mean_squared_error(g['y'], g['pred'])):.2f}\n"
             f"MAE {mean_absolute_error(g['y'], g['pred']):.2f}",
             ha="center", va="top", fontsize=14, zorder=10,
             bbox=dict(boxstyle="round,pad=0.30", fc="white", ec="#BFBFBF", lw=1.0, alpha=1.0))

ax2.set_xticks(np.arange(len(SYSTEMS)))
ax2.set_xticklabels(SYSTEMS, fontsize=14)
ax2.set_ylabel("Residual error (MPa)", fontsize=16, labelpad=9)
ax2.set_title("(b) Residual distribution by mixture system", fontsize=17, fontweight="bold", pad=10)
ax2.tick_params(axis="y", labelsize=13, length=5, width=1.0)
ax2.yaxis.set_major_locator(MultipleLocator(10))
ax2.yaxis.set_minor_locator(MultipleLocator(5))
ax2.grid(axis="y", linestyle="--", linewidth=0.55, alpha=0.25)

ax3 = fig.add_subplot(gs[1, :])
srt = res.sort_values("y").reset_index(drop=True)
idx = np.arange(len(srt))

ax3.plot(idx, srt["y"], color="#333333", linewidth=1.8, zorder=3)
for s in SYSTEMS:
    m = (srt["system"] == s).values
    ax3.vlines(idx[m], srt.loc[m, "y"], srt.loc[m, "pred"],
               color=SYS_COLORS[s], linewidth=0.9, alpha=0.55, zorder=2)
    ax3.scatter(idx[m], srt.loc[m, "pred"], s=26, color=SYS_COLORS[s],
                alpha=0.85, edgecolors="white", linewidths=0.25, zorder=4)

handles = [Line2D([], [], color="#333333", linewidth=1.8, label="Experimental")]
handles += [Line2D([], [], marker="o", linestyle="none", color=SYS_COLORS[s],
                   markersize=9, label=f"{s} predicted") for s in SYSTEMS]
ax3.legend(handles=handles, loc="upper left", fontsize=14, frameon=True, ncol=2)

ax3.set_xlim(0, len(srt) - 1)
ax3.set_xlabel("Specimen rank by experimental strength", fontsize=16, labelpad=9)
ax3.set_ylabel("CS (MPa)", fontsize=16, labelpad=9)
ax3.set_title("(c) Test specimens ordered by experimental strength",
              fontsize=17, fontweight="bold", pad=10)
ax3.tick_params(axis="both", labelsize=13, length=5, width=1.0)
ax3.yaxis.set_major_locator(MultipleLocator(20))
ax3.grid(axis="y", linestyle="--", linewidth=0.55, alpha=0.22)

for ax in (ax1, ax2, ax3):
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.05)

FIG_PATH = "/content/CatBoost_BO_System_Performance.png"
fig.savefig(FIG_PATH, dpi=600, facecolor="white", bbox_inches="tight")
plt.show()

files.download(FIG_PATH)

## **Scatter Plot**

In [ ]:
# =============================================================================
# 25 scatter plots: 5 models × 5 optimizers
# =============================================================================

!pip install -q xgboost catboost

import os
import joblib
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
from matplotlib.ticker import MultipleLocator

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"

bundle_paths = {
    ("RF", "GWO"):       "RF_GWO_Bundle.joblib",
    ("XGB", "GWO"):      "XGBoost_GWO_Bundle.joblib",
    ("SVR", "GWO"):      "SVR_GWO_Bundle.joblib",
    ("ANN", "GWO"):      "ANN_GWO_Bundle.joblib",
    ("CatBoost", "GWO"): "CatBoost_GWO_Bundle.joblib",

    ("RF", "GFO"):       "RF_GFO_Bundle.joblib",
    ("XGB", "GFO"):      "XGBoost_GFO_Bundle.joblib",
    ("SVR", "GFO"):      "SVR_GFO_Bundle.joblib",
    ("ANN", "GFO"):      "ANN_GFO_Bundle.joblib",
    ("CatBoost", "GFO"): "CatBoost_GFO_Bundle.joblib",

    ("RF", "WOA"):       "RF_WOA_Bundle.joblib",
    ("XGB", "WOA"):      "XGBoost_WOA_Bundle.joblib",
    ("SVR", "WOA"):      "SVR_WOA_Bundle.joblib",
    ("ANN", "WOA"):      "ANN_WOA_Bundle.joblib",
    ("CatBoost", "WOA"): "CatBoost_WOA_Bundle.joblib",

    ("RF", "DE"):        "RF_DE_Bundle.joblib",
    ("XGB", "DE"):       "XGBoost_DE_Bundle.joblib",
    ("SVR", "DE"):       "SVR_DE_Bundle.joblib",
    ("ANN", "DE"):       "ANN_DE_Bundle.joblib",
    ("CatBoost", "DE"):  "CatBoost_DE_Bundle.joblib",

    ("RF", "BO"):        "RF_BO_Bundle.joblib",
    ("XGB", "BO"):       "XGBoost_BO_Bundle.joblib",
    ("SVR", "BO"):       "SVR_BO_Bundle.joblib",
    ("ANN", "BO"):       "ANN_BO_Bundle.joblib",
    ("CatBoost", "BO"):  "CatBoost_BO_Bundle.joblib",
}

model_order = ["RF", "XGB", "SVR", "ANN", "CatBoost"]
optimizer_order = ["GWO", "GFO", "WOA", "DE", "BO"]

train_color = "#0B4F8A"
test_color = "#9B1D20"


def density_jitter(x, y, rng, x_span, y_span, bins=42, frac=0.004):
    """Small density-aware micro-jitter for better point separation."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    H, xedges, yedges = np.histogram2d(x, y, bins=bins)
    ix = np.clip(np.searchsorted(xedges, x, side="right") - 1, 0, bins - 1)
    iy = np.clip(np.searchsorted(yedges, y, side="right") - 1, 0, bins - 1)

    dens = H[ix, iy]
    dens = dens / dens.max() if dens.max() > 0 else dens

    jx = rng.normal(0, frac * x_span * (0.35 + dens), size=len(x))
    jy = rng.normal(0, frac * y_span * (0.35 + dens), size=len(y))

    return x + jx, y + jy


# stored predictions from the development-fitted model, no refitting
plot_data = {}
global_max = 0

for key, fname in bundle_paths.items():
    fpath = os.path.join(SAVE_DIR, fname)
    assert os.path.exists(fpath), f"Missing file: {fpath}"

    bundle = joblib.load(fpath)

    y = bundle["y"]
    y_dev = np.asarray(y.iloc[bundle["dev_index"]]).ravel()
    y_test = np.asarray(y.iloc[bundle["test_index"]]).ravel()
    y_dev_pred = np.asarray(bundle["y_dev_pred"]).ravel()
    y_test_pred = np.asarray(bundle["y_test_pred"]).ravel()

    plot_data[key] = {
        "y_train": y_dev,
        "y_test": y_test,
        "y_train_pred": y_dev_pred,
        "y_test_pred": y_test_pred,
        "test_r2": bundle["cv_results"].loc["Holdout", "val_R2"],
    }

    global_max = max(global_max, y_dev.max(), y_test.max(), y_dev_pred.max(), y_test_pred.max())

upper = int(np.ceil(global_max / 10.0) * 10)
line_x = np.linspace(0, upper, 300)

fig, axes = plt.subplots(nrows=5, ncols=5, figsize=(26.0, 18.8))
rng = np.random.default_rng(SEED)

for i, model_name in enumerate(model_order):
    for j, optimizer_name in enumerate(optimizer_order):
        ax = axes[i, j]
        d = plot_data[(model_name, optimizer_name)]
        first = (i == 0 and j == 0)

        xtr, ytr = density_jitter(d["y_train"], d["y_train_pred"], rng, upper, upper)
        xte, yte = density_jitter(d["y_test"], d["y_test_pred"], rng, upper, upper)

        ax.scatter(xtr, ytr, s=16, color=train_color, alpha=0.75, edgecolors="white",
                   linewidths=0.20, label="Training Set" if first else None, zorder=3)
        ax.scatter(xte, yte, s=18, color=test_color, alpha=0.75, edgecolors="white",
                   linewidths=0.20, label="Testing Set" if first else None, zorder=4)

        ax.plot(line_x, line_x, color="black", lw=1.4,
                label="Ideal (1:1)" if first else None, zorder=2)
        ax.plot(line_x, 1.10 * line_x, color="gray", lw=1.1, ls=":",
                label="±10% Error" if first else None, zorder=1)
        ax.plot(line_x, 0.90 * line_x, color="gray", lw=1.1, ls=":", zorder=1)

        ax.set_title(f"{model_name}-{optimizer_name}", fontsize=26, fontweight="bold", pad=8)

        ax.text(0.06, 0.90, f"R$^2$ = {d['test_r2']:.4f}", transform=ax.transAxes,
                ha="left", va="top", fontsize=20,
                bbox=dict(boxstyle="round,pad=0.28", fc="white", ec="#BFBFBF", lw=1.0, alpha=0.95))

        ax.set_xlim(0, upper)
        ax.set_ylim(0, upper)
        ax.set_xlabel("Experimental CS (MPa)", fontsize=20)
        ax.set_ylabel("Predicted CS (MPa)", fontsize=20)

        ax.xaxis.set_major_locator(MultipleLocator(20))
        ax.xaxis.set_minor_locator(MultipleLocator(10))
        ax.yaxis.set_major_locator(MultipleLocator(10))
        ax.yaxis.set_minor_locator(MultipleLocator(5))

        ax.tick_params(axis="x", which="major", labelsize=12, length=5, width=1.0, direction="out")
        ax.tick_params(axis="x", which="minor", length=2.5, width=0.8, direction="out")
        ax.tick_params(axis="y", which="major", labelsize=12, length=5, width=1.0, direction="out")
        ax.tick_params(axis="y", which="minor", length=3, width=0.8, direction="out")

        ax.grid(True, which="major", linestyle="--", linewidth=0.55, alpha=0.18)
        for spine in ax.spines.values():
            spine.set_color("black")
            spine.set_linewidth(1.05)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 1.025),
           ncol=4, frameon=True, fontsize=26, markerscale=3.0)

fig.tight_layout(rect=[0, 0, 1, 0.975], w_pad=2.6, h_pad=2.0)

FIG_PATH = "/content/Scatter_25_Model_Optimizer_Plots.png"
fig.savefig(FIG_PATH, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

files.download(FIG_PATH)

## **Errors Boxplots**

In [ ]:
# =============================================================================
# Models Performance — Validation RMSE box plot
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from google.colab import files

assert "model_optimizer_folds" in globals(), "Run the summary-table cell first."

plot_df = model_optimizer_folds.copy()

for col in ["Model", "Optimizer", "val_RMSE"]:
    assert col in plot_df.columns, f"`{col}` is missing from model_optimizer_folds."

model_order = ["RF", "XGBoost", "SVR", "ANN", "CatBoost"]
optimizer_order = ["GWO", "GFO", "WOA", "DE", "BO"]

optimizer_colors = {
    "GWO": "#2CA25F",
    "GFO": "#3182BD",
    "WOA": "#DE2D26",
    "DE":  "#F16913",
    "BO":  "#756BB1",
}

fig, ax = plt.subplots(figsize=(12.6, 5.8))

base_positions = np.arange(len(model_order))
offsets = [-0.30, -0.15, 0.00, 0.15, 0.30]
width = 0.12

rng = np.random.default_rng(SEED)

for j, opt in enumerate(optimizer_order):
    data_for_opt = []
    positions_for_opt = []

    for i, model in enumerate(model_order):
        vals = plot_df.loc[
            (plot_df["Model"].astype(str) == model) &
            (plot_df["Optimizer"].astype(str) == opt), "val_RMSE"
        ].dropna().values

        if len(vals) == 0:
            continue

        pos = base_positions[i] + offsets[j]
        data_for_opt.append(vals)
        positions_for_opt.append(pos)

        jitter = rng.normal(0, 0.012, size=len(vals))
        ax.scatter(np.full(len(vals), pos) + jitter, vals, s=28,
                   color=optimizer_colors[opt], edgecolor="black",
                   linewidth=0.4, alpha=0.75, zorder=3)

    bp = ax.boxplot(data_for_opt, positions=positions_for_opt, widths=width,
                    patch_artist=True, showfliers=False,
                    medianprops=dict(color="black", linewidth=1.6),
                    whiskerprops=dict(color="black", linewidth=1.2),
                    capprops=dict(color="black", linewidth=1.2),
                    boxprops=dict(edgecolor="black", linewidth=1.5))

    for box in bp["boxes"]:
        box.set_facecolor(optimizer_colors[opt])
        box.set_alpha(0.78)

ax.set_xticks(base_positions)
ax.set_xticklabels(model_order, fontsize=14)
ax.set_ylabel("Validation RMSE (MPa)", fontsize=14, labelpad=10)

ax.tick_params(axis="y", labelsize=13)
ax.grid(axis="y", linestyle="--", linewidth=0.7, alpha=0.35)
ax.grid(axis="x", visible=False)

for spine in ax.spines.values():
    spine.set_color("black")
    spine.set_linewidth(1.5)

legend_handles = [Patch(facecolor=optimizer_colors[opt], edgecolor="black", alpha=0.78, label=opt)
                  for opt in optimizer_order]

ax.legend(handles=legend_handles, title="Optimizer", title_fontsize=13,
          fontsize=13, frameon=True, loc="upper right")

plt.tight_layout()

FIG_PATH = "/content/Model_Optimizer_Validation_RMSE_Boxplot.png"
plt.savefig(FIG_PATH, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

files.download(FIG_PATH)

## **Residual Errors**

In [ ]:
# =============================================================================
# Residual-error beeswarm plot for 25 model–optimizer combinations
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from google.colab import files

assert "plot_data" in globals(), "Run the 25-scatter-plot cell first so `plot_data` exists."

model_order = ["RF", "XGB", "SVR", "ANN", "CatBoost"]
optimizer_order = ["GWO", "GFO", "WOA", "DE", "BO"]

COLOR_BY = "Model"

optimizer_colors = {
    "GWO": "#2CA25F",
    "GFO": "#3182BD",
    "WOA": "#DE2D26",
    "DE":  "#F16913",
    "BO":  "#756BB1",
}

model_colors = {
    "RF": "#1B4F72",
    "XGB": "#1B7837",
    "SVR": "#5E3C99",
    "ANN": "#B2182B",
    "CatBoost": "#006D77",
}

band_color = "#FFF2A8"

residual_rows = []
for model_name in model_order:
    for optimizer_name in optimizer_order:
        d = plot_data[(model_name, optimizer_name)]
        residuals = np.asarray(d["y_test_pred"]).ravel() - np.asarray(d["y_test"]).ravel()
        residual_rows.append(pd.DataFrame({
            "Model": model_name,
            "Optimizer": optimizer_name,
            "Combination": f"{model_name}-{optimizer_name}",
            "Residual": residuals,
        }))

residual_df = pd.concat(residual_rows, ignore_index=True)

combo_order = [f"{m}-{o}" for m in model_order for o in optimizer_order]
x_positions = np.arange(len(combo_order))

fig, ax = plt.subplots(figsize=(21.0, 6.6))
rng = np.random.default_rng(SEED)

ax.axhspan(-5, 5, color=band_color, alpha=0.48, zorder=1)

for x, combo in zip(x_positions, combo_order):
    sub = residual_df.loc[residual_df["Combination"] == combo]
    vals = sub["Residual"].values
    dot_color = (optimizer_colors[sub["Optimizer"].iloc[0]] if COLOR_BY == "Optimizer"
                 else model_colors[sub["Model"].iloc[0]])

    # beeswarm-style density jitter
    bins = np.linspace(vals.min(), vals.max(), 32)
    bin_id = np.digitize(vals, bins)
    counts = np.array([np.sum(bin_id == b) for b in bin_id], dtype=float)
    counts = counts / counts.max() if counts.max() > 0 else counts

    jitter = rng.normal(0, 0.055 + 0.18 * counts, size=len(vals))
    ax.scatter(np.full(len(vals), x) + jitter, vals, s=20, color=dot_color,
               alpha=0.68, edgecolors="white", linewidths=0.20, zorder=3)

    median_val = np.median(vals)
    ax.plot([x - 0.28, x + 0.28], [median_val, median_val],
            color="black", linewidth=2.2, zorder=4)

ax.axhline(0, color="black", linewidth=1.5, zorder=2)

ax.set_xticks(x_positions)
ax.set_xticklabels(combo_order, rotation=45, ha="right", fontsize=16)
ax.set_ylabel("Residual error (MPa)", fontsize=24)

ax.tick_params(axis="y", which="major", labelsize=18, length=5, width=1.0, direction="out")
ax.tick_params(axis="y", which="minor", length=3, width=0.8, direction="out")
ax.yaxis.set_major_locator(MultipleLocator(10))
ax.yaxis.set_minor_locator(MultipleLocator(5))

ax.grid(axis="y", which="major", linestyle="--", linewidth=0.55, alpha=0.25)
ax.grid(axis="x", visible=False)

for spine in ax.spines.values():
    spine.set_color("black")
    spine.set_linewidth(1.05)

ref_handles = [
    Line2D([0], [0], color="black", linewidth=1.5, label="Zero error"),
    Patch(facecolor=band_color, edgecolor="gray", alpha=0.48, label="±5 MPa error band"),
]
opt_handles = [Patch(facecolor=optimizer_colors[o], edgecolor="black", alpha=0.75, label=o)
               for o in optimizer_order]

legend1 = ax.legend(handles=ref_handles, loc="upper right", frameon=True, fontsize=18)
ax.add_artist(legend1)

fig.tight_layout()

FIG_PATH = "/content/Residual_Error_Beeswarm_25_Combinations.png"
fig.savefig(FIG_PATH, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

files.download(FIG_PATH)

In [ ]:
# =============================================================================
# 25 residual histograms with normal fit: 5 models × 5 optimizers
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from matplotlib.ticker import MultipleLocator
from google.colab import files

assert "plot_data" in globals(), "Run the 25-scatter-plot cell first so `plot_data` exists."

model_order = ["RF", "XGB", "SVR", "ANN", "CatBoost"]
optimizer_order = ["GWO", "GFO", "WOA", "DE", "BO"]

hist_color = "#BFBFBF"
fit_color = "#D7301F"

residuals = {k: np.asarray(d["y_test_pred"]).ravel() - np.asarray(d["y_test"]).ravel()
             for k, d in plot_data.items()}

lim = float(np.ceil(max(np.abs(r).max() for r in residuals.values()) / 5) * 5)
bins = np.linspace(-lim, lim, 41)
grid_x = np.linspace(-lim, lim, 400)
ymax = 0

fig, axes = plt.subplots(nrows=5, ncols=5, figsize=(22.0, 17.5))

for i, model_name in enumerate(model_order):
    for j, optimizer_name in enumerate(optimizer_order):
        ax = axes[i, j]
        r = residuals[(model_name, optimizer_name)]
        mu, sigma = norm.fit(r)

        ax.hist(r, bins=bins, density=True, color=hist_color,
                edgecolor="black", linewidth=0.5, zorder=2)
        ax.plot(grid_x, norm.pdf(grid_x, mu, sigma), color=fit_color, linewidth=2.0, zorder=3)

        # narrow stacked legend so it sits beside the distribution rather than over it
        ax.legend(handles=[plt.Line2D([], [], color=fit_color, linewidth=2.0),
                           plt.Line2D([], [], linestyle="none"),
                           plt.Line2D([], [], linestyle="none")],
                  labels=["Normal fit", f"μ = {mu:.2f}", f"σ = {sigma:.2f}"],
                  fontsize=12, frameon=True, loc="upper right",
                  handlelength=1.0, handletextpad=0.5,
                  labelspacing=0.35, borderpad=0.5)

        ax.set_title(f"{model_name}-{optimizer_name}", fontsize=24, fontweight="bold", pad=8)
        ax.set_xlabel("Residual error (MPa)", fontsize=16)
        ax.set_ylabel("Probability density", fontsize=16)
        ax.set_xlim(-lim, lim)

        ax.tick_params(axis="both", labelsize=12, length=5, width=1.0, direction="out")
        ax.xaxis.set_major_locator(MultipleLocator(20))
        ax.xaxis.set_minor_locator(MultipleLocator(10))
        ax.grid(axis="y", linestyle="--", linewidth=0.55, alpha=0.20)

        for spine in ax.spines.values():
            spine.set_color("black")
            spine.set_linewidth(1.05)

        ymax = max(ymax, ax.get_ylim()[1])

# shared y scale so the peaks are directly comparable across panels
for ax in axes.ravel():
    ax.set_ylim(0, ymax)

fig.tight_layout(w_pad=2.6, h_pad=2.0)

FIG_PATH = "/content/Residual_Histograms_25_Combinations.png"
fig.savefig(FIG_PATH, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

files.download(FIG_PATH)

# **SHAP Analysis**

In [ ]:
# =============================================================================
# Cell 1 — CatBoost-BO SHAP analysis + SHAP summary plot
# =============================================================================

!pip install shap -q

import os
import joblib
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors
from google.colab import files

BUNDLE_PATH = (
    "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper/"
    "CatBoost_BO_Bundle.joblib"
)

assert os.path.exists(BUNDLE_PATH), f"Bundle not found: {BUNDLE_PATH}"

bundle = joblib.load(BUNDLE_PATH)

model = bundle["model_full_data"]
X_shap = bundle["X"].copy()
feature_names = list(bundle["feature_names"])

if list(X_shap.columns) != feature_names:
    raise ValueError("Feature names or feature order do not match the saved CatBoost-BO model.")

SHAP_LABELS = {
    "Total binder (kg/m3)": "Total binder",
    "Coarse aggregate (kg/m3)": "Coarse aggregate",
    "Fine aggregate (kg in 1m3 mix)": "Fine aggregate",
    "Concentration (M) NaOH": "NaOH molarity",
    "Superplasticizer (kg in 1m3 mix)": "Superplasticizer",
    "Water_binder_ratio": "Water/binder",
    "Initial curing time (day)": "Curing time",
    "Initial curing temp (C)": "Curing temperature",
    "Age_days": "Age",
    "CaO_SiO2_molar": r"$\mathrm{CaO/SiO_2}$",
    "SiO2_Al2O3_molar": r"$\mathrm{SiO_2/Al_2O_3}$",
    "Na2O_Al2O3_molar": r"$\mathrm{Na_2O/Al_2O_3}$",
    "Fe2O3_Al2O3_molar": r"$\mathrm{Fe_2O_3/Al_2O_3}$",
    "MgO_Al2O3_molar": r"$\mathrm{MgO/Al_2O_3}$",
    "Ms_activator": r"$\mathrm{M_s}$",
    "H2O_Na2O_molar": r"$\mathrm{H_2O/Na_2O}$",
}

shap_feature_labels = [SHAP_LABELS.get(c, c) for c in X_shap.columns]

SHAP_CMAP = colors.LinearSegmentedColormap.from_list(
    "teal_white_orange",
    [
        (0.00, "#01665E"),
        (0.25, "#5AB4AC"),
        (0.50, "#F7F7F7"),
        (0.75, "#D8B365"),
        (1.00, "#8C510A"),
    ],
    N=256,
)

explainer = shap.TreeExplainer(model)
shap_values_raw = explainer(X_shap)

shap_values = shap.Explanation(
    values=shap_values_raw.values,
    base_values=shap_values_raw.base_values,
    data=X_shap.values,
    feature_names=shap_feature_labels,
)

X_shap_display = X_shap.copy()
X_shap_display.columns = shap_feature_labels

plt.figure(figsize=(8.6, 7.6))

shap.plots.beeswarm(
    shap_values,
    max_display=X_shap.shape[1],
    color=SHAP_CMAP,
    plot_size=None,
    show=False,
)

ax = plt.gca()
ax.set_xlabel("SHAP value", fontsize=18, labelpad=9)

ax.tick_params(axis="x", labelsize=14)
ax.tick_params(axis="y", labelsize=16)
ax.grid(False)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

ax.spines["bottom"].set_linewidth(1.4)
ax.spines["bottom"].set_color("black")

cax = plt.gcf().axes[-1]
cax.set_ylabel("")
cax.tick_params(axis="y", labelsize=14)

plt.tight_layout()

FIG_PATH = "/content/CatBoost_BO_SHAP_Summary_Beeswarm.png"
plt.savefig(FIG_PATH, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

files.download(FIG_PATH)

In [ ]:
# =============================================================================
# Cell 2 — SHAP value ranking based on mean absolute SHAP values
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

assert "shap_values" in globals(), "Run Cell 1 first. `shap_values` not found."
assert "X_shap" in globals(), "Run Cell 1 first. `X_shap` not found."
assert "shap_feature_labels" in globals(), "Run Cell 1 first. `shap_feature_labels` not found."
assert "SHAP_CMAP" in globals(), "Run Cell 1 first. `SHAP_CMAP` not found."

mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

shap_ranking_df = pd.DataFrame({
    "Rank": np.arange(1, len(shap_feature_labels) + 1),
    "Feature": shap_feature_labels,
    "Original feature": list(X_shap.columns),
    "Mean |SHAP value|": mean_abs_shap,
})

shap_ranking_df = (
    shap_ranking_df
    .sort_values("Mean |SHAP value|", ascending=False)
    .reset_index(drop=True)
)

shap_ranking_df["Rank"] = np.arange(1, len(shap_ranking_df) + 1)
shap_ranking_df["Relative importance (%)"] = (
    100 * shap_ranking_df["Mean |SHAP value|"] / shap_ranking_df["Mean |SHAP value|"].sum()
)

display(
    shap_ranking_df.round({
        "Mean |SHAP value|": 4,
        "Relative importance (%)": 2,
    })
)

plot_df = shap_ranking_df.sort_values("Mean |SHAP value|", ascending=True).copy()

# Smooth rank-based coloring instead of value-based coloring
# This avoids one dark brown bar and many nearly identical blue bars.
color_positions = np.linspace(0.08, 0.92, len(plot_df))
bar_colors = SHAP_CMAP(color_positions)

fig, ax = plt.subplots(figsize=(8.6, 7.6))

ax.barh(
    plot_df["Feature"],
    plot_df["Mean |SHAP value|"],
    color=bar_colors,
    edgecolor="black",
    linewidth=0.45,
)

ax.set_xlabel("Mean absolute SHAP value", fontsize=18, labelpad=9)
ax.set_ylabel("")

ax.tick_params(axis="x", labelsize=14)
ax.tick_params(axis="y", labelsize=16)

ax.grid(axis="x", linestyle="--", linewidth=0.5, alpha=0.55)
ax.grid(axis="y", visible=False)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

ax.spines["bottom"].set_linewidth(1.4)
ax.spines["bottom"].set_color("black")

plt.tight_layout()

FIG_PATH = "/content/CatBoost_BO_SHAP_Value_Ranking.png"
plt.savefig(FIG_PATH, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

files.download(FIG_PATH)

In [ ]:
# =============================================================================
# Cell 3 — SHAP river-flow style decision plot
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

assert "shap" in globals(), "Run Cell 1 first. `shap` not found."
assert "model" in globals(), "Run Cell 1 first. `model` not found."
assert "X_shap" in globals(), "Run Cell 1 first. `X_shap` not found."
assert "X_shap_display" in globals(), "Run Cell 1 first. `X_shap_display` not found."
assert "shap_values" in globals(), "Run Cell 1 first. `shap_values` not found."
assert "SHAP_CMAP" in globals(), "Run Cell 1 first. `SHAP_CMAP` not found."

pred_for_order = model.predict(X_shap)
sample_idx = np.argsort(pred_for_order)

base_value = shap_values.base_values
if np.ndim(base_value) > 0:
    base_value = float(np.mean(base_value))
else:
    base_value = float(base_value)

feature_order = np.argsort(np.abs(shap_values.values).mean(axis=0))[::-1]

plt.figure(figsize=(9.2, 7.6))

shap.decision_plot(
    base_value=base_value,
    shap_values=shap_values.values[sample_idx],
    features=X_shap_display.iloc[sample_idx],
    feature_names=shap_feature_labels,
    feature_order=feature_order,
    plot_color=SHAP_CMAP,
    show=False,
    ignore_warnings=True,
)

ax = plt.gca()
ax.set_xlabel("Model output value: predicted CS (MPa)", fontsize=16, labelpad=9)

ax.tick_params(axis="x", labelsize=14)
ax.tick_params(axis="y", labelsize=16)
ax.grid(True, linestyle="--", linewidth=0.45, alpha=0.50)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

ax.spines["bottom"].set_linewidth(1.4)
ax.spines["bottom"].set_color("black")
ax.spines["left"].set_linewidth(1.2)
ax.spines["left"].set_color("black")

plt.tight_layout()

FIG_PATH = "/content/CatBoost_BO_SHAP_River_Flow.png"
plt.savefig(FIG_PATH, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

files.download(FIG_PATH)

In [ ]:
# =============================================================================
# Cell 4 — SHAP value distribution of different input variables
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from google.colab import files

try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
except Exception:
    !pip install statsmodels -q
    from statsmodels.nonparametric.smoothers_lowess import lowess

assert "X_shap" in globals(), "Run Cell 1 first. `X_shap` not found."
assert "shap_values" in globals(), "Run Cell 1 first. `shap_values` not found."
assert "shap_feature_labels" in globals(), "Run Cell 1 first. `shap_feature_labels` not found."
assert "SHAP_CMAP" in globals(), "Run Cell 1 first. `SHAP_CMAP` not found."

TITLE_FS, LABEL_FS, TICK_FS = 16, 14, 12
LOWESS_FRAC, BOOTSTRAP_N, GRID_N = 0.28, 120, 160

rng = np.random.default_rng(SEED)

mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
feature_order = list(np.argsort(mean_abs_shap)[::-1])

# Unicode subscripts rather than mathtext, so the whole panel title renders bold
TITLE_LABELS = {
    "Total binder (kg/m3)": "Total binder",
    "Coarse aggregate (kg/m3)": "Coarse aggregate",
    "Fine aggregate (kg in 1m3 mix)": "Fine aggregate",
    "Concentration (M) NaOH": "NaOH molarity",
    "Superplasticizer (kg in 1m3 mix)": "Superplasticizer",
    "Water_binder_ratio": "Water/binder",
    "Initial curing time (day)": "Curing time",
    "Initial curing temp (C)": "Curing temperature",
    "Age_days": "Age",
    "CaO_SiO2_molar": "CaO/SiO₂",
    "SiO2_Al2O3_molar": "SiO₂/Al₂O₃",
    "Na2O_Al2O3_molar": "Na₂O/Al₂O₃",
    "Fe2O3_Al2O3_molar": "Fe₂O₃/Al₂O₃",
    "MgO_Al2O3_molar": "MgO/Al₂O₃",
    "Ms_activator": "Mₛ",
    "H2O_Na2O_molar": "H₂O/Na₂O",
}


def lowess_with_bootstrap_ci(x, y, frac, n_boot, grid_n, rng):
    """LOWESS trend with a percentile bootstrap band, returning None when the feature is too sparse."""
    tmp = pd.DataFrame({"x": x, "y": y}).replace([np.inf, -np.inf], np.nan).dropna()
    if len(tmp) < 25 or tmp["x"].nunique() < 4:
        return None, None, None, None

    tmp = tmp.sort_values("x")
    x_arr = tmp["x"].values.astype(float)
    y_arr = tmp["y"].values.astype(float)
    x_grid = np.linspace(x_arr.min(), x_arr.max(), grid_n)

    try:
        smooth = lowess(y_arr, x_arr, frac=frac, it=0, return_sorted=True)
        y_smooth = np.interp(x_grid, smooth[:, 0], smooth[:, 1])
    except Exception:
        return None, None, None, None

    boot = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(x_arr), len(x_arr))
        order = np.argsort(x_arr[idx])
        xb, yb = x_arr[idx][order], y_arr[idx][order]
        if len(np.unique(xb)) < 4:
            continue
        try:
            sm_b = lowess(yb, xb, frac=frac, it=0, return_sorted=True)
            boot.append(np.interp(x_grid, sm_b[:, 0], sm_b[:, 1]))
        except Exception:
            continue

    if len(boot) < 20:
        return x_grid, y_smooth, None, None

    boot = np.asarray(boot)
    return x_grid, y_smooth, np.percentile(boot, 2.5, axis=0), np.percentile(boot, 97.5, axis=0)


n_features = len(feature_order)
ncols = 4
nrows = int(np.ceil(n_features / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(15.5, 3.2 * nrows))
axes = np.asarray(axes).ravel()

for panel_i, feat_i in enumerate(feature_order):
    ax = axes[panel_i]

    xvals = X_shap.iloc[:, feat_i].values.astype(float)
    yvals = shap_values.values[:, feat_i].astype(float)
    valid = np.isfinite(xvals) & np.isfinite(yvals)
    xvals, yvals = xvals[valid], yvals[valid]

    if len(xvals) == 0:
        ax.axis("off")
        continue

    ax.scatter(xvals, yvals, c=xvals, cmap=SHAP_CMAP,
               norm=Normalize(vmin=xvals.min(), vmax=xvals.max()),
               s=18, alpha=0.72, edgecolors="none", rasterized=True, zorder=2)

    x_grid, y_smooth, y_low, y_high = lowess_with_bootstrap_ci(
        xvals, yvals, LOWESS_FRAC, BOOTSTRAP_N, GRID_N, rng)

    if x_grid is not None:
        if y_low is not None:
            ax.fill_between(x_grid, y_low, y_high, color="#F4A582",
                            alpha=0.28, linewidth=0, zorder=1)
        ax.plot(x_grid, y_smooth, color="#D7301F", linewidth=1.8, alpha=0.98, zorder=3)

    ax.axhline(0, color="black", linewidth=0.75, alpha=0.72, zorder=0)

    raw_name = X_shap.columns[feat_i]
    label = TITLE_LABELS.get(raw_name, shap_feature_labels[feat_i])
    ax.set_title(label, fontsize=TITLE_FS, fontweight="bold", pad=7)

    ax.tick_params(axis="both", labelsize=TICK_FS)
    ax.grid(True, linestyle="--", linewidth=0.35, alpha=0.42)

    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_linewidth(1.1)
    ax.spines["bottom"].set_color("black")
    ax.spines["left"].set_linewidth(1.1)
    ax.spines["left"].set_color("black")

    ax.set_ylabel("SHAP value" if panel_i % ncols == 0 else "", fontsize=LABEL_FS)
    ax.set_xlabel("Feature value" if panel_i >= n_features - ncols else "", fontsize=LABEL_FS)

for k in range(n_features, len(axes)):
    axes[k].axis("off")

plt.tight_layout()

FIG_PATH = "/content/CatBoost_BO_SHAP_Feature_Distributions.png"
plt.savefig(FIG_PATH, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()

files.download(FIG_PATH)

# **MOO Optimization**

## **A: Locked Price + Ambient Curing**

In [ ]:
# =============================================================================
# MOO Step 1 — Load the surrogate and build the conformal prediction layer
# =============================================================================

import os
import joblib
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold, train_test_split

SAVE_DIR = "/content/drive/MyDrive/Geopolymer_ML_Journal_Paper"
BUNDLE_PATH = os.path.join(SAVE_DIR, "CatBoost_BO_Bundle.joblib")

assert os.path.exists(BUNDLE_PATH), f"Bundle not found: {BUNDLE_PATH}"

bundle = joblib.load(BUNDLE_PATH)

X_all = bundle["X"]
y_all = bundle["y"]
dev_index = bundle["dev_index"]
test_index = bundle["test_index"]
best_params = dict(bundle["best_parameters"])

assert list(X_all.columns) == list(INPUTS), "Bundle feature order does not match INPUTS."

ALPHA = 0.10
CAL_FRACTION = 0.20
N_DIFF_FOLDS = 5

# calibration rows must be unseen by the fitted model, so the development set is
# split again and the tuned model is refit on the proper-training part only
proper_index, cal_index = train_test_split(dev_index, test_size=CAL_FRACTION,
                                           random_state=SEED, shuffle=True)

X_proper, y_proper = X_all.iloc[proper_index], y_all.iloc[proper_index]
X_cal, y_cal = X_all.iloc[cal_index], y_all.iloc[cal_index]
X_test, y_test = X_all.iloc[test_index], y_all.iloc[test_index]

point_model = CatBoostRegressor(**best_params)
point_model.fit(X_proper, y_proper)

# out-of-fold residuals, so the difficulty model learns errors the point model
# actually makes on data it has not seen
oof_pred = np.zeros(len(X_proper))
for tr, va in KFold(n_splits=N_DIFF_FOLDS, shuffle=True, random_state=SEED).split(X_proper):
    m = CatBoostRegressor(**best_params)
    m.fit(X_proper.iloc[tr], y_proper.iloc[tr])
    oof_pred[va] = m.predict(X_proper.iloc[va])

oof_abs_residual = np.abs(np.asarray(y_proper) - oof_pred)

# shallower than the point model, since it fits error magnitude rather than strength
difficulty_params = {**best_params, "depth": 4, "iterations": 400}
difficulty_model = CatBoostRegressor(**difficulty_params)
difficulty_model.fit(X_proper, np.log(oof_abs_residual + 1.0))

SIGMA_FLOOR = float(np.quantile(np.exp(difficulty_model.predict(X_proper)), 0.05))


def sigma_of(X_features):
    return np.maximum(np.exp(difficulty_model.predict(X_features)), SIGMA_FLOOR)


cal_pred = point_model.predict(X_cal)
cal_sigma = sigma_of(X_cal)
cal_residual = np.asarray(y_cal) - cal_pred

n_cal = len(cal_residual)
level = min(np.ceil((n_cal + 1) * (1 - ALPHA)) / n_cal, 1.0)

Q_TWO_SIDED = float(np.quantile(np.abs(cal_residual) / cal_sigma, level, method="higher"))
Q_ONE_SIDED = float(np.quantile((-cal_residual) / cal_sigma, level, method="higher"))


def predict_cs_bounds(X_features):
    """Point prediction and one-sided conformal lower bound for a feature frame."""
    point = point_model.predict(X_features)
    return point, point - Q_ONE_SIDED * sigma_of(X_features)


test_point, test_lower = predict_cs_bounds(X_test)
test_sigma = sigma_of(X_test)

coverage_one = float(np.mean(y_test >= test_lower))
coverage_two = float(np.mean((y_test >= test_point - Q_TWO_SIDED * test_sigma) &
                             (y_test <= test_point + Q_TWO_SIDED * test_sigma)))

print(f"Proper training {len(proper_index)} | calibration {n_cal} | held out {len(test_index)}")
print(f"Nominal coverage {1 - ALPHA:.2f} | sigma floor {SIGMA_FLOOR:.3f} MPa")
print()
print(f"One-sided lower bound  q = {Q_ONE_SIDED:.3f}  ->  coverage {coverage_one:.4f}  "
      f"| mean margin {np.mean(Q_ONE_SIDED * test_sigma):.3f} MPa")
print(f"Two-sided interval     q = {Q_TWO_SIDED:.3f}  ->  coverage {coverage_two:.4f}  "
      f"| mean width {np.mean(2 * Q_TWO_SIDED * test_sigma):.3f} MPa")
print()

bands = pd.cut(test_point, bins=[0, 20, 30, 40, 50, np.inf],
               labels=["<20", "20-30", "30-40", "40-50", ">50"])
display(pd.DataFrame({"band": bands,
                      "covered": y_test.values >= test_lower,
                      "margin": Q_ONE_SIDED * test_sigma})
        .groupby("band", observed=True)
        .agg(n=("covered", "size"), coverage=("covered", "mean"), mean_margin=("margin", "mean"))
        .round(4))

conformal = {"alpha": ALPHA, "point_model": point_model, "difficulty_model": difficulty_model,
             "sigma_floor": SIGMA_FLOOR, "q_one_sided": Q_ONE_SIDED, "q_two_sided": Q_TWO_SIDED,
             "proper_index": proper_index, "cal_index": cal_index,
             "coverage_one_sided": coverage_one, "coverage_two_sided": coverage_two}

In [ ]:
# =============================================================================
# MOO Step 2 — Precursor source catalogues and the analytic feature map
# =============================================================================

import numpy as np
import pandas as pd

OXIDES = ["SiO2", "Al2O3", "Fe2O3", "CaO", "MgO", "Na2O"]
NAOH_TO_NA2O = 61.979 / (2 * 39.997)
M = {"SiO2": 60.084, "Al2O3": 101.961, "CaO": 56.077,
     "MgO": 40.304, "Fe2O3": 159.687, "Na2O": 61.979, "H2O": 18.015}
AGE_FIXED = 28

# catalogues taken only from single-precursor mixes, where the reported blended
# composition is the composition of that one precursor
fa_only = df[(df["GGBFS (kg/m3)"] == 0) & (df["FA (kg/m3)"] > 0)]
slag_only = df[(df["FA (kg/m3)"] == 0) & (df["GGBFS (kg/m3)"] > 0)]

FA_SOURCES = fa_only[OXIDES].round(3).drop_duplicates().reset_index(drop=True)
SLAG_SOURCES = slag_only[OXIDES].round(3).drop_duplicates().reset_index(drop=True)

FA_ARRAY = FA_SOURCES.values.astype(float)
SLAG_ARRAY = SLAG_SOURCES.values.astype(float)

RAW_VARS = [
    "FA (kg/m3)",
    "GGBFS (kg/m3)",
    "Coarse aggregate (kg/m3)",
    "Fine aggregate (kg in 1m3 mix)",
    "Total Na2SiO3 (kg in 1m3 of mix)",
    "SiO2 (Dry)",
    "Na2O (Dry)",
    "NaOH (Dry)",
    "Concentration (M) NaOH",
    "Superplasticizer (kg in 1m3 mix)",
    "Total water (in solutions + additional) (kg in 1m3 mix)",
    "Initial curing time (day)",
    "Initial curing temp (C)",
]

R = {v: j for j, v in enumerate(RAW_VARS)}


def build_features(x, fa_oxides, slag_oxides, age=AGE_FIXED):
    """Map one raw mixture plus its two precursor compositions to the 16 model features."""
    fa, slag = float(x["FA (kg/m3)"]), float(x["GGBFS (kg/m3)"])
    sio2_dry, na2o_dry = float(x["SiO2 (Dry)"]), float(x["Na2O (Dry)"])
    sh = float(x["NaOH (Dry)"])
    water = float(x["Total water (in solutions + additional) (kg in 1m3 mix)"])

    binder = fa + slag
    if binder <= 0:
        return None

    blend = (fa * np.asarray(fa_oxides, float) + slag * np.asarray(slag_oxides, float)) / binder
    ox = dict(zip(OXIDES, blend))
    b_ox = {k: binder * ox[k] / 100.0 for k in OXIDES}

    act_na2o = na2o_dry + sh * NAOH_TO_NA2O
    if act_na2o <= 0:
        return None

    sio2_mol = (b_ox["SiO2"] + sio2_dry) / M["SiO2"]
    al2o3_mol = b_ox["Al2O3"] / M["Al2O3"]
    if sio2_mol <= 0 or al2o3_mol <= 0:
        return None

    return pd.Series({
        "Total binder (kg/m3)": binder,
        "Coarse aggregate (kg/m3)": float(x["Coarse aggregate (kg/m3)"]),
        "Fine aggregate (kg in 1m3 mix)": float(x["Fine aggregate (kg in 1m3 mix)"]),
        "Concentration (M) NaOH": float(x["Concentration (M) NaOH"]),
        "Superplasticizer (kg in 1m3 mix)": float(x["Superplasticizer (kg in 1m3 mix)"]),
        "Water_binder_ratio": water / binder,
        "Initial curing time (day)": float(x["Initial curing time (day)"]),
        "Initial curing temp (C)": float(x["Initial curing temp (C)"]),
        "Age_days": float(age),
        "CaO_SiO2_molar": (b_ox["CaO"] / M["CaO"]) / sio2_mol,
        "SiO2_Al2O3_molar": sio2_mol / al2o3_mol,
        "Na2O_Al2O3_molar": ((b_ox["Na2O"] + act_na2o) / M["Na2O"]) / al2o3_mol,
        "Fe2O3_Al2O3_molar": (b_ox["Fe2O3"] / M["Fe2O3"]) / al2o3_mol,
        "MgO_Al2O3_molar": (b_ox["MgO"] / M["MgO"]) / al2o3_mol,
        "Ms_activator": (sio2_dry / M["SiO2"]) / (act_na2o / M["Na2O"]),
        "H2O_Na2O_molar": (water / M["H2O"]) / (act_na2o / M["Na2O"]),
    })[INPUTS]


def build_features_batch(Xr, fa_idx, slag_idx, age=AGE_FIXED):
    """Vectorized feature map. fa_idx and slag_idx are integer catalogue rows."""
    fa, slag = Xr[:, R["FA (kg/m3)"]], Xr[:, R["GGBFS (kg/m3)"]]
    sio2_dry, na2o_dry = Xr[:, R["SiO2 (Dry)"]], Xr[:, R["Na2O (Dry)"]]
    sh = Xr[:, R["NaOH (Dry)"]]
    water = Xr[:, R["Total water (in solutions + additional) (kg in 1m3 mix)"]]

    binder = np.maximum(fa + slag, 1e-9)
    blend = (fa[:, None] * FA_ARRAY[fa_idx] + slag[:, None] * SLAG_ARRAY[slag_idx]) / binder[:, None]
    b_ox = {k: binder * blend[:, i] / 100.0 for i, k in enumerate(OXIDES)}

    act_na2o = np.maximum(na2o_dry + sh * NAOH_TO_NA2O, 1e-12)
    act_na2o_mol = act_na2o / M["Na2O"]
    sio2_mol = np.maximum((b_ox["SiO2"] + sio2_dry) / M["SiO2"], 1e-12)
    al2o3_mol = np.maximum(b_ox["Al2O3"] / M["Al2O3"], 1e-12)

    return pd.DataFrame({
        "Total binder (kg/m3)": binder,
        "Coarse aggregate (kg/m3)": Xr[:, R["Coarse aggregate (kg/m3)"]],
        "Fine aggregate (kg in 1m3 mix)": Xr[:, R["Fine aggregate (kg in 1m3 mix)"]],
        "Concentration (M) NaOH": Xr[:, R["Concentration (M) NaOH"]],
        "Superplasticizer (kg in 1m3 mix)": Xr[:, R["Superplasticizer (kg in 1m3 mix)"]],
        "Water_binder_ratio": water / binder,
        "Initial curing time (day)": Xr[:, R["Initial curing time (day)"]],
        "Initial curing temp (C)": Xr[:, R["Initial curing temp (C)"]],
        "Age_days": np.full(len(Xr), float(age)),
        "CaO_SiO2_molar": (b_ox["CaO"] / M["CaO"]) / sio2_mol,
        "SiO2_Al2O3_molar": sio2_mol / al2o3_mol,
        "Na2O_Al2O3_molar": ((b_ox["Na2O"] + act_na2o) / M["Na2O"]) / al2o3_mol,
        "Fe2O3_Al2O3_molar": (b_ox["Fe2O3"] / M["Fe2O3"]) / al2o3_mol,
        "MgO_Al2O3_molar": (b_ox["MgO"] / M["MgO"]) / al2o3_mol,
        "Ms_activator": (sio2_dry / M["SiO2"]) / act_na2o_mol,
        "H2O_Na2O_molar": (water / M["H2O"]) / act_na2o_mol,
    })[INPUTS]


# ---- validation 1, single-row map against the training feature matrix -------
rebuilt = [build_features(df.iloc[i], df.iloc[i][OXIDES].values, df.iloc[i][OXIDES].values,
                          age=df.iloc[i]["Age_days"]) for i in range(len(df))]
diff_single = (pd.DataFrame(rebuilt, index=df.index)[INPUTS] - X).abs().max().max()

# ---- validation 2, batch map against the single-row map --------------------
probe = df.sample(200, random_state=SEED)
single_probe = pd.DataFrame([build_features(probe.iloc[i], FA_ARRAY[0], SLAG_ARRAY[0])
                             for i in range(len(probe))])[INPUTS]
batch_probe = build_features_batch(probe[RAW_VARS].values,
                                   np.zeros(len(probe), dtype=int),
                                   np.zeros(len(probe), dtype=int))
diff_batch = np.abs(single_probe.values - batch_probe.values).max()

print(f"Fly ash sources: {len(FA_SOURCES)} | slag sources: {len(SLAG_SOURCES)} "
      f"| source combinations: {len(FA_SOURCES) * len(SLAG_SOURCES)}")
print()
print(f"Single-row map vs training features : max abs diff {diff_single:.3e}")
print(f"Batch map vs single-row map         : max abs diff {diff_batch:.3e}")

assert diff_single < 1e-8, "Analytic feature map does not reproduce the training features."
assert diff_batch < 1e-8, "Batch feature map disagrees with the single-row map."
print("\nBoth feature maps verified.")

In [ ]:
# =============================================================================
# MOO Step 3 — Search domain and seed pool, ambient curing
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors

Q_LOW, Q_HIGH = 0.01, 0.99

VOLUME_LOW, VOLUME_HIGH = 0.97, 1.03          # physical closure, not a data quantile
GUARD_PERCENTILE = 0.75

# ambient curing. Temperature and time are fixed, so they are not design
# variables and no curing energy is consumed. Preprocessing set curing time to
# zero for every row below 30 C, so time = 0 is the consistent ambient value.
AMBIENT_TEMP = 25.0
AMBIENT_TIME = 0.0
AMBIENT_SELECT_MAX = 30.0                     # observed rows counted as ambient

opt_data = (df[RAW_VARS].apply(pd.to_numeric, errors="coerce")
            .dropna().drop_duplicates().reset_index(drop=True))

DENSITY = {"FA (kg/m3)": 2300.0, "GGBFS (kg/m3)": 2860.0,
           "Coarse aggregate (kg/m3)": 2740.0, "Fine aggregate (kg in 1m3 mix)": 2640.0,
           "Superplasticizer (kg in 1m3 mix)": 1100.0,
           "Total water (in solutions + additional) (kg in 1m3 mix)": 1000.0}
SILICATE_SOLID_DENSITY = 2400.0
NAOH_SOLID_DENSITY = 2130.0


def volume_of(Xr):
    v = sum(Xr[:, R[k]] / DENSITY[k] for k in DENSITY)
    v += (Xr[:, R["SiO2 (Dry)"]] + Xr[:, R["Na2O (Dry)"]]) / SILICATE_SOLID_DENSITY
    v += Xr[:, R["NaOH (Dry)"]] / NAOH_SOLID_DENSITY
    return v


def agg_volume_of(Xr):
    return (Xr[:, R["Coarse aggregate (kg/m3)"]] / DENSITY["Coarse aggregate (kg/m3)"] +
            Xr[:, R["Fine aggregate (kg in 1m3 mix)"]] / DENSITY["Fine aggregate (kg in 1m3 mix)"])


O = opt_data.values.astype(float)
q = lambda a: np.nanquantile(a, [Q_LOW, Q_HIGH])

bounds_df = pd.DataFrame({"low": opt_data.quantile(Q_LOW), "high": opt_data.quantile(Q_HIGH)})
bounds_df["high"] = np.where(bounds_df["high"] <= bounds_df["low"],
                             bounds_df["low"] + 1e-9, bounds_df["high"])
LOW = bounds_df.loc[RAW_VARS, "low"].values
HIGH = bounds_df.loc[RAW_VARS, "high"].values

# pin curing so the optimizer cannot vary it
LOW[R["Initial curing temp (C)"]] = AMBIENT_TEMP
HIGH[R["Initial curing temp (C)"]] = AMBIENT_TEMP + 1e-6
LOW[R["Initial curing time (day)"]] = AMBIENT_TIME
HIGH[R["Initial curing time (day)"]] = AMBIENT_TIME + 1e-6

binder_o = O[:, R["FA (kg/m3)"]] + O[:, R["GGBFS (kg/m3)"]]
vol_o = volume_of(O)
ss_o, sh_o = O[:, R["Total Na2SiO3 (kg in 1m3 of mix)"]], O[:, R["NaOH (Dry)"]]
water_o = O[:, R["Total water (in solutions + additional) (kg in 1m3 mix)"]]

BINDER_LOW, BINDER_HIGH = q(binder_o)
AA_BINDER_LOW, AA_BINDER_HIGH = q((ss_o + sh_o) / binder_o)
SS_SH_LOW, SS_SH_HIGH = q(np.where(sh_o > 0, ss_o / np.where(sh_o > 0, sh_o, 1), np.nan))
WB_LOW, WB_HIGH = q(water_o / np.maximum(binder_o, 1e-9))
AGG_FRAC_LOW, AGG_FRAC_HIGH = q(agg_volume_of(O) / vol_o)
SS_SOLIDS_LOW, SS_SOLIDS_HIGH = q((O[:, R["SiO2 (Dry)"]] + O[:, R["Na2O (Dry)"]]) / ss_o)
SS_MS_LOW, SS_MS_HIGH = q((O[:, R["SiO2 (Dry)"]] / M["SiO2"]) / (O[:, R["Na2O (Dry)"]] / M["Na2O"]))

domain_scaler = MinMaxScaler().fit(opt_data)
nn_domain = NearestNeighbors(n_neighbors=6).fit(domain_scaler.transform(opt_data))
dist_o, _ = nn_domain.kneighbors(domain_scaler.transform(opt_data))
DOMAIN_DISTANCE_LIMIT = float(np.quantile(dist_o[:, -1], GUARD_PERCENTILE))


def _iv(v, low, high):
    scale = max(abs(high - low), 1e-9)
    return (np.maximum(low - v, 0.0) + np.maximum(v - high, 0.0)) / scale


def violation_batch(Xr):
    """Total normalized constraint violation. Zero means feasible.
    Curing is fixed, so no curing constraints. Precursor proportioning is free."""
    cv = np.zeros(len(Xr))

    for j in range(len(RAW_VARS)):
        cv += _iv(Xr[:, j], LOW[j], HIGH[j])

    fa, slag = Xr[:, R["FA (kg/m3)"]], Xr[:, R["GGBFS (kg/m3)"]]
    ss, sh = Xr[:, R["Total Na2SiO3 (kg in 1m3 of mix)"]], Xr[:, R["NaOH (Dry)"]]
    sio2_dry, na2o_dry = Xr[:, R["SiO2 (Dry)"]], Xr[:, R["Na2O (Dry)"]]
    water = Xr[:, R["Total water (in solutions + additional) (kg in 1m3 mix)"]]

    binder = np.maximum(fa + slag, 1e-9)
    ss_s, sh_s = np.maximum(ss, 1e-9), np.maximum(sh, 1e-9)

    cv += _iv(binder, BINDER_LOW, BINDER_HIGH)
    cv += _iv((ss + sh) / binder, AA_BINDER_LOW, AA_BINDER_HIGH)
    cv += _iv(ss / sh_s, SS_SH_LOW, SS_SH_HIGH)
    cv += _iv(water / binder, WB_LOW, WB_HIGH)

    vol = volume_of(Xr)
    cv += _iv(vol, VOLUME_LOW, VOLUME_HIGH)
    cv += _iv(agg_volume_of(Xr) / np.maximum(vol, 1e-9), AGG_FRAC_LOW, AGG_FRAC_HIGH)

    cv += _iv((sio2_dry + na2o_dry) / ss_s, SS_SOLIDS_LOW, SS_SOLIDS_HIGH)
    cv += _iv((sio2_dry / M["SiO2"]) / np.maximum(na2o_dry / M["Na2O"], 1e-12), SS_MS_LOW, SS_MS_HIGH)

    d, _ = nn_domain.kneighbors(domain_scaler.transform(pd.DataFrame(Xr, columns=RAW_VARS)),
                                n_neighbors=1, return_distance=True)
    cv += np.maximum(0.0, (d[:, 0] - DOMAIN_DISTANCE_LIMIT) / DOMAIN_DISTANCE_LIMIT)

    return cv


def rebuild_domain():
    """Feasible seed pool. Volume closure is NOT applied here — observed mixtures
    are batched per cubic metre by definition, and any closure error reflects the
    assumed densities rather than the mixture. It still governs candidates.
    Seeds come from ambient-cured mixtures with their curing forced."""
    global feasible_X, best_lower_seed, SEED_POOL

    ambient_rows = O[O[:, R["Initial curing temp (C)"]] <= AMBIENT_SELECT_MAX].copy()
    ambient_rows[:, R["Initial curing temp (C)"]] = AMBIENT_TEMP
    ambient_rows[:, R["Initial curing time (day)"]] = AMBIENT_TIME

    saved_lo, saved_hi = VOLUME_LOW, VOLUME_HIGH
    globals()["VOLUME_LOW"], globals()["VOLUME_HIGH"] = -1e9, 1e9
    mask = violation_batch(ambient_rows) <= 1e-10
    globals()["VOLUME_LOW"], globals()["VOLUME_HIGH"] = saved_lo, saved_hi

    feasible_X = ambient_rows[mask]

    best_lower_seed = np.full(len(feasible_X), -np.inf)
    bfa = np.zeros(len(feasible_X), dtype=int)
    bsl = np.zeros(len(feasible_X), dtype=int)

    for fa_i in range(len(FA_ARRAY)):
        for slag_i in range(len(SLAG_ARRAY)):
            _, lo = predict_cs_bounds(build_features_batch(feasible_X,
                                                           np.full(len(feasible_X), fa_i),
                                                           np.full(len(feasible_X), slag_i)))
            better = lo > best_lower_seed
            best_lower_seed = np.where(better, lo, best_lower_seed)
            bfa = np.where(better, fa_i, bfa)
            bsl = np.where(better, slag_i, bsl)

    SEED_POOL = np.column_stack([feasible_X, bfa + 0.5, bsl + 0.5])
    return int(len(feasible_X))


n_ambient = int((O[:, R["Initial curing temp (C)"]] <= AMBIENT_SELECT_MAX).sum())
n_feasible = rebuild_domain()

sf = feasible_X[:, R["GGBFS (kg/m3)"]] / np.maximum(
    feasible_X[:, R["FA (kg/m3)"]] + feasible_X[:, R["GGBFS (kg/m3)"]], 1e-9)

print(f"Unique mixtures {len(opt_data)} | ambient-cured {n_ambient} | seed pool {n_feasible}")
print(f"Curing fixed at {AMBIENT_TEMP:.1f} C for {AMBIENT_TIME:.1f} days — no curing energy")
print(f"Seed pool slag fraction: {sf.min():.3f} to {sf.max():.3f} (mean {sf.mean():.3f})")
print()
print(f"Binder           {BINDER_LOW:9.2f} to {BINDER_HIGH:9.2f} kg/m3")
print(f"Activator/binder {AA_BINDER_LOW:9.4f} to {AA_BINDER_HIGH:9.4f}")
print(f"Silicate / NaOH  {SS_SH_LOW:9.4f} to {SS_SH_HIGH:9.4f}")
print(f"Water / binder   {WB_LOW:9.4f} to {WB_HIGH:9.4f}")
print(f"Volume           {VOLUME_LOW:9.4f} to {VOLUME_HIGH:9.4f} m3 (candidates only)")
print(f"Aggregate vol.   {AGG_FRAC_LOW:9.4f} to {AGG_FRAC_HIGH:9.4f}")
print(f"Silicate solids  {SS_SOLIDS_LOW:9.4f} to {SS_SOLIDS_HIGH:9.4f}")
print(f"Silicate Ms      {SS_MS_LOW:9.4f} to {SS_MS_HIGH:9.4f}")
print(f"Domain guard     {GUARD_PERCENTILE:.2f} percentile, limit {DOMAIN_DISTANCE_LIMIT:.4f}")
print()
print(f"Max reachable CS_lower across the ambient seed pool: {best_lower_seed.max():.2f} MPa")

In [ ]:
# =============================================================================
# MOO Step 4 — CO2, cost and energy objective functions
# =============================================================================

import numpy as np
import pandas as pd

# CO2, kg per kg. Alsalman et al. 2021. Sodium silicate on 48% solution basis,
# applied to the solution mass column, verified against the dataset CO2 column.
EF_CO2 = {"FA": 0.004, "GGBFS": 0.052, "Aggregates": 0.0048,
          "SP": 1.880, "NaOH_dry": 1.915, "SS_solution": 0.360}

# Embodied energy, MJ per kg. Alsalman et al. 2021.
EF_ENERGY = {"FA": 0.033, "GGBFS": 0.857, "Aggregates": 0.083,
             "SP": 29.1, "NaOH_dry": 20.5, "SS_solution": 5.371}

# Unit price, USD per kg. Fly ash and GGBFS are US market prices for 2025, where
# fly ash is scarce and costs roughly 0.12 USD/kg against 0.05 for slag. The
# remaining figures follow Katlav and Turk 2026, Table 2. The silicate price is a
# dry-basis figure and is applied to the dry solids. Superplasticizer is a
# placeholder pending a citation.
EF_COST = {"FA": 0.12, "GGBFS": 0.05, "Coarse": 0.014, "Fine": 0.01,
           "NaOH_dry": 0.38, "SS_dry": 0.17, "Water": 0.001, "SP": 1.00}

# Price of the curing energy, USD per MJ. US EIA industrial average retail
# electricity price, 8.62 cents/kWh in 2025, divided by 3.6 MJ per kWh. Only the
# curing term is priced, since material embodied energy is already reflected in
# the material unit prices and would otherwise be double counted.
CURING_ENERGY_PRICE = 0.024

CURING_CO2_SLOPE, CURING_CO2_INTERCEPT = 0.6417, 16.0417
CURING_ENERGY_PER_C_PER_DAY, HEATING_SOLUTION_ENERGY = 2.440, 5.8


def objectives_batch(Xr, sp_price=None, curing_price=None, fa_price=None, slag_price=None):
    """Return CO2 (kg/m3), cost (USD/m3) and embodied energy (MJ/m3).
    Prices are overridable so scenarios can be swept without redefining this."""
    sp_price = EF_COST["SP"] if sp_price is None else sp_price
    curing_price = CURING_ENERGY_PRICE if curing_price is None else curing_price
    fa_price = EF_COST["FA"] if fa_price is None else fa_price
    slag_price = EF_COST["GGBFS"] if slag_price is None else slag_price

    fa, slag = Xr[:, R["FA (kg/m3)"]], Xr[:, R["GGBFS (kg/m3)"]]
    coarse = Xr[:, R["Coarse aggregate (kg/m3)"]]
    fine = Xr[:, R["Fine aggregate (kg in 1m3 mix)"]]
    ss = Xr[:, R["Total Na2SiO3 (kg in 1m3 of mix)"]]
    sio2_dry, na2o_dry = Xr[:, R["SiO2 (Dry)"]], Xr[:, R["Na2O (Dry)"]]
    sh = Xr[:, R["NaOH (Dry)"]]
    sp = Xr[:, R["Superplasticizer (kg in 1m3 mix)"]]
    water = Xr[:, R["Total water (in solutions + additional) (kg in 1m3 mix)"]]
    temp, time = Xr[:, R["Initial curing temp (C)"]], Xr[:, R["Initial curing time (day)"]]

    heat = (temp >= 30.0) & (time > 0.0)

    curing_co2 = np.where(heat, (CURING_CO2_SLOPE * temp - CURING_CO2_INTERCEPT) * time, 0.0)
    curing_energy = np.where(heat, CURING_ENERGY_PER_C_PER_DAY * temp * time
                             + HEATING_SOLUTION_ENERGY, 0.0)

    co2 = (EF_CO2["FA"] * fa + EF_CO2["GGBFS"] * slag + EF_CO2["Aggregates"] * (coarse + fine)
           + EF_CO2["SS_solution"] * ss + EF_CO2["NaOH_dry"] * sh + EF_CO2["SP"] * sp
           + curing_co2)

    energy = (EF_ENERGY["FA"] * fa + EF_ENERGY["GGBFS"] * slag
              + EF_ENERGY["Aggregates"] * (coarse + fine)
              + EF_ENERGY["SS_solution"] * ss + EF_ENERGY["NaOH_dry"] * sh + EF_ENERGY["SP"] * sp
              + curing_energy)

    cost = (fa_price * fa + slag_price * slag + EF_COST["Coarse"] * coarse
            + EF_COST["Fine"] * fine + EF_COST["SS_dry"] * (sio2_dry + na2o_dry)
            + EF_COST["NaOH_dry"] * sh + sp_price * sp + EF_COST["Water"] * water
            + curing_price * curing_energy)

    return co2, cost, energy


# ---- validation against the dataset CO2 column -------------------------------
co2_all, cost_all, energy_all = objectives_batch(df[RAW_VARS].values.astype(float))
co2_reported = pd.to_numeric(
    df["CO2 footprint (kg emision per 1m3 of samples produced and cured)"], errors="coerce")

m = co2_reported.notna()
r = np.corrcoef(co2_all[m], co2_reported[m])[0, 1]
mae = float(np.mean(np.abs(co2_all[m] - co2_reported[m])))

print(f"CO2 reconstruction vs dataset column: r = {r:.6f} | MAE = {mae:.4f} kg/m3")
assert r > 0.999, "CO2 reconstruction does not match the dataset column."

co2_f, cost_f, energy_f = objectives_batch(feasible_X)
summary = pd.DataFrame({"CO2_kg_m3": co2_f, "Cost_USD_m3": cost_f, "Energy_MJ_m3": energy_f})

print(f"\nFly ash {EF_COST['FA']:.3f} vs GGBFS {EF_COST['GGBFS']:.3f} USD/kg "
      f"(ratio {EF_COST['FA'] / EF_COST['GGBFS']:.2f}) | curing {CURING_ENERGY_PRICE:.3f} USD/MJ")
print("Objectives across the feasible domain")
display(summary.describe().loc[["min", "mean", "max"]].round(2))
print("\nPearson correlation")
display(summary.corr().round(4))

In [ ]:
# =============================================================================
# MOO Step 5 — NSGA-II, maximize CS lower bound, minimize CO2 and cost
# =============================================================================

!pip install -q pymoo

import numpy as np
import pandas as pd
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.optimize import minimize
from pymoo.indicators.hv import HV

POP_SIZE = 600
N_GEN = 1000
N_SEEDS = 5
PERTURB = 0.02
CONVERGENCE_TOL_PCT = 3.0

# reference point for hypervolume: worse than any attainable value on each axis
REF_POINT = np.array([0.0, 320.0, 140.0])

N_RAW = len(RAW_VARS)
LOW_FULL = np.concatenate([LOW, [0.0, 0.0]])
HIGH_FULL = np.concatenate([HIGH, [len(FA_ARRAY) - 1e-6, len(SLAG_ARRAY) - 1e-6]])


def split_decision(Z):
    """Separate the raw mixture part from the two floored source indices."""
    Xr = Z[:, :N_RAW]
    fa_idx = np.clip(np.floor(Z[:, N_RAW]).astype(int), 0, len(FA_ARRAY) - 1)
    slag_idx = np.clip(np.floor(Z[:, N_RAW + 1]).astype(int), 0, len(SLAG_ARRAY) - 1)
    return Xr, fa_idx, slag_idx


class MixtureProblem(Problem):
    def __init__(self):
        super().__init__(n_var=N_RAW + 2, n_obj=3, n_ieq_constr=1,
                         xl=LOW_FULL, xu=HIGH_FULL)

    def _evaluate(self, Z, out, *args, **kwargs):
        Xr, fa_idx, slag_idx = split_decision(Z)
        co2, cost, _ = objectives_batch(Xr)
        _, lower = predict_cs_bounds(build_features_batch(Xr, fa_idx, slag_idx))

        # strength is maximized, so it enters negated
        out["F"] = np.column_stack([-lower, co2, cost])
        out["G"] = violation_batch(Xr).reshape(-1, 1)


def non_dominated(F, Z):
    keep = np.ones(len(F), dtype=bool)
    for i in range(len(F)):
        if keep[i] and (np.all(F <= F[i], axis=1) & np.any(F < F[i], axis=1)).any():
            keep[i] = False
    F_nd, Z_nd = F[keep], Z[keep]
    _, uniq = np.unique(F_nd.round(4), axis=0, return_index=True)
    return F_nd[uniq], Z_nd[uniq]


hv_indicator = HV(ref_point=REF_POINT)
seed_rows, F_all, Z_all = [], [], []

for s in range(N_SEEDS):
    rng = np.random.default_rng(SEED + s)
    Z0 = SEED_POOL[rng.integers(0, len(SEED_POOL), POP_SIZE)].copy()
    Z0 += rng.normal(0.0, PERTURB, Z0.shape) * (HIGH_FULL - LOW_FULL)[None, :]
    Z0 = np.clip(Z0, LOW_FULL, HIGH_FULL)

    res = minimize(MixtureProblem(),
                   NSGA2(pop_size=POP_SIZE, sampling=Z0,
                         crossover=SBX(prob=0.9, eta=15), mutation=PM(eta=20),
                         eliminate_duplicates=True),
                   ("n_gen", N_GEN), seed=SEED + s, verbose=False)

    if res.F is None or len(res.F) == 0:
        seed_rows.append({"seed": s, "n": 0})
        continue

    F, Z = np.atleast_2d(res.F), np.atleast_2d(res.X)
    F_all.append(F)
    Z_all.append(Z)

    seed_rows.append({"seed": s, "n": len(F),
                      "CS_max": -F[:, 0].min(), "CO2_min": F[:, 1].min(),
                      "Cost_min": F[:, 2].min(), "hypervolume": hv_indicator(F)})

    print(f"seed {s} -> {len(F)} solutions | CS_lower max {-F[:, 0].min():.2f} "
          f"| CO2 min {F[:, 1].min():.2f} | cost min {F[:, 2].min():.2f}")

assert F_all, "No feasible solutions in any run."

F_nd, Z_nd = non_dominated(np.vstack(F_all), np.vstack(Z_all))
order = np.argsort(-F_nd[:, 0])
F_nd, Z_nd = F_nd[order], Z_nd[order]

Xr_nd, fa_nd, slag_nd = split_decision(Z_nd)
point_nd, lower_nd = predict_cs_bounds(build_features_batch(Xr_nd, fa_nd, slag_nd))
co2_nd, cost_nd, energy_nd = objectives_batch(Xr_nd)

results = {"F": F_nd, "Z": Z_nd, "Xr": Xr_nd, "fa_idx": fa_nd, "slag_idx": slag_nd,
           "CS_point": point_nd, "CS_lower": lower_nd,
           "CO2": co2_nd, "Cost": cost_nd, "Energy": energy_nd}
results_full = {k: (v.copy() if hasattr(v, "copy") else v) for k, v in results.items()}

seed_df = pd.DataFrame(seed_rows)
hv_cv = 100 * seed_df["hypervolume"].std() / seed_df["hypervolume"].mean()

print("\nPer-run fronts")
display(seed_df.round(3))
print(f"\nHypervolume CV across seeds: {hv_cv:.3f}%  ->  "
      f"{'converged' if hv_cv < CONVERGENCE_TOL_PCT else 'NOT converged'}")

print(f"\nMerged front: {len(F_nd)} non-dominated solutions")
print(f"  CS lower bound  {lower_nd.min():7.2f} to {lower_nd.max():7.2f} MPa")
print(f"  CS point        {point_nd.min():7.2f} to {point_nd.max():7.2f} MPa")
print(f"  CO2             {co2_nd.min():7.2f} to {co2_nd.max():7.2f} kg/m3")
print(f"  Cost            {cost_nd.min():7.2f} to {cost_nd.max():7.2f} USD/m3")
print(f"  Energy          {energy_nd.min():7.2f} to {energy_nd.max():7.2f} MJ/m3")

In [ ]:
# =============================================================================
# MOO Step 6 — Front diagnostics, extrapolation and margin behaviour
# =============================================================================

import numpy as np
import pandas as pd

Xr = results["Xr"]
lower, point = results["CS_lower"], results["CS_point"]
margin = point - lower

d, _ = nn_domain.kneighbors(domain_scaler.transform(pd.DataFrame(Xr, columns=RAW_VARS)),
                            n_neighbors=1, return_distance=True)
nn_dist = d[:, 0]

obs_cs = pd.to_numeric(df["converted CS_Mpa"], errors="coerce")
amb_mask = pd.to_numeric(df["Initial curing temp (C)"], errors="coerce") <= AMBIENT_SELECT_MAX
obs_cs_amb = obs_cs[amb_mask]

slag_frac = Xr[:, R["GGBFS (kg/m3)"]] / np.maximum(
    Xr[:, R["FA (kg/m3)"]] + Xr[:, R["GGBFS (kg/m3)"]], 1e-9)

print(f"Observed CS maximum, all rows: {obs_cs.max():.2f} MPa")
print(f"Observed CS maximum, ambient-cured rows only: {obs_cs_amb.max():.2f} MPa "
      f"(99th pct {obs_cs_amb.quantile(0.99):.2f})")
print(f"Front solutions above the ambient observed max (point): {int((point > obs_cs_amb.max()).sum())}")
print(f"Front solutions above it (lower bound): {int((lower > obs_cs_amb.max()).sum())}")
print(f"Domain guard limit: {DOMAIN_DISTANCE_LIMIT:.4f}")
print(f"Margin correlation with strength: {np.corrcoef(lower, margin)[0, 1]:.4f}")
print()

diag = pd.DataFrame({"CS_lower": lower, "margin": margin, "nn_dist": nn_dist,
                     "CO2": results["CO2"], "Cost": results["Cost"],
                     "Energy": results["Energy"], "slag_frac": slag_frac})

bands = pd.cut(diag["CS_lower"], bins=[0, 20, 30, 40, 50, 60, np.inf],
               labels=["<20", "20-30", "30-40", "40-50", "50-60", ">60"])

summary = diag.groupby(bands, observed=True).agg(
    n=("CS_lower", "size"), margin_mean=("margin", "mean"),
    nn_dist_mean=("nn_dist", "mean"), CO2_min=("CO2", "min"),
    Cost_min=("Cost", "min"), slag_frac_mean=("slag_frac", "mean"),
    slag_frac_max=("slag_frac", "max"))
summary["pct_at_guard"] = diag.groupby(bands, observed=True)["nn_dist"].apply(
    lambda s: 100 * (s >= 0.99 * DOMAIN_DISTANCE_LIMIT).mean())
summary["observed_rows"] = [int(((obs_cs_amb >= lo) & (obs_cs_amb < hi)).sum())
                            for lo, hi in [(0, 20), (20, 30), (30, 40), (40, 50),
                                           (50, 60), (60, 999)][:len(summary)]]

print("Front behaviour by strength band, against ambient observed data density")
display(summary.round(4))

print(f"\nSlag fraction across the front: {slag_frac.min():.3f} to {slag_frac.max():.3f} "
      f"(mean {slag_frac.mean():.3f})")
print(f"Solutions with slag above 25% of binder: {int((slag_frac > 0.25).sum())} of {len(slag_frac)}")
print(f"corr(slag fraction, CS_lower) = {np.corrcoef(slag_frac, lower)[0, 1]:.4f}")

In [ ]:
# =============================================================================
# MOO Step 6b — Cap at the limit of data support and screen uncertain solutions
# =============================================================================

import numpy as np
import pandas as pd

obs_cs = pd.to_numeric(df["converted CS_Mpa"], errors="coerce")
amb_mask = pd.to_numeric(df["Initial curing temp (C)"], errors="coerce") <= AMBIENT_SELECT_MAX
CS_CAP = float(obs_cs[amb_mask].quantile(0.99))
MARGIN_MAX = 8.0

margin_full = results_full["CS_point"] - results_full["CS_lower"]
keep = (results_full["CS_lower"] <= CS_CAP) & (margin_full <= MARGIN_MAX)

for k in ["F", "Z", "Xr", "fa_idx", "slag_idx", "CS_point", "CS_lower", "CO2", "Cost", "Energy"]:
    results[k] = results_full[k][keep]

lower, point = results["CS_lower"], results["CS_point"]
margin = point - lower
slag_frac = results["Xr"][:, R["GGBFS (kg/m3)"]] / np.maximum(
    results["Xr"][:, R["FA (kg/m3)"]] + results["Xr"][:, R["GGBFS (kg/m3)"]], 1e-9)

print(f"Cap at the 99th percentile of ambient observed CS = {CS_CAP:.2f} MPa")
print(f"Margin screen at {MARGIN_MAX:.1f} MPa (held-out mean 5.08)")
print(f"  removed  {int((~keep).sum())} of {len(keep)} | retained {int(keep.sum())}")
print()
print(f"  CS lower bound  {lower.min():7.2f} to {lower.max():7.2f} MPa")
print(f"  CS point        {point.min():7.2f} to {point.max():7.2f} MPa")
print(f"  margin          {margin.min():7.2f} to {margin.max():7.2f} (mean {margin.mean():.2f})")
print(f"  CO2             {results['CO2'].min():7.2f} to {results['CO2'].max():7.2f} kg/m3")
print(f"  Cost            {results['Cost'].min():7.2f} to {results['Cost'].max():7.2f} USD/m3")
print(f"  Energy          {results['Energy'].min():7.2f} to {results['Energy'].max():7.2f} MJ/m3")
print(f"  slag fraction   {slag_frac.min():7.3f} to {slag_frac.max():7.3f} (mean {slag_frac.mean():.3f})")
print(f"  margin correlation with strength: {np.corrcoef(lower, margin)[0, 1]:.4f}")

print("\nPreview of the three table strengths")
for t in [30, 40, 50]:
    if lower.max() < t:
        print(f"  {t} MPa -> not reachable")
        continue
    near = np.abs(lower - t) <= 1.0
    k = int(np.flatnonzero(near)[np.argmin(results["CO2"][near])]) if near.any() \
        else int(np.abs(lower - t).argmin())
    print(f"  nearest to {t} MPa -> CS_lower {lower[k]:6.2f} | CS_point {point[k]:6.2f} "
          f"| margin {margin[k]:5.2f} | CO2 {results['CO2'][k]:7.2f} "
          f"| cost {results['Cost'][k]:6.2f} | slag/binder {slag_frac[k]:.3f}")

In [ ]:
# =============================================================================
# MOO Step 7 — Reliability flagging (no deletion) and TOPSIS selection
# =============================================================================

import numpy as np
import pandas as pd

NN_DISTANCE_MAX = DOMAIN_DISTANCE_LIMIT
BOUND_TOL = 0.02
MAX_NEAR_BOUNDS = 6

# variables whose optimum genuinely sits at a bound are excluded from the count
BOUNDARY_EXEMPT = {"GGBFS (kg/m3)", "Superplasticizer (kg in 1m3 mix)",
                   "NaOH (Dry)", "Total Na2SiO3 (kg in 1m3 of mix)",
                   "Initial curing time (day)", "Initial curing temp (C)"}

TOPSIS_WEIGHTS = {"CS": 0.40, "CO2": 0.25, "Cost": 0.20, "Energy": 0.15}
_w = sum(TOPSIS_WEIGHTS.values())
TOPSIS_WEIGHTS = {k: v / _w for k, v in TOPSIS_WEIGHTS.items()}

short_var = {
    "FA (kg/m3)": "FA", "GGBFS (kg/m3)": "GGBFS",
    "Coarse aggregate (kg/m3)": "Coarse agg", "Fine aggregate (kg in 1m3 mix)": "Fine agg",
    "Total Na2SiO3 (kg in 1m3 of mix)": "Na2SiO3", "SiO2 (Dry)": "SiO2 dry",
    "Na2O (Dry)": "Na2O dry", "NaOH (Dry)": "NaOH dry",
    "Concentration (M) NaOH": "NaOH molarity", "Superplasticizer (kg in 1m3 mix)": "SP",
    "Total water (in solutions + additional) (kg in 1m3 mix)": "Water",
    "Initial curing time (day)": "Curing time", "Initial curing temp (C)": "Curing temp",
}


def topsis_closeness(matrix, directions, weights):
    norm = np.sqrt((matrix ** 2).sum(axis=0))
    norm[norm == 0.0] = 1e-12
    V = (matrix / norm) * weights

    ideal_best = np.where(directions > 0, V.max(axis=0), V.min(axis=0))
    ideal_worst = np.where(directions > 0, V.min(axis=0), V.max(axis=0))

    d_best = np.sqrt(((V - ideal_best) ** 2).sum(axis=1))
    d_worst = np.sqrt(((V - ideal_worst) ** 2).sum(axis=1))

    return d_worst / np.maximum(d_best + d_worst, 1e-12)


Xr = results["Xr"]

pareto_flagged_df = pd.DataFrame(Xr, columns=RAW_VARS)
pareto_flagged_df["CS_point_MPa"] = results["CS_point"]
pareto_flagged_df["CS_lower_MPa"] = results["CS_lower"]
pareto_flagged_df["Margin_MPa"] = results["CS_point"] - results["CS_lower"]
pareto_flagged_df["CO2_kg_m3"] = results["CO2"]
pareto_flagged_df["Cost_USD_m3"] = results["Cost"]
pareto_flagged_df["Energy_MJ_m3"] = results["Energy"]
pareto_flagged_df["FA_source"] = results["fa_idx"]
pareto_flagged_df["Slag_source"] = results["slag_idx"]
pareto_flagged_df["Slag_fraction"] = Xr[:, R["GGBFS (kg/m3)"]] / np.maximum(
    Xr[:, R["FA (kg/m3)"]] + Xr[:, R["GGBFS (kg/m3)"]], 1e-9)

pareto_flagged_df["CO2_per_MPa"] = pareto_flagged_df["CO2_kg_m3"] / pareto_flagged_df["CS_lower_MPa"]
pareto_flagged_df["Cost_per_MPa"] = pareto_flagged_df["Cost_USD_m3"] / pareto_flagged_df["CS_lower_MPa"]
pareto_flagged_df["Energy_per_MPa"] = pareto_flagged_df["Energy_MJ_m3"] / pareto_flagged_df["CS_lower_MPa"]

d, nn_idx = nn_domain.kneighbors(domain_scaler.transform(pd.DataFrame(Xr, columns=RAW_VARS)),
                                 n_neighbors=1, return_distance=True)
pareto_flagged_df["NN_distance"] = d[:, 0]
pareto_flagged_df["Nearest_real_mix_index"] = nn_idx[:, 0]

near = np.zeros(len(pareto_flagged_df), dtype=int)
for j, v in enumerate(RAW_VARS):
    if v in BOUNDARY_EXEMPT:
        continue
    span = max(HIGH[j] - LOW[j], 1e-9)
    near += ((Xr[:, j] <= LOW[j] + BOUND_TOL * span) |
             (Xr[:, j] >= HIGH[j] - BOUND_TOL * span)).astype(int)

pareto_flagged_df["n_near_bounds_scored"] = near
pareto_flagged_df["pass_local_domain"] = pareto_flagged_df["NN_distance"] <= NN_DISTANCE_MAX + 1e-9
pareto_flagged_df["pass_boundary_pressure"] = near <= MAX_NEAR_BOUNDS
pareto_flagged_df["conservative_pass"] = (pareto_flagged_df["pass_local_domain"] &
                                          pareto_flagged_df["pass_boundary_pressure"])

matrix = pareto_flagged_df[["CS_lower_MPa", "CO2_kg_m3", "Cost_USD_m3", "Energy_MJ_m3"]].to_numpy(float)
directions = np.array([1.0, -1.0, -1.0, -1.0])
weights = np.array([TOPSIS_WEIGHTS["CS"], TOPSIS_WEIGHTS["CO2"],
                    TOPSIS_WEIGHTS["Cost"], TOPSIS_WEIGHTS["Energy"]])
pareto_flagged_df["TOPSIS_closeness"] = topsis_closeness(matrix, directions, weights)

print(f"TOPSIS weights: CS {TOPSIS_WEIGHTS['CS']:.2f} | CO2 {TOPSIS_WEIGHTS['CO2']:.2f} "
      f"| cost {TOPSIS_WEIGHTS['Cost']:.2f} | energy {TOPSIS_WEIGHTS['Energy']:.2f}")
print(f"Reliability thresholds: NN distance <= {NN_DISTANCE_MAX:.4f}, at most {MAX_NEAR_BOUNDS} "
      f"scored variables within {int(BOUND_TOL * 100)}% of a bound")
print(f"Exempt from the bound count: {', '.join(sorted(short_var[v] for v in BOUNDARY_EXEMPT))}")
print()

display(pd.DataFrame([{
    "n_solutions": len(pareto_flagged_df),
    "nn_dist_max": pareto_flagged_df["NN_distance"].max(),
    "near_bounds_median": int(np.median(near)),
    "near_bounds_max": int(near.max()),
    "pass_local_domain": int(pareto_flagged_df["pass_local_domain"].sum()),
    "pass_boundary_pressure": int(pareto_flagged_df["pass_boundary_pressure"].sum()),
    "conservative_pass": int(pareto_flagged_df["conservative_pass"].sum()),
}]).set_index("n_solutions").round(4))

reliable = pareto_flagged_df[pareto_flagged_df["conservative_pass"]]
pick_from = reliable if len(reliable) > 0 else pareto_flagged_df
recommended_idx = int(pick_from["TOPSIS_closeness"].idxmax())
rec = pareto_flagged_df.loc[recommended_idx]

print(f"\nTOPSIS recommended solution (index {recommended_idx})")
print(f"  CS lower {rec['CS_lower_MPa']:.2f} | CS point {rec['CS_point_MPa']:.2f} "
      f"| margin {rec['Margin_MPa']:.2f} MPa")
print(f"  CO2 {rec['CO2_kg_m3']:.2f} kg/m3 | cost {rec['Cost_USD_m3']:.2f} USD/m3 "
      f"| energy {rec['Energy_MJ_m3']:.2f} MJ/m3")
print(f"  FA {rec['FA (kg/m3)']:.1f} | GGBFS {rec['GGBFS (kg/m3)']:.1f} kg/m3 "
      f"| slag/binder {rec['Slag_fraction']:.3f}")
print(f"  closeness {rec['TOPSIS_closeness']:.4f} | reliable {bool(rec['conservative_pass'])}")

front_ambient = pareto_flagged_df.copy()
rec_idx_ambient = recommended_idx
print(f"Snapshot saved: front_ambient ({len(front_ambient)} rows), rec_idx_ambient = {rec_idx_ambient}")

In [ ]:
# =============================================================================
# MOO Step 8 — Fly ash to slag price ratio sensitivity, ambient curing
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.optimize import minimize
from google.colab import files

# slag is held at its US market price and the fly ash price is varied, so the
# ratio spans Katlav's inverted case (0.26) through to the US case (2.40)
SLAG_PRICE_FIXED = 0.05
PRICE_RATIOS = [0.26, 0.50, 0.75, 1.00, 1.50, 2.40, 4.00]

N_SEEDS_SENS = 3
N_GEN_SENS = 800

BASE_FA, BASE_SLAG = EF_COST["FA"], EF_COST["GGBFS"]
rows = []

for ratio in PRICE_RATIOS:
    EF_COST["GGBFS"] = SLAG_PRICE_FIXED
    EF_COST["FA"] = ratio * SLAG_PRICE_FIXED

    F_s, Z_s = [], []
    for s in range(N_SEEDS_SENS):
        rng = np.random.default_rng(SEED + s)
        Z0 = SEED_POOL[rng.integers(0, len(SEED_POOL), POP_SIZE)].copy()
        Z0 += rng.normal(0.0, PERTURB, Z0.shape) * (HIGH_FULL - LOW_FULL)[None, :]
        Z0 = np.clip(Z0, LOW_FULL, HIGH_FULL)

        res = minimize(MixtureProblem(),
                       NSGA2(pop_size=POP_SIZE, sampling=Z0,
                             crossover=SBX(prob=0.9, eta=15), mutation=PM(eta=20),
                             eliminate_duplicates=True),
                       ("n_gen", N_GEN_SENS), seed=SEED + s, verbose=False)

        if res.F is not None and len(res.F) > 0:
            F_s.append(np.atleast_2d(res.F))
            Z_s.append(np.atleast_2d(res.X))

    if not F_s:
        print(f"ratio {ratio:.2f} -> no feasible solutions")
        continue

    Fn, Zn = non_dominated(np.vstack(F_s), np.vstack(Z_s))
    Xr_s, fa_s, sl_s = split_decision(Zn)
    pt, lo = predict_cs_bounds(build_features_batch(Xr_s, fa_s, sl_s))
    co2_s, cost_s, en_s = objectives_batch(Xr_s)

    keep = (lo <= CS_CAP) & ((pt - lo) <= MARGIN_MAX)
    Xr_s, lo, pt = Xr_s[keep], lo[keep], pt[keep]
    co2_s, cost_s, en_s = co2_s[keep], cost_s[keep], en_s[keep]

    slag = Xr_s[:, R["GGBFS (kg/m3)"]]
    fa = Xr_s[:, R["FA (kg/m3)"]]
    frac = slag / np.maximum(fa + slag, 1e-9)

    mat = np.column_stack([lo, co2_s, cost_s, en_s])
    close = topsis_closeness(mat, np.array([1.0, -1.0, -1.0, -1.0]),
                             np.array([0.40, 0.25, 0.20, 0.15]))
    k = int(np.argmax(close))

    rows.append({"FA_slag_ratio": ratio, "FA_price": EF_COST["FA"], "n": len(lo),
                 "CS_max": lo.max(), "CO2_min": co2_s.min(), "Cost_min": cost_s.min(),
                 "slag_frac_mean": frac.mean(), "slag_frac_median": np.median(frac),
                 "pct_slag_gt25": 100 * (frac > 0.25).mean(),
                 "corr_slag_CS": np.corrcoef(frac, lo)[0, 1],
                 "rec_CS": lo[k], "rec_slag_frac": frac[k], "rec_FA": fa[k],
                 "rec_GGBFS": slag[k], "rec_CO2": co2_s[k], "rec_cost": cost_s[k]})

    print(f"ratio {ratio:4.2f} (FA {EF_COST['FA']:.3f}) -> n {len(lo):4d} "
          f"| slag_frac mean {frac.mean():.3f} | >25% slag {100 * (frac > 0.25).mean():5.1f}% "
          f"| rec slag {frac[k]:.3f} at CS {lo[k]:.1f}")

EF_COST["FA"], EF_COST["GGBFS"] = BASE_FA, BASE_SLAG
price_sensitivity = pd.DataFrame(rows)

print(f"\nPrices restored to FA {EF_COST['FA']:.3f} / GGBFS {EF_COST['GGBFS']:.3f} USD/kg")
display(price_sensitivity.round(4))

fig, (axL, axR) = plt.subplots(1, 2, figsize=(15.5, 6.2))

axL.plot(price_sensitivity["FA_slag_ratio"], price_sensitivity["slag_frac_mean"],
         marker="o", markersize=9, linewidth=2.2, color="#0033CC", label="Front mean")
axL.plot(price_sensitivity["FA_slag_ratio"], price_sensitivity["rec_slag_frac"],
         marker="s", markersize=9, linewidth=2.2, color="#D00000", label="TOPSIS recommended")
axL.axvline(2.40, color="black", ls="--", lw=1.4, alpha=0.7)
axL.text(2.40, axL.get_ylim()[1] * 0.96, " US 2025", fontsize=12, va="top")
axL.axvline(0.26, color="gray", ls=":", lw=1.4, alpha=0.7)
axL.text(0.26, axL.get_ylim()[1] * 0.96, " Katlav", fontsize=12, va="top")

axL.set_xlabel("Fly ash to slag price ratio", fontsize=16, labelpad=9)
axL.set_ylabel("Slag fraction of binder", fontsize=16, labelpad=9)
axL.set_title("Precursor choice against relative price", fontsize=17, fontweight="bold", pad=10)
axL.tick_params(axis="both", labelsize=13)
axL.grid(linestyle="--", linewidth=0.55, alpha=0.22)
axL.legend(fontsize=12, frameon=True, loc="lower right")

axR.plot(price_sensitivity["FA_slag_ratio"], price_sensitivity["pct_slag_gt25"],
         marker="o", markersize=9, linewidth=2.2, color="#00802B")
axR.set_xlabel("Fly ash to slag price ratio", fontsize=16, labelpad=9)
axR.set_ylabel("Front share with slag above 25% (%)", fontsize=16, labelpad=9)
axR.set_title("Share of blended solutions", fontsize=17, fontweight="bold", pad=10)
axR.tick_params(axis="both", labelsize=13)
axR.set_ylim(0, 100)
axR.grid(linestyle="--", linewidth=0.55, alpha=0.22)

for ax in (axL, axR):
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.05)

fig.tight_layout(w_pad=2.6)

FIG_PATH = "/content/MOO_price_ratio_sensitivity.png"
fig.savefig(FIG_PATH, dpi=600, facecolor="white", bbox_inches="tight")
plt.show()

files.download(FIG_PATH)

price_ambient = price_sensitivity.copy()
print(f"Snapshot saved: price_ambient ({len(price_ambient)} rows)")

## **B: Curing Heat Cost**

In [ ]:
# =============================================================================
# MOO Step 3 — Search domain and seed pool, heat curing allowed
# =============================================================================

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors

Q_LOW, Q_HIGH = 0.01, 0.99

VOLUME_LOW, VOLUME_HIGH = 0.97, 1.03          # physical closure, not a data quantile
AMBIENT_TEMP = 25.0
AMBIENT_TEMP_LIMIT = 30.0
HEAT_TEMP_LOW, HEAT_TEMP_HIGH = 40.0, 100.0   # 1st and 99th percentile of heat-cured rows
GUARD_PERCENTILE = 0.75

opt_data = (df[RAW_VARS].apply(pd.to_numeric, errors="coerce")
            .dropna().drop_duplicates().reset_index(drop=True))

DENSITY = {"FA (kg/m3)": 2300.0, "GGBFS (kg/m3)": 2860.0,
           "Coarse aggregate (kg/m3)": 2740.0, "Fine aggregate (kg in 1m3 mix)": 2640.0,
           "Superplasticizer (kg in 1m3 mix)": 1100.0,
           "Total water (in solutions + additional) (kg in 1m3 mix)": 1000.0}
SILICATE_SOLID_DENSITY = 2400.0
NAOH_SOLID_DENSITY = 2130.0


def volume_of(Xr):
    v = sum(Xr[:, R[k]] / DENSITY[k] for k in DENSITY)
    v += (Xr[:, R["SiO2 (Dry)"]] + Xr[:, R["Na2O (Dry)"]]) / SILICATE_SOLID_DENSITY
    v += Xr[:, R["NaOH (Dry)"]] / NAOH_SOLID_DENSITY
    return v


def agg_volume_of(Xr):
    return (Xr[:, R["Coarse aggregate (kg/m3)"]] / DENSITY["Coarse aggregate (kg/m3)"] +
            Xr[:, R["Fine aggregate (kg in 1m3 mix)"]] / DENSITY["Fine aggregate (kg in 1m3 mix)"])


O = opt_data.values.astype(float)
q = lambda a: np.nanquantile(a, [Q_LOW, Q_HIGH])

bounds_df = pd.DataFrame({"low": opt_data.quantile(Q_LOW), "high": opt_data.quantile(Q_HIGH)})
bounds_df["high"] = np.where(bounds_df["high"] <= bounds_df["low"],
                             bounds_df["low"] + 1e-9, bounds_df["high"])
LOW = bounds_df.loc[RAW_VARS, "low"].values
HIGH = bounds_df.loc[RAW_VARS, "high"].values

binder_o = O[:, R["FA (kg/m3)"]] + O[:, R["GGBFS (kg/m3)"]]
vol_o = volume_of(O)
ss_o, sh_o = O[:, R["Total Na2SiO3 (kg in 1m3 of mix)"]], O[:, R["NaOH (Dry)"]]
water_o = O[:, R["Total water (in solutions + additional) (kg in 1m3 mix)"]]

BINDER_LOW, BINDER_HIGH = q(binder_o)
AA_BINDER_LOW, AA_BINDER_HIGH = q((ss_o + sh_o) / binder_o)
SS_SH_LOW, SS_SH_HIGH = q(np.where(sh_o > 0, ss_o / np.where(sh_o > 0, sh_o, 1), np.nan))
WB_LOW, WB_HIGH = q(water_o / np.maximum(binder_o, 1e-9))
AGG_FRAC_LOW, AGG_FRAC_HIGH = q(agg_volume_of(O) / vol_o)
SS_SOLIDS_LOW, SS_SOLIDS_HIGH = q((O[:, R["SiO2 (Dry)"]] + O[:, R["Na2O (Dry)"]]) / ss_o)
SS_MS_LOW, SS_MS_HIGH = q((O[:, R["SiO2 (Dry)"]] / M["SiO2"]) / (O[:, R["Na2O (Dry)"]] / M["Na2O"]))

heat_mask = O[:, R["Initial curing temp (C)"]] > AMBIENT_TEMP_LIMIT
# the 99th percentile of heat duration is 28 days, an entry that almost
# certainly records total age rather than the heating cycle. The distribution is
# 1 day at the median and 3 days at the 95th percentile, so the upper limit is
# set there.
HEAT_TIME_LOW = float(np.nanquantile(O[heat_mask, R["Initial curing time (day)"]], Q_LOW))
HEAT_TIME_HIGH = float(np.nanquantile(O[heat_mask, R["Initial curing time (day)"]], 0.95))

# curing temp is bimodal, so its global percentiles are meaningless as a box
# bound. Set the search range explicitly and let the curing rules carve out the
# two valid regions inside it.
LOW[R["Initial curing temp (C)"]] = AMBIENT_TEMP
HIGH[R["Initial curing temp (C)"]] = HEAT_TEMP_HIGH
LOW[R["Initial curing time (day)"]] = 0.0
HIGH[R["Initial curing time (day)"]] = HEAT_TIME_HIGH

domain_scaler = MinMaxScaler().fit(opt_data)
nn_domain = NearestNeighbors(n_neighbors=6).fit(domain_scaler.transform(opt_data))
dist_o, _ = nn_domain.kneighbors(domain_scaler.transform(opt_data))
DOMAIN_DISTANCE_LIMIT = float(np.quantile(dist_o[:, -1], GUARD_PERCENTILE))


def _iv(v, low, high):
    scale = max(abs(high - low), 1e-9)
    return (np.maximum(low - v, 0.0) + np.maximum(v - high, 0.0)) / scale

def snap_curing(Xr):
    """Any solution below the ambient limit is set to the Scenario 1 ambient
    condition, so ambient is a single fixed state rather than a free warm band."""
    Xr = Xr.copy()
    amb = Xr[:, R["Initial curing temp (C)"]] < AMBIENT_TEMP_LIMIT
    Xr[amb, R["Initial curing temp (C)"]] = AMBIENT_TEMP
    Xr[amb, R["Initial curing time (day)"]] = 0.0
    return Xr

def violation_batch(Xr):
    """Total normalized constraint violation. Zero means feasible.
    Curing is a decision variable here, so the curing rules are enforced."""
    cv = np.zeros(len(Xr))

    for j in range(len(RAW_VARS)):
        cv += _iv(Xr[:, j], LOW[j], HIGH[j])

    fa, slag = Xr[:, R["FA (kg/m3)"]], Xr[:, R["GGBFS (kg/m3)"]]
    ss, sh = Xr[:, R["Total Na2SiO3 (kg in 1m3 of mix)"]], Xr[:, R["NaOH (Dry)"]]
    sio2_dry, na2o_dry = Xr[:, R["SiO2 (Dry)"]], Xr[:, R["Na2O (Dry)"]]
    water = Xr[:, R["Total water (in solutions + additional) (kg in 1m3 mix)"]]
    temp, time = Xr[:, R["Initial curing temp (C)"]], Xr[:, R["Initial curing time (day)"]]

    binder = np.maximum(fa + slag, 1e-9)
    ss_s, sh_s = np.maximum(ss, 1e-9), np.maximum(sh, 1e-9)

    cv += _iv(binder, BINDER_LOW, BINDER_HIGH)
    cv += _iv((ss + sh) / binder, AA_BINDER_LOW, AA_BINDER_HIGH)
    cv += _iv(ss / sh_s, SS_SH_LOW, SS_SH_HIGH)
    cv += _iv(water / binder, WB_LOW, WB_HIGH)

    vol = volume_of(Xr)
    cv += _iv(vol, VOLUME_LOW, VOLUME_HIGH)
    cv += _iv(agg_volume_of(Xr) / np.maximum(vol, 1e-9), AGG_FRAC_LOW, AGG_FRAC_HIGH)

    cv += _iv((sio2_dry + na2o_dry) / ss_s, SS_SOLIDS_LOW, SS_SOLIDS_HIGH)
    cv += _iv((sio2_dry / M["SiO2"]) / np.maximum(na2o_dry / M["Na2O"], 1e-12), SS_MS_LOW, SS_MS_HIGH)

    # curing time is zero below 30 C in the training data, and heat curing is
    # only observed between 40 and 100 C
    ambient = temp < AMBIENT_TEMP_LIMIT
    cv += np.where(ambient, time / max(HEAT_TIME_HIGH, 1e-9), 0.0)
    cv += np.where(~ambient, _iv(temp, HEAT_TEMP_LOW, HEAT_TEMP_HIGH), 0.0)
    cv += np.where(~ambient, _iv(time, HEAT_TIME_LOW, HEAT_TIME_HIGH), 0.0)

    d, _ = nn_domain.kneighbors(domain_scaler.transform(pd.DataFrame(Xr, columns=RAW_VARS)),
                                n_neighbors=1, return_distance=True)
    cv += np.maximum(0.0, (d[:, 0] - DOMAIN_DISTANCE_LIMIT) / DOMAIN_DISTANCE_LIMIT)

    return cv


def rebuild_domain():
    """Feasible seed pool. Volume closure is NOT applied here — observed mixtures
    are batched per cubic metre by definition, and any closure error reflects the
    assumed densities rather than the mixture. It still governs candidates."""
    global feasible_X, best_lower_seed, SEED_POOL

    saved_lo, saved_hi = VOLUME_LOW, VOLUME_HIGH
    globals()["VOLUME_LOW"], globals()["VOLUME_HIGH"] = -1e9, 1e9
    O_snap = snap_curing(O)
    mask = violation_batch(O_snap) <= 1e-10
    globals()["VOLUME_LOW"], globals()["VOLUME_HIGH"] = saved_lo, saved_hi

    feasible_X = O_snap[mask]

    best_lower_seed = np.full(len(feasible_X), -np.inf)
    bfa = np.zeros(len(feasible_X), dtype=int)
    bsl = np.zeros(len(feasible_X), dtype=int)

    for fa_i in range(len(FA_ARRAY)):
        for slag_i in range(len(SLAG_ARRAY)):
            _, lo = predict_cs_bounds(build_features_batch(feasible_X,
                                                           np.full(len(feasible_X), fa_i),
                                                           np.full(len(feasible_X), slag_i)))
            better = lo > best_lower_seed
            best_lower_seed = np.where(better, lo, best_lower_seed)
            bfa = np.where(better, fa_i, bfa)
            bsl = np.where(better, slag_i, bsl)

    SEED_POOL = np.column_stack([feasible_X, bfa + 0.5, bsl + 0.5])
    return int(len(feasible_X))


n_feasible = rebuild_domain()

sf = feasible_X[:, R["GGBFS (kg/m3)"]] / np.maximum(
    feasible_X[:, R["FA (kg/m3)"]] + feasible_X[:, R["GGBFS (kg/m3)"]], 1e-9)
heat_share = 100 * ((feasible_X[:, R["Initial curing temp (C)"]] >= AMBIENT_TEMP_LIMIT) &
                    (feasible_X[:, R["Initial curing time (day)"]] > 0.0)).mean()

print(f"Unique mixtures {len(opt_data)} | seed pool {n_feasible} "
      f"| heat-cured share {heat_share:.1f}%")
print(f"Curing is a DECISION VARIABLE — energy and cost apply")
print(f"Seed pool slag fraction: {sf.min():.3f} to {sf.max():.3f} (mean {sf.mean():.3f})")
print()
print(f"Binder           {BINDER_LOW:9.2f} to {BINDER_HIGH:9.2f} kg/m3")
print(f"Activator/binder {AA_BINDER_LOW:9.4f} to {AA_BINDER_HIGH:9.4f}")
print(f"Silicate / NaOH  {SS_SH_LOW:9.4f} to {SS_SH_HIGH:9.4f}")
print(f"Water / binder   {WB_LOW:9.4f} to {WB_HIGH:9.4f}")
print(f"Volume           {VOLUME_LOW:9.4f} to {VOLUME_HIGH:9.4f} m3 (candidates only)")
print(f"Aggregate vol.   {AGG_FRAC_LOW:9.4f} to {AGG_FRAC_HIGH:9.4f}")
print(f"Silicate solids  {SS_SOLIDS_LOW:9.4f} to {SS_SOLIDS_HIGH:9.4f}")
print(f"Silicate Ms      {SS_MS_LOW:9.4f} to {SS_MS_HIGH:9.4f}")
print(f"Heat curing      {HEAT_TEMP_LOW:9.1f} to {HEAT_TEMP_HIGH:9.1f} C, "
      f"{HEAT_TIME_LOW:.2f} to {HEAT_TIME_HIGH:.2f} days")
print(f"Domain guard     {GUARD_PERCENTILE:.2f} percentile, limit {DOMAIN_DISTANCE_LIMIT:.4f}")
print()
print(f"Max reachable CS_lower across the seed pool: {best_lower_seed.max():.2f} MPa")

In [ ]:
# =============================================================================
# MOO Step 4 — CO2, cost and energy objective functions
# =============================================================================

import numpy as np
import pandas as pd

# CO2, kg per kg. Alsalman et al. 2021. Sodium silicate on 48% solution basis,
# applied to the solution mass column, verified against the dataset CO2 column.
EF_CO2 = {"FA": 0.004, "GGBFS": 0.052, "Aggregates": 0.0048,
          "SP": 1.880, "NaOH_dry": 1.915, "SS_solution": 0.360}

# Embodied energy, MJ per kg. Alsalman et al. 2021.
EF_ENERGY = {"FA": 0.033, "GGBFS": 0.857, "Aggregates": 0.083,
             "SP": 29.1, "NaOH_dry": 20.5, "SS_solution": 5.371}

# Unit price, USD per kg. Fly ash and GGBFS are US market prices for 2025, where
# fly ash is scarce and costs roughly 0.12 USD/kg against 0.05 for slag. The
# remaining figures follow Katlav and Turk 2026, Table 2.
EF_COST = {"FA": 0.12, "GGBFS": 0.05, "Coarse": 0.014, "Fine": 0.01,
           "NaOH_dry": 0.38, "SS_dry": 0.17, "Water": 0.001, "SP": 1.00}

# Price of the curing energy, USD per MJ. The curing model gives the energy
# delivered to the concrete, so the energy drawn from the grid is higher. At
# roughly 30% chamber efficiency the EIA industrial rate of 0.024 USD/MJ becomes
# about 0.08, so 0.10 is adopted here. Only the curing term is priced, since
# material embodied energy is already reflected in the material unit prices.
CURING_ENERGY_PRICE = 0.10

CURING_CO2_SLOPE, CURING_CO2_INTERCEPT = 0.6417, 16.0417
CURING_ENERGY_PER_C_PER_DAY, HEATING_SOLUTION_ENERGY = 2.440, 5.8


def objectives_batch(Xr, sp_price=None, curing_price=None, fa_price=None, slag_price=None):
    """Return CO2 (kg/m3), cost (USD/m3) and embodied energy (MJ/m3).
    Prices are overridable so scenarios can be swept without redefining this."""
    sp_price = EF_COST["SP"] if sp_price is None else sp_price
    curing_price = CURING_ENERGY_PRICE if curing_price is None else curing_price
    fa_price = EF_COST["FA"] if fa_price is None else fa_price
    slag_price = EF_COST["GGBFS"] if slag_price is None else slag_price

    fa, slag = Xr[:, R["FA (kg/m3)"]], Xr[:, R["GGBFS (kg/m3)"]]
    coarse = Xr[:, R["Coarse aggregate (kg/m3)"]]
    fine = Xr[:, R["Fine aggregate (kg in 1m3 mix)"]]
    ss = Xr[:, R["Total Na2SiO3 (kg in 1m3 of mix)"]]
    sio2_dry, na2o_dry = Xr[:, R["SiO2 (Dry)"]], Xr[:, R["Na2O (Dry)"]]
    sh = Xr[:, R["NaOH (Dry)"]]
    sp = Xr[:, R["Superplasticizer (kg in 1m3 mix)"]]
    water = Xr[:, R["Total water (in solutions + additional) (kg in 1m3 mix)"]]
    temp, time = Xr[:, R["Initial curing temp (C)"]], Xr[:, R["Initial curing time (day)"]]

    heat = (temp >= AMBIENT_TEMP_LIMIT) & (time > 0.0)

    curing_co2 = np.where(heat, (CURING_CO2_SLOPE * temp - CURING_CO2_INTERCEPT) * time, 0.0)
    curing_energy = np.where(heat, CURING_ENERGY_PER_C_PER_DAY * temp * time
                             + HEATING_SOLUTION_ENERGY, 0.0)

    co2 = (EF_CO2["FA"] * fa + EF_CO2["GGBFS"] * slag + EF_CO2["Aggregates"] * (coarse + fine)
           + EF_CO2["SS_solution"] * ss + EF_CO2["NaOH_dry"] * sh + EF_CO2["SP"] * sp
           + curing_co2)

    energy = (EF_ENERGY["FA"] * fa + EF_ENERGY["GGBFS"] * slag
              + EF_ENERGY["Aggregates"] * (coarse + fine)
              + EF_ENERGY["SS_solution"] * ss + EF_ENERGY["NaOH_dry"] * sh + EF_ENERGY["SP"] * sp
              + curing_energy)

    cost = (fa_price * fa + slag_price * slag + EF_COST["Coarse"] * coarse
            + EF_COST["Fine"] * fine + EF_COST["SS_dry"] * (sio2_dry + na2o_dry)
            + EF_COST["NaOH_dry"] * sh + sp_price * sp + EF_COST["Water"] * water
            + curing_price * curing_energy)

    return co2, cost, energy


# ---- validation against the dataset CO2 column -------------------------------
co2_all, cost_all, energy_all = objectives_batch(df[RAW_VARS].values.astype(float))
co2_reported = pd.to_numeric(
    df["CO2 footprint (kg emision per 1m3 of samples produced and cured)"], errors="coerce")

m = co2_reported.notna()
r = np.corrcoef(co2_all[m], co2_reported[m])[0, 1]
mae = float(np.mean(np.abs(co2_all[m] - co2_reported[m])))

print(f"CO2 reconstruction vs dataset column: r = {r:.6f} | MAE = {mae:.4f} kg/m3")
assert r > 0.999, "CO2 reconstruction does not match the dataset column."

co2_f, cost_f, energy_f = objectives_batch(feasible_X)
summary = pd.DataFrame({"CO2_kg_m3": co2_f, "Cost_USD_m3": cost_f, "Energy_MJ_m3": energy_f})

heat_f = ((feasible_X[:, R["Initial curing temp (C)"]] >= AMBIENT_TEMP_LIMIT) &
          (feasible_X[:, R["Initial curing time (day)"]] > 0.0))
ce = np.where(heat_f, CURING_ENERGY_PER_C_PER_DAY * feasible_X[:, R["Initial curing temp (C)"]]
              * feasible_X[:, R["Initial curing time (day)"]] + HEATING_SOLUTION_ENERGY, 0.0)
share = CURING_ENERGY_PRICE * ce / np.maximum(cost_f, 1e-9)

print(f"\nFly ash {EF_COST['FA']:.3f} vs GGBFS {EF_COST['GGBFS']:.3f} USD/kg "
      f"(ratio {EF_COST['FA'] / EF_COST['GGBFS']:.2f}) | curing {CURING_ENERGY_PRICE:.3f} USD/MJ")
print(f"Curing share of total cost among heat-cured mixtures: "
      f"mean {100 * share[heat_f].mean():.1f}% | max {100 * share[heat_f].max():.1f}%")
print("\nObjectives across the feasible domain")
display(summary.describe().loc[["min", "mean", "max"]].round(2))
print("\nPearson correlation")
display(summary.corr().round(4))

In [ ]:
# =============================================================================
# MOO Step 5 — NSGA-II, maximize CS lower bound, minimize CO2 and cost
# =============================================================================

!pip install -q pymoo

import numpy as np
import pandas as pd
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.optimize import minimize
from pymoo.indicators.hv import HV

POP_SIZE = 600
N_GEN = 1500
N_SEEDS = 8
PERTURB = 0.02
CONVERGENCE_TOL_PCT = 3.0

# worse than any attainable value on each axis. The cost axis is raised above the
# Scenario 1 value because heat curing extends the reachable cost range to about
# 174 USD/m3, so hypervolume here is a within-scenario convergence diagnostic and
# is not compared across scenarios.
REF_POINT = np.array([0.0, 320.0, 200.0])

N_RAW = len(RAW_VARS)
LOW_FULL = np.concatenate([LOW, [0.0, 0.0]])
HIGH_FULL = np.concatenate([HIGH, [len(FA_ARRAY) - 1e-6, len(SLAG_ARRAY) - 1e-6]])


def split_decision(Z):
    """Separate the raw mixture part from the two floored source indices."""
    Xr = Z[:, :N_RAW]
    fa_idx = np.clip(np.floor(Z[:, N_RAW]).astype(int), 0, len(FA_ARRAY) - 1)
    slag_idx = np.clip(np.floor(Z[:, N_RAW + 1]).astype(int), 0, len(SLAG_ARRAY) - 1)
    return Xr, fa_idx, slag_idx


class MixtureProblem(Problem):
    def __init__(self):
        super().__init__(n_var=N_RAW + 2, n_obj=3, n_ieq_constr=1,
                         xl=LOW_FULL, xu=HIGH_FULL)

    def _evaluate(self, Z, out, *args, **kwargs):
        Xr, fa_idx, slag_idx = split_decision(Z)

        # ambient is a single fixed state, so the same snapped array must feed the
        # constraints, the objectives and the feature map
        Xr = snap_curing(Xr)

        co2, cost, _ = objectives_batch(Xr)
        _, lower = predict_cs_bounds(build_features_batch(Xr, fa_idx, slag_idx))

        # strength is maximized, so it enters negated
        out["F"] = np.column_stack([-lower, co2, cost])
        out["G"] = violation_batch(Xr).reshape(-1, 1)


def non_dominated(F, Z):
    keep = np.ones(len(F), dtype=bool)
    for i in range(len(F)):
        if keep[i] and (np.all(F <= F[i], axis=1) & np.any(F < F[i], axis=1)).any():
            keep[i] = False
    F_nd, Z_nd = F[keep], Z[keep]
    _, uniq = np.unique(F_nd.round(4), axis=0, return_index=True)
    return F_nd[uniq], Z_nd[uniq]


hv_indicator = HV(ref_point=REF_POINT)
seed_rows, F_all, Z_all = [], [], []

for s in range(N_SEEDS):
    rng = np.random.default_rng(SEED + s)
    Z0 = SEED_POOL[rng.integers(0, len(SEED_POOL), POP_SIZE)].copy()
    Z0 += rng.normal(0.0, PERTURB, Z0.shape) * (HIGH_FULL - LOW_FULL)[None, :]
    Z0 = np.clip(Z0, LOW_FULL, HIGH_FULL)

    res = minimize(MixtureProblem(),
                   NSGA2(pop_size=POP_SIZE, sampling=Z0,
                         crossover=SBX(prob=0.9, eta=15), mutation=PM(eta=20),
                         eliminate_duplicates=True),
                   ("n_gen", N_GEN), seed=SEED + s, verbose=False)

    if res.F is None or len(res.F) == 0:
        seed_rows.append({"seed": s, "n": 0})
        continue

    F, Z = np.atleast_2d(res.F), np.atleast_2d(res.X)
    F_all.append(F)
    Z_all.append(Z)

    seed_rows.append({"seed": s, "n": len(F),
                      "CS_max": -F[:, 0].min(), "CO2_min": F[:, 1].min(),
                      "Cost_min": F[:, 2].min(), "hypervolume": hv_indicator(F)})

    print(f"seed {s} -> {len(F)} solutions | CS_lower max {-F[:, 0].min():.2f} "
          f"| CO2 min {F[:, 1].min():.2f} | cost min {F[:, 2].min():.2f}")

assert F_all, "No feasible solutions in any run."

F_nd, Z_nd = non_dominated(np.vstack(F_all), np.vstack(Z_all))
order = np.argsort(-F_nd[:, 0])
F_nd, Z_nd = F_nd[order], Z_nd[order]

# pymoo returns the unsnapped decision vector, so snap again before anything
# downstream reads the curing columns
Xr_nd, fa_nd, slag_nd = split_decision(Z_nd)
Xr_nd = snap_curing(Xr_nd)

point_nd, lower_nd = predict_cs_bounds(build_features_batch(Xr_nd, fa_nd, slag_nd))
co2_nd, cost_nd, energy_nd = objectives_batch(Xr_nd)

results = {"F": F_nd, "Z": Z_nd, "Xr": Xr_nd, "fa_idx": fa_nd, "slag_idx": slag_nd,
           "CS_point": point_nd, "CS_lower": lower_nd,
           "CO2": co2_nd, "Cost": cost_nd, "Energy": energy_nd}
results_full = {k: (v.copy() if hasattr(v, "copy") else v) for k, v in results.items()}

seed_df = pd.DataFrame(seed_rows)
hv_cv = 100 * seed_df["hypervolume"].std() / seed_df["hypervolume"].mean()

heat_nd = ((Xr_nd[:, R["Initial curing temp (C)"]] >= AMBIENT_TEMP_LIMIT) &
           (Xr_nd[:, R["Initial curing time (day)"]] > 0.0))

print("\nPer-run fronts")
display(seed_df.round(3))
print(f"\nHypervolume CV across {N_SEEDS} seeds: {hv_cv:.3f}%  ->  "
      f"{'converged' if hv_cv < CONVERGENCE_TOL_PCT else 'NOT converged'}")

# a single lagging seed inflates the CV, so report the spread as well
hv = seed_df["hypervolume"]
print(f"  hypervolume min {hv.min():.0f} | median {hv.median():.0f} | max {hv.max():.0f}")
print(f"  worst seed is {100 * (1 - hv.min() / hv.max()):.2f}% below the best")

print(f"\nMerged front: {len(F_nd)} non-dominated solutions")
print(f"  CS lower bound  {lower_nd.min():7.2f} to {lower_nd.max():7.2f} MPa")
print(f"  CS point        {point_nd.min():7.2f} to {point_nd.max():7.2f} MPa")
print(f"  CO2             {co2_nd.min():7.2f} to {co2_nd.max():7.2f} kg/m3")
print(f"  Cost            {cost_nd.min():7.2f} to {cost_nd.max():7.2f} USD/m3")
print(f"  Energy          {energy_nd.min():7.2f} to {energy_nd.max():7.2f} MJ/m3")
print(f"  heat cured      {int(heat_nd.sum())} of {len(heat_nd)} ({100 * heat_nd.mean():.1f}%)")

In [ ]:
# =============================================================================
# MOO Step 6 — Front diagnostics, extrapolation and margin behaviour
# =============================================================================

import numpy as np
import pandas as pd

Xr = results["Xr"]
lower, point = results["CS_lower"], results["CS_point"]
margin = point - lower

d, _ = nn_domain.kneighbors(domain_scaler.transform(pd.DataFrame(Xr, columns=RAW_VARS)),
                            n_neighbors=1, return_distance=True)
nn_dist = d[:, 0]

obs_cs = pd.to_numeric(df["converted CS_Mpa"], errors="coerce")
slag_frac = Xr[:, R["GGBFS (kg/m3)"]] / np.maximum(
    Xr[:, R["FA (kg/m3)"]] + Xr[:, R["GGBFS (kg/m3)"]], 1e-9)
temp = Xr[:, R["Initial curing temp (C)"]]
cure_time = Xr[:, R["Initial curing time (day)"]]
heat = (temp >= AMBIENT_TEMP_LIMIT) & (cure_time > 0.0)

print(f"Observed CS maximum: {obs_cs.max():.2f} MPa (99th pct {obs_cs.quantile(0.99):.2f})")
print(f"Front solutions above the observed max (point): {int((point > obs_cs.max()).sum())}")
print(f"Front solutions above it (lower bound): {int((lower > obs_cs.max()).sum())}")
print(f"Domain guard limit: {DOMAIN_DISTANCE_LIMIT:.4f}")
print(f"Margin correlation with strength: {np.corrcoef(lower, margin)[0, 1]:.4f}")
print(f"Heat-cured share of the front: {100 * heat.mean():.1f}%")
print()

# temp and time are blanked on ambient solutions so the band means describe the
# heat cycle actually chosen rather than being diluted by the fixed 25 C
diag = pd.DataFrame({"CS_lower": lower, "margin": margin, "nn_dist": nn_dist,
                     "CO2": results["CO2"], "Cost": results["Cost"],
                     "Energy": results["Energy"], "slag_frac": slag_frac,
                     "heat": heat.astype(float),
                     "heat_temp": np.where(heat, temp, np.nan),
                     "heat_time": np.where(heat, cure_time, np.nan)})

BAND_EDGES = {"<20": (0, 20), "20-30": (20, 30), "30-40": (30, 40), "40-50": (40, 50),
              "50-60": (50, 60), "60-70": (60, 70), ">70": (70, np.inf)}

bands = pd.cut(diag["CS_lower"], bins=[0, 20, 30, 40, 50, 60, 70, np.inf],
               labels=list(BAND_EDGES))

summary = diag.groupby(bands, observed=True).agg(
    n=("CS_lower", "size"), margin_mean=("margin", "mean"),
    nn_dist_mean=("nn_dist", "mean"), CO2_min=("CO2", "min"),
    Cost_min=("Cost", "min"), slag_frac_mean=("slag_frac", "mean"),
    slag_frac_med=("slag_frac", "median"), heat_share=("heat", "mean"),
    heat_temp_mean=("heat_temp", "mean"), heat_time_mean=("heat_time", "mean"))

summary["heat_share"] *= 100
summary["observed_rows"] = [int(((obs_cs >= BAND_EDGES[b][0]) & (obs_cs < BAND_EDGES[b][1])).sum())
                            for b in summary.index.astype(str)]

print("Front behaviour by strength band, heat share in percent")
display(summary.round(4))

print(f"\nSlag fraction: {slag_frac.min():.3f} to {slag_frac.max():.3f} (mean {slag_frac.mean():.3f})")
print(f"corr(slag fraction, CS_lower) = {np.corrcoef(slag_frac, lower)[0, 1]:.4f}")
print(f"corr(heat flag, CS_lower)     = {np.corrcoef(heat.astype(float), lower)[0, 1]:.4f}")

In [ ]:
# =============================================================================
# MOO Step 6b — Cap at the limit of data support and screen uncertain solutions
# =============================================================================

import numpy as np
import pandas as pd

obs_cs = pd.to_numeric(df["converted CS_Mpa"], errors="coerce")
CS_CAP = float(obs_cs.quantile(0.99))
MARGIN_MAX = 8.0

margin_full = results_full["CS_point"] - results_full["CS_lower"]
keep = (results_full["CS_lower"] <= CS_CAP) & (margin_full <= MARGIN_MAX)

for k in ["F", "Z", "Xr", "fa_idx", "slag_idx", "CS_point", "CS_lower", "CO2", "Cost", "Energy"]:
    results[k] = results_full[k][keep]

lower, point = results["CS_lower"], results["CS_point"]
margin = point - lower
slag_frac = results["Xr"][:, R["GGBFS (kg/m3)"]] / np.maximum(
    results["Xr"][:, R["FA (kg/m3)"]] + results["Xr"][:, R["GGBFS (kg/m3)"]], 1e-9)
temp = results["Xr"][:, R["Initial curing temp (C)"]]
cure_time = results["Xr"][:, R["Initial curing time (day)"]]
heat = (temp >= AMBIENT_TEMP_LIMIT) & (cure_time > 0.0)

print(f"Cap at the 99th percentile of observed CS = {CS_CAP:.2f} MPa")
print(f"Margin screen at {MARGIN_MAX:.1f} MPa (held-out mean 5.08)")
print(f"  removed {int((~keep).sum())} of {len(keep)} | retained {int(keep.sum())}")
print()
print(f"  CS lower bound  {lower.min():7.2f} to {lower.max():7.2f} MPa")
print(f"  CS point        {point.min():7.2f} to {point.max():7.2f} MPa")
print(f"  margin          {margin.min():7.2f} to {margin.max():7.2f} (mean {margin.mean():.2f})")
print(f"  CO2             {results['CO2'].min():7.2f} to {results['CO2'].max():7.2f} kg/m3")
print(f"  Cost            {results['Cost'].min():7.2f} to {results['Cost'].max():7.2f} USD/m3")
print(f"  Energy          {results['Energy'].min():7.2f} to {results['Energy'].max():7.2f} MJ/m3")
print(f"  slag fraction   {slag_frac.min():7.3f} to {slag_frac.max():7.3f} (mean {slag_frac.mean():.3f})")
print(f"  margin correlation with strength: {np.corrcoef(lower, margin)[0, 1]:.4f}")

print(f"\nCuring regime on the retained front")
print(f"  ambient  {int((~heat).sum()):5d} ({100 * (~heat).mean():.1f}%)")
print(f"  heat     {int(heat.sum()):5d} ({100 * heat.mean():.1f}%)")
if heat.any():
    print(f"  heat cycle among heated solutions: {temp[heat].min():.1f} to {temp[heat].max():.1f} C, "
          f"{cure_time[heat].min():.2f} to {cure_time[heat].max():.2f} days "
          f"(mean {temp[heat].mean():.1f} C for {cure_time[heat].mean():.2f} days)")

print("\nPreview of the three table strengths")
for t in [30, 50, 70]:
    if lower.max() < t:
        print(f"  {t} MPa -> not reachable")
        continue
    near = np.abs(lower - t) <= 1.0
    k = int(np.flatnonzero(near)[np.argmin(results["CO2"][near])]) if near.any() \
        else int(np.abs(lower - t).argmin())
    regime = "heat" if heat[k] else "ambient"
    print(f"  nearest to {t} MPa -> CS_lower {lower[k]:6.2f} | margin {margin[k]:5.2f} "
          f"| CO2 {results['CO2'][k]:7.2f} | cost {results['Cost'][k]:6.2f} "
          f"| slag {slag_frac[k]:.3f} | {regime} {temp[k]:5.1f} C for {cure_time[k]:.2f} d")

In [ ]:
# =============================================================================
# MOO Step 7 — Reliability flagging (no deletion) and TOPSIS selection
# =============================================================================

import numpy as np
import pandas as pd

NN_DISTANCE_MAX = DOMAIN_DISTANCE_LIMIT
BOUND_TOL = 0.02
MAX_NEAR_BOUNDS = 6

BOUNDARY_EXEMPT = {"GGBFS (kg/m3)", "Superplasticizer (kg in 1m3 mix)",
                   "NaOH (Dry)", "Total Na2SiO3 (kg in 1m3 of mix)",
                   "Initial curing time (day)", "Initial curing temp (C)"}

# identical to Scenario 1 so the two recommendations stay comparable. Curing
# energy also enters cost, so heated solutions carry it twice at a small weight.
TOPSIS_WEIGHTS = {"CS": 0.40, "CO2": 0.25, "Cost": 0.20, "Energy": 0.15}
_w = sum(TOPSIS_WEIGHTS.values())
TOPSIS_WEIGHTS = {k: v / _w for k, v in TOPSIS_WEIGHTS.items()}

short_var = {
    "FA (kg/m3)": "FA", "GGBFS (kg/m3)": "GGBFS",
    "Coarse aggregate (kg/m3)": "Coarse agg", "Fine aggregate (kg in 1m3 mix)": "Fine agg",
    "Total Na2SiO3 (kg in 1m3 of mix)": "Na2SiO3", "SiO2 (Dry)": "SiO2 dry",
    "Na2O (Dry)": "Na2O dry", "NaOH (Dry)": "NaOH dry",
    "Concentration (M) NaOH": "NaOH molarity", "Superplasticizer (kg in 1m3 mix)": "SP",
    "Total water (in solutions + additional) (kg in 1m3 mix)": "Water",
    "Initial curing time (day)": "Curing time", "Initial curing temp (C)": "Curing temp",
}


def topsis_closeness(matrix, directions, weights):
    norm = np.sqrt((matrix ** 2).sum(axis=0))
    norm[norm == 0.0] = 1e-12
    V = (matrix / norm) * weights

    ideal_best = np.where(directions > 0, V.max(axis=0), V.min(axis=0))
    ideal_worst = np.where(directions > 0, V.min(axis=0), V.max(axis=0))

    d_best = np.sqrt(((V - ideal_best) ** 2).sum(axis=1))
    d_worst = np.sqrt(((V - ideal_worst) ** 2).sum(axis=1))

    return d_worst / np.maximum(d_best + d_worst, 1e-12)


Xr = results["Xr"]

pareto_flagged_df = pd.DataFrame(Xr, columns=RAW_VARS)
pareto_flagged_df["CS_point_MPa"] = results["CS_point"]
pareto_flagged_df["CS_lower_MPa"] = results["CS_lower"]
pareto_flagged_df["Margin_MPa"] = results["CS_point"] - results["CS_lower"]
pareto_flagged_df["CO2_kg_m3"] = results["CO2"]
pareto_flagged_df["Cost_USD_m3"] = results["Cost"]
pareto_flagged_df["Energy_MJ_m3"] = results["Energy"]
pareto_flagged_df["FA_source"] = results["fa_idx"]
pareto_flagged_df["Slag_source"] = results["slag_idx"]
pareto_flagged_df["Slag_fraction"] = Xr[:, R["GGBFS (kg/m3)"]] / np.maximum(
    Xr[:, R["FA (kg/m3)"]] + Xr[:, R["GGBFS (kg/m3)"]], 1e-9)

pareto_flagged_df["Curing_regime"] = np.where(
    (Xr[:, R["Initial curing temp (C)"]] >= AMBIENT_TEMP_LIMIT) &
    (Xr[:, R["Initial curing time (day)"]] > 0.0), "Heat", "Ambient")

pareto_flagged_df["CO2_per_MPa"] = pareto_flagged_df["CO2_kg_m3"] / pareto_flagged_df["CS_lower_MPa"]
pareto_flagged_df["Cost_per_MPa"] = pareto_flagged_df["Cost_USD_m3"] / pareto_flagged_df["CS_lower_MPa"]
pareto_flagged_df["Energy_per_MPa"] = pareto_flagged_df["Energy_MJ_m3"] / pareto_flagged_df["CS_lower_MPa"]

d, nn_idx = nn_domain.kneighbors(domain_scaler.transform(pd.DataFrame(Xr, columns=RAW_VARS)),
                                 n_neighbors=1, return_distance=True)
pareto_flagged_df["NN_distance"] = d[:, 0]
pareto_flagged_df["Nearest_real_mix_index"] = nn_idx[:, 0]

near = np.zeros(len(pareto_flagged_df), dtype=int)
for j, v in enumerate(RAW_VARS):
    if v in BOUNDARY_EXEMPT:
        continue
    span = max(HIGH[j] - LOW[j], 1e-9)
    near += ((Xr[:, j] <= LOW[j] + BOUND_TOL * span) |
             (Xr[:, j] >= HIGH[j] - BOUND_TOL * span)).astype(int)

pareto_flagged_df["n_near_bounds_scored"] = near
pareto_flagged_df["pass_local_domain"] = pareto_flagged_df["NN_distance"] <= NN_DISTANCE_MAX + 1e-9
pareto_flagged_df["pass_boundary_pressure"] = near <= MAX_NEAR_BOUNDS
pareto_flagged_df["conservative_pass"] = (pareto_flagged_df["pass_local_domain"] &
                                          pareto_flagged_df["pass_boundary_pressure"])

matrix = pareto_flagged_df[["CS_lower_MPa", "CO2_kg_m3", "Cost_USD_m3", "Energy_MJ_m3"]].to_numpy(float)
directions = np.array([1.0, -1.0, -1.0, -1.0])
weights = np.array([TOPSIS_WEIGHTS["CS"], TOPSIS_WEIGHTS["CO2"],
                    TOPSIS_WEIGHTS["Cost"], TOPSIS_WEIGHTS["Energy"]])
pareto_flagged_df["TOPSIS_closeness"] = topsis_closeness(matrix, directions, weights)

print(f"TOPSIS weights: CS {TOPSIS_WEIGHTS['CS']:.2f} | CO2 {TOPSIS_WEIGHTS['CO2']:.2f} "
      f"| cost {TOPSIS_WEIGHTS['Cost']:.2f} | energy {TOPSIS_WEIGHTS['Energy']:.2f}")
print(f"Reliability thresholds: NN distance <= {NN_DISTANCE_MAX:.4f}, at most {MAX_NEAR_BOUNDS} "
      f"scored variables within {int(BOUND_TOL * 100)}% of a bound")
print(f"Exempt from the bound count: {', '.join(sorted(short_var[v] for v in BOUNDARY_EXEMPT))}")
print()

display(pd.DataFrame([{
    "n_solutions": len(pareto_flagged_df),
    "nn_dist_max": pareto_flagged_df["NN_distance"].max(),
    "near_bounds_median": int(np.median(near)),
    "near_bounds_max": int(near.max()),
    "pass_local_domain": int(pareto_flagged_df["pass_local_domain"].sum()),
    "pass_boundary_pressure": int(pareto_flagged_df["pass_boundary_pressure"].sum()),
    "conservative_pass": int(pareto_flagged_df["conservative_pass"].sum()),
}]).set_index("n_solutions").round(4))

print("\nFront composition by curing regime")
regime_summary = pareto_flagged_df.groupby("Curing_regime").agg(
    n=("CS_lower_MPa", "size"), CS_lower_mean=("CS_lower_MPa", "mean"),
    CS_lower_max=("CS_lower_MPa", "max"), CO2_mean=("CO2_kg_m3", "mean"),
    Cost_mean=("Cost_USD_m3", "mean"), Energy_mean=("Energy_MJ_m3", "mean"),
    slag_frac_mean=("Slag_fraction", "mean"), CO2_per_MPa_mean=("CO2_per_MPa", "mean"),
    Cost_per_MPa_mean=("Cost_per_MPa", "mean"))
display(regime_summary.round(3))

reliable = pareto_flagged_df[pareto_flagged_df["conservative_pass"]]
pick_from = reliable if len(reliable) > 0 else pareto_flagged_df
recommended_idx = int(pick_from["TOPSIS_closeness"].idxmax())
rec = pareto_flagged_df.loc[recommended_idx]

print(f"\nTOPSIS recommended solution (index {recommended_idx})")
print(f"  CS lower {rec['CS_lower_MPa']:.2f} | CS point {rec['CS_point_MPa']:.2f} "
      f"| margin {rec['Margin_MPa']:.2f} MPa")
print(f"  CO2 {rec['CO2_kg_m3']:.2f} kg/m3 | cost {rec['Cost_USD_m3']:.2f} USD/m3 "
      f"| energy {rec['Energy_MJ_m3']:.2f} MJ/m3")
print(f"  FA {rec['FA (kg/m3)']:.1f} | GGBFS {rec['GGBFS (kg/m3)']:.1f} kg/m3 "
      f"| slag/binder {rec['Slag_fraction']:.3f}")
print(f"  regime {rec['Curing_regime']} at {rec['Initial curing temp (C)']:.1f} C "
      f"for {rec['Initial curing time (day)']:.2f} days")
print(f"  closeness {rec['TOPSIS_closeness']:.4f} | reliable {bool(rec['conservative_pass'])}")

front_heat = pareto_flagged_df.copy()
rec_idx_heat = recommended_idx
print(f"Snapshot saved: front_heat ({len(front_heat)} rows), rec_idx_heat = {rec_idx_heat}")

In [ ]:
# =============================================================================
# MOO Step 8 — Fly ash to slag price ratio sensitivity
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.crossover.sbx import SBX
from pymoo.operators.mutation.pm import PM
from pymoo.optimize import minimize
from google.colab import files

# slag is held at its US market price and the fly ash price is varied, so the
# ratio spans Katlav's inverted case (0.26) through to the US case (2.40)
SLAG_PRICE_FIXED = 0.05
PRICE_RATIOS = [0.26, 0.50, 0.75, 1.00, 1.50, 2.40, 4.00]

N_SEEDS_SENS = 3
N_GEN_SENS = 800

BASE_FA, BASE_SLAG = EF_COST["FA"], EF_COST["GGBFS"]
rows = []

try:
    for ratio in PRICE_RATIOS:
        EF_COST["GGBFS"] = SLAG_PRICE_FIXED
        EF_COST["FA"] = ratio * SLAG_PRICE_FIXED

        F_s, Z_s = [], []
        for s in range(N_SEEDS_SENS):
            rng = np.random.default_rng(SEED + s)
            Z0 = SEED_POOL[rng.integers(0, len(SEED_POOL), POP_SIZE)].copy()
            Z0 += rng.normal(0.0, PERTURB, Z0.shape) * (HIGH_FULL - LOW_FULL)[None, :]
            Z0 = np.clip(Z0, LOW_FULL, HIGH_FULL)

            res = minimize(MixtureProblem(),
                           NSGA2(pop_size=POP_SIZE, sampling=Z0,
                                 crossover=SBX(prob=0.9, eta=15), mutation=PM(eta=20),
                                 eliminate_duplicates=True),
                           ("n_gen", N_GEN_SENS), seed=SEED + s, verbose=False)

            if res.F is not None and len(res.F) > 0:
                F_s.append(np.atleast_2d(res.F))
                Z_s.append(np.atleast_2d(res.X))

        if not F_s:
            print(f"ratio {ratio:.2f} -> no feasible solutions")
            continue

        Fn, Zn = non_dominated(np.vstack(F_s), np.vstack(Z_s))

        # pymoo returns the unsnapped decision vector, so snap before recomputing
        # anything, exactly as in Step 5
        Xr_s, fa_s, sl_s = split_decision(Zn)
        Xr_s = snap_curing(Xr_s)

        pt, lo = predict_cs_bounds(build_features_batch(Xr_s, fa_s, sl_s))
        co2_s, cost_s, en_s = objectives_batch(Xr_s)

        keep = (lo <= CS_CAP) & ((pt - lo) <= MARGIN_MAX)
        Xr_s, lo, pt = Xr_s[keep], lo[keep], pt[keep]
        co2_s, cost_s, en_s = co2_s[keep], cost_s[keep], en_s[keep]

        slag = Xr_s[:, R["GGBFS (kg/m3)"]]
        fa = Xr_s[:, R["FA (kg/m3)"]]
        frac = slag / np.maximum(fa + slag, 1e-9)
        temp = Xr_s[:, R["Initial curing temp (C)"]]
        cure_time = Xr_s[:, R["Initial curing time (day)"]]
        heat_s = (temp >= AMBIENT_TEMP_LIMIT) & (cure_time > 0.0)

        mat = np.column_stack([lo, co2_s, cost_s, en_s])
        close = topsis_closeness(mat, np.array([1.0, -1.0, -1.0, -1.0]),
                                 np.array([0.40, 0.25, 0.20, 0.15]))
        k = int(np.argmax(close))

        rows.append({"FA_slag_ratio": ratio, "FA_price": EF_COST["FA"], "n": len(lo),
                     "CS_max": lo.max(), "CO2_min": co2_s.min(), "Cost_min": cost_s.min(),
                     "slag_frac_mean": frac.mean(), "slag_frac_median": np.median(frac),
                     "pct_slag_gt25": 100 * (frac > 0.25).mean(),
                     "temp_mean": temp.mean(),
                     "pct_heat": 100 * heat_s.mean(),
                     "corr_slag_CS": np.corrcoef(frac, lo)[0, 1],
                     "rec_CS": lo[k], "rec_slag_frac": frac[k], "rec_FA": fa[k],
                     "rec_GGBFS": slag[k], "rec_temp": temp[k],
                     "rec_regime": "Heat" if heat_s[k] else "Ambient",
                     "rec_CO2": co2_s[k], "rec_cost": cost_s[k]})

        print(f"ratio {ratio:4.2f} (FA {EF_COST['FA']:.3f}) -> n {len(lo):4d} "
              f"| slag_frac mean {frac.mean():.3f} | >25% slag {100 * (frac > 0.25).mean():5.1f}% "
              f"| heat {100 * heat_s.mean():5.1f}% | rec slag {frac[k]:.3f} at CS {lo[k]:.1f}")

finally:
    EF_COST["FA"], EF_COST["GGBFS"] = BASE_FA, BASE_SLAG

price_sensitivity = pd.DataFrame(rows)

print(f"\nPrices restored to FA {EF_COST['FA']:.3f} / GGBFS {EF_COST['GGBFS']:.3f} USD/kg")
display(price_sensitivity.round(4))

fig, (axL, axR) = plt.subplots(1, 2, figsize=(15.5, 6.2))

axL.plot(price_sensitivity["FA_slag_ratio"], price_sensitivity["slag_frac_mean"],
         marker="o", markersize=9, linewidth=2.2, color="#0033CC", label="Front mean")
axL.plot(price_sensitivity["FA_slag_ratio"], price_sensitivity["rec_slag_frac"],
         marker="s", markersize=9, linewidth=2.2, color="#D00000", label="TOPSIS recommended")
axL.axvline(2.40, color="black", ls="--", lw=1.4, alpha=0.7)
axL.text(2.40, axL.get_ylim()[1] * 0.96, " US 2025", fontsize=12, va="top")
axL.axvline(0.26, color="gray", ls=":", lw=1.4, alpha=0.7)
axL.text(0.26, axL.get_ylim()[1] * 0.96, " Katlav", fontsize=12, va="top")

axL.set_xlabel("Fly ash to slag price ratio", fontsize=16, labelpad=9)
axL.set_ylabel("Slag fraction of binder", fontsize=16, labelpad=9)
axL.set_title("Precursor choice against relative price", fontsize=17, fontweight="bold", pad=10)
axL.tick_params(axis="both", labelsize=13)
axL.grid(linestyle="--", linewidth=0.55, alpha=0.22)
axL.legend(fontsize=12, frameon=True, loc="lower right")

axR.plot(price_sensitivity["FA_slag_ratio"], price_sensitivity["pct_slag_gt25"],
         marker="o", markersize=9, linewidth=2.2, color="#00802B")
axR.set_xlabel("Fly ash to slag price ratio", fontsize=16, labelpad=9)
axR.set_ylabel("Front share with slag above 25% (%)", fontsize=16, labelpad=9)
axR.set_title("Share of blended solutions", fontsize=17, fontweight="bold", pad=10)
axR.tick_params(axis="both", labelsize=13)
axR.set_ylim(0, 100)
axR.grid(linestyle="--", linewidth=0.55, alpha=0.22)

for ax in (axL, axR):
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.05)

fig.tight_layout(w_pad=2.6)

FIG_PATH = "/content/MOO_price_ratio_sensitivity_heat.png"
fig.savefig(FIG_PATH, dpi=600, facecolor="white", bbox_inches="tight")
plt.show()

files.download(FIG_PATH)

price_heat = price_sensitivity.copy()
print(f"Snapshot saved: price_heat ({len(price_heat)} rows)")

In [ ]:
for name in ["front_ambient", "rec_idx_ambient", "price_ambient",
             "front_heat", "rec_idx_heat", "price_heat"]:
    print(name, "OK" if name in globals() else "MISSING")

In [ ]:
# =============================================================================
# Figure 1 — Strength against CO2, both scenarios, colour = energy
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from google.colab import files

for _n in ["front_ambient", "rec_idx_ambient", "front_heat", "rec_idx_heat"]:
    assert _n in globals(), f"{_n} is missing. Re-run the snapshot lines."

SAVE_DIR = globals().get("SAVE_DIR", "/content")
os.makedirs(SAVE_DIR, exist_ok=True)

C_AMB, C_HEAT = "#0033CC", "#D00000"
GRADES_AMB, GRADES_HEAT = [30, 40, 50], [30, 50, 70]
GRADE_TOL = 1.0

A = front_ambient.copy()
H = front_heat.copy()

# Scenario 1 has no regime column because every solution there is ambient by
# construction, so it is added here to keep the two frames interchangeable
if "Curing_regime" not in A.columns:
    A["Curing_regime"] = "Ambient"

REC_A = A.loc[rec_idx_ambient]
REC_H = H.loc[rec_idx_heat]


def grade_row(front, target, tol=GRADE_TOL):
    """Lowest CO2 solution within tol of the target strength. Falls back to the
    nearest solution if the window is empty, and reports which rule was used."""
    d = (front["CS_lower_MPa"] - target).abs()
    near = front[d <= tol]
    if len(near) == 0:
        return int(d.idxmin()), "nearest"
    return int(near["CO2_kg_m3"].idxmin()), "min CO2"


# a common energy scale so a colour means the same thing in both panels
e_all = np.concatenate([A["Energy_MJ_m3"].values, H["Energy_MJ_m3"].values])
vmin, vmax = float(e_all.min()), float(e_all.max())

fig1, axes = plt.subplots(1, 2, figsize=(16.8, 6.8), sharex=True, sharey=True)

star_handle = Line2D([0], [0], marker="*", color="w", markerfacecolor="none",
                     markeredgecolor="k", markeredgewidth=1.8, markersize=18,
                     label="TOPSIS recommended")
circ_handle = Line2D([0], [0], marker="o", color="w", markerfacecolor="grey",
                     markeredgecolor="k", markersize=10, label="Ambient cured")
tri_handle = Line2D([0], [0], marker="^", color="w", markerfacecolor="grey",
                    markeredgecolor="k", markersize=11, label="Heat cured")

panels = [(axes[0], A, REC_A, "(a) Ambient curing only", [circ_handle, star_handle]),
          (axes[1], H, REC_H, "(b) Heat curing allowed", [circ_handle, tri_handle, star_handle])]

for ax, f, rec, title, handles in panels:
    for regime, marker in [("Ambient", "o"), ("Heat", "^")]:
        sub = f[f["Curing_regime"] == regime]
        if len(sub) == 0:
            continue
        ax.scatter(sub["CS_lower_MPa"], sub["CO2_kg_m3"], c=sub["Energy_MJ_m3"],
                   cmap="viridis", vmin=vmin, vmax=vmax, marker=marker, s=44,
                   edgecolor="k", linewidth=0.3, alpha=0.9, zorder=2)

    ax.scatter(rec["CS_lower_MPa"], rec["CO2_kg_m3"], marker="*", s=620,
               facecolor="none", edgecolor="k", linewidth=2.2, zorder=10)

    ax.set_xlabel("Conformal lower bound of compressive strength (MPa)",
                  fontsize=15, labelpad=9)
    ax.set_ylabel("CO$_2$ (kg/m$^3$)", fontsize=15, labelpad=9)
    ax.set_title(title, fontsize=16, fontweight="bold", pad=10)
    ax.tick_params(axis="both", labelsize=13, labelleft=True, labelbottom=True)
    ax.grid(linestyle="--", linewidth=0.55, alpha=0.22)
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.05)

    ax.legend(handles=handles, loc="upper left", fontsize=12.5, frameon=True)

    sm = ScalarMappable(cmap="viridis", norm=plt.Normalize(vmin=vmin, vmax=vmax))
    sm.set_array([])
    cb = fig1.colorbar(sm, ax=ax, pad=0.02, fraction=0.048)
    cb.set_label("Energy (MJ/m$^3$)", fontsize=13.5, labelpad=10)
    cb.ax.tick_params(labelsize=12)

fig1.tight_layout(w_pad=2.4)

F1 = os.path.join(SAVE_DIR, "MOO_fig1_strength_co2.png")
fig1.savefig(F1, dpi=600, facecolor="white", bbox_inches="tight", pad_inches=0.05)
print("Saved:", F1)
plt.show()
files.download(F1)

In [ ]:
# =============================================================================
# Figure 2 — Three-objective Pareto sets, one panel per scenario
# =============================================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from google.colab import files

assert "A" in globals() and "H" in globals(), "Run the Figure 1 cell first."

# closeness is computed within each scenario, so the shared scale only makes the
# two panels read the same way, it does not make the values interchangeable
t_all = np.concatenate([A["TOPSIS_closeness"].values, H["TOPSIS_closeness"].values])
tmin, tmax = float(t_all.min()), float(t_all.max())

co2_lim = (min(A["CO2_kg_m3"].min(), H["CO2_kg_m3"].min()) - 4,
           max(A["CO2_kg_m3"].max(), H["CO2_kg_m3"].max()) + 4)
cost_lim = (min(A["Cost_USD_m3"].min(), H["Cost_USD_m3"].min()) - 3,
            max(A["Cost_USD_m3"].max(), H["Cost_USD_m3"].max()) + 3)
cs_lim = (0, max(A["CS_lower_MPa"].max(), H["CS_lower_MPa"].max()) + 4)

star_h = Line2D([0], [0], marker="*", color="w", markerfacecolor="none",
                markeredgecolor="k", markeredgewidth=1.8, markersize=19,
                label="TOPSIS recommended")
circ_h = Line2D([0], [0], marker="o", color="w", markerfacecolor="grey",
                markeredgecolor="k", markersize=11, label="Ambient cured")
tri_h = Line2D([0], [0], marker="^", color="w", markerfacecolor="grey",
               markeredgecolor="k", markersize=12, label="Heat cured")

fig2 = plt.figure(figsize=(17.2, 7.4))
panels = [(A, REC_A, "(a) Ambient curing only", [circ_h, star_h], 1),
          (H, REC_H, "(b) Heat curing allowed", [circ_h, tri_h, star_h], 2)]

for f, rec, title, handles, pos in panels:
    ax = fig2.add_subplot(1, 2, pos, projection="3d")

    for regime, marker in [("Ambient", "o"), ("Heat", "^")]:
        sub = f[f["Curing_regime"] == regime]
        if len(sub) == 0:
            continue
        sc = ax.scatter(sub["CO2_kg_m3"], sub["Cost_USD_m3"], sub["CS_lower_MPa"],
                        c=sub["TOPSIS_closeness"], cmap="plasma", vmin=tmin, vmax=tmax,
                        marker=marker, s=32, edgecolor="k", linewidth=0.18,
                        alpha=0.88, depthshade=False)

    star = ax.scatter(rec["CO2_kg_m3"], rec["Cost_USD_m3"], rec["CS_lower_MPa"],
                      marker="*", s=760, facecolor="none", edgecolor="k",
                      linewidth=2.3, depthshade=False)
    star.set_zorder(20)

    ax.set_xlim(*co2_lim)
    ax.set_ylim(*cost_lim)
    ax.set_zlim(*cs_lim)

    # labels and ticks pulled in towards the axes, and the z label kept tight so
    # the colourbar can sit close beside it
    ax.set_xlabel("CO$_2$ (kg/m$^3$)", fontsize=15, labelpad=4)
    ax.set_ylabel("Cost (USD/m$^3$)", fontsize=15, labelpad=6)
    ax.set_zlabel("CS lower bound (MPa)", fontsize=15, labelpad=4)
    ax.set_title(title, fontsize=16, fontweight="bold", pad=26)
    ax.tick_params(axis="x", labelsize=13, pad=-1)
    ax.tick_params(axis="y", labelsize=13, pad=-1)
    ax.tick_params(axis="z", labelsize=13, pad=1)
    ax.view_init(elev=20, azim=-60)

    # a single horizontal row under the panel title, so the entries sit in the
    # blank band above the 3D box rather than cutting into it
    ax.legend(handles=handles, loc="upper center", fontsize=12.5, frameon=False,
              ncol=len(handles), columnspacing=1.6, handletextpad=0.4,
              borderpad=0.2, bbox_to_anchor=(0.5, 1.05))

    cb = fig2.colorbar(sc, ax=ax, pad=0.06, shrink=0.58, aspect=22)
    cb.set_label("TOPSIS closeness", fontsize=13.5, labelpad=8)
    cb.ax.tick_params(labelsize=12)

fig2.subplots_adjust(left=0.0, right=0.99, top=0.94, bottom=0.02, wspace=0.02)

F2 = os.path.join(SAVE_DIR, "MOO_fig2_3d_pareto.png")
fig2.savefig(F2, dpi=600, facecolor="white", bbox_inches="tight", pad_inches=0.04)
print("Saved:", F2)
plt.show()
files.download(F2)

In [ ]:
# =============================================================================
# Figure 3 — Parallel coordinates, one panel per scenario
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from google.colab import files

assert "A" in globals() and "H" in globals(), "Run the Figure 1 cell first."

pc_axes = ["CS_lower_MPa", "CO2_kg_m3", "Cost_USD_m3", "Energy_MJ_m3",
           "CO2_per_MPa", "Cost_per_MPa"]
pc_labels = ["CS lower\n(MPa)", "Total CO$_2$\n(kg/m$^3$)", "Total cost\n(USD/m$^3$)",
             "Total energy\n(MJ/m$^3$)", "CO$_2$/CS", "Cost/CS"]

# both panels share one min and max per axis, so a height means the same thing
# on the left and on the right
both = pd.concat([A[pc_axes], H[pc_axes]], axis=0).to_numpy(float)
mins, maxs = both.min(axis=0), both.max(axis=0)
spans = np.where((maxs - mins) == 0, 1e-12, maxs - mins)

xpos = np.arange(len(pc_axes))
fig3, axes = plt.subplots(1, 2, figsize=(18.0, 6.8), sharey=True)

panels = [(axes[0], A, REC_A, "(a) Ambient curing only"),
          (axes[1], H, REC_H, "(b) Heat curing allowed")]

for ax, f, rec, title in panels:
    # inside panel (b) the lines are split by regime, which shows where the
    # heated solutions sit on every axis at once
    for regime, colour in [("Ambient", C_AMB), ("Heat", C_HEAT)]:
        sub = f[f["Curing_regime"] == regime]
        if len(sub) == 0:
            continue
        Mn = (sub[pc_axes].to_numpy(float) - mins) / spans
        for i in range(len(Mn)):
            ax.plot(xpos, Mn[i], color=colour, alpha=0.09, linewidth=0.7, zorder=2)

    rn = (rec[pc_axes].to_numpy(float) - mins) / spans
    ax.plot(xpos, rn, color="black", linewidth=3.2, zorder=7)
    ax.plot(xpos, rn, color="white", linewidth=1.3, zorder=8)

    for x in xpos:
        ax.axvline(x, color="grey", alpha=0.4, linewidth=0.8, zorder=1)

    ax.set_xlim(xpos[0], xpos[-1])
    ax.set_ylim(0.0, 1.0)
    ax.margins(x=0, y=0)
    ax.set_xticks(xpos)
    ax.set_xticklabels(pc_labels, fontsize=13)
    ax.set_yticks([0, 0.5, 1.0])
    ax.set_yticklabels(["min", "mid", "max"], fontsize=13)
    ax.set_title(title, fontsize=16, fontweight="bold", pad=10)

    ax.tick_params(axis="both", length=0, colors="black", labelleft=True)
    ax.tick_params(axis="x", pad=14)
    ax.tick_params(axis="y", pad=8)

    for spine in ax.spines.values():
        spine.set_linewidth(1.6)
        spine.set_color("black")

    handles = [Line2D([0], [0], color=C_AMB, linewidth=2.6, label="Ambient cured")]
    if (f["Curing_regime"] == "Heat").any():
        handles.append(Line2D([0], [0], color=C_HEAT, linewidth=2.6, label="Heat cured"))
    handles.append(Line2D([0], [0], color="black", linewidth=3.0,
                          label="TOPSIS recommended"))
    ax.legend(handles=handles, loc="upper right", fontsize=12.5, frameon=True)

fig3.tight_layout(w_pad=2.2)

F3 = os.path.join(SAVE_DIR, "MOO_fig3_parallel_coords.png")
fig3.savefig(F3, dpi=600, facecolor="white", bbox_inches="tight", pad_inches=0.06)
print("Saved:", F3)
plt.show()
files.download(F3)

In [ ]:
# =============================================================================
# Table Y — Optimized mixtures, representatives and grade references
# =============================================================================

import os
import numpy as np
import pandas as pd
from google.colab import files

assert "A" in globals() and "H" in globals(), "Run the Figure 1 cell first."

AGE = globals().get("AGE_FIXED", 28)
GRADE_TOL = 1.0


def grade_row(front, target, tol=GRADE_TOL):
    """Best compromise solution within tol of the target strength, ranked by the
    same TOPSIS closeness used for the headline recommendation. Selecting on
    minimum CO2 instead would always return the fly ash extreme, since fly ash
    emits 0.004 against 0.052 kg per kg for slag."""
    d = (front["CS_lower_MPa"] - target).abs()
    near = front[d <= tol]
    if len(near) == 0:
        return int(d.idxmin()), "nearest"
    return int(near["TOPSIS_closeness"].idxmax()), "max TOPSIS"


ROUND = {"FA (kg/m3)": 1, "GGBFS (kg/m3)": 1, "Coarse aggregate (kg/m3)": 1,
         "Fine aggregate (kg/m3)": 1, "Na2SiO3 solution (kg/m3)": 1,
         "NaOH molarity (M)": 1, "NaOH dry (kg/m3)": 2, "Superplasticizer (kg/m3)": 2,
         "Total water (kg/m3)": 1, "Curing temp (C)": 1, "Curing time (day)": 2,
         "Age (days)": 0, "Slag fraction of binder": 3, "CS lower bound (MPa)": 2,
         "CS point prediction (MPa)": 2, "Conformal margin (MPa)": 2,
         "CO2 (kg/m3)": 2, "Cost (USD/m3)": 2, "Energy (MJ/m3)": 1,
         "CO2 per MPa": 3, "Cost per MPa": 3, "TOPSIS closeness": 4,
         "NN distance": 4, "Variables near a search bound": 0}


def mix_record(r):
    return {
        "FA (kg/m3)": r["FA (kg/m3)"],
        "GGBFS (kg/m3)": r["GGBFS (kg/m3)"],
        "Coarse aggregate (kg/m3)": r["Coarse aggregate (kg/m3)"],
        "Fine aggregate (kg/m3)": r["Fine aggregate (kg in 1m3 mix)"],
        "Na2SiO3 solution (kg/m3)": r["Total Na2SiO3 (kg in 1m3 of mix)"],
        "NaOH molarity (M)": r["Concentration (M) NaOH"],
        "NaOH dry (kg/m3)": r["NaOH (Dry)"],
        "Superplasticizer (kg/m3)": r["Superplasticizer (kg in 1m3 mix)"],
        "Total water (kg/m3)": r["Total water (in solutions + additional) (kg in 1m3 mix)"],
        "Curing regime": r["Curing_regime"],
        "Curing temp (C)": r["Initial curing temp (C)"],
        "Curing time (day)": r["Initial curing time (day)"],
        "Age (days)": AGE,
        "Slag fraction of binder": r["Slag_fraction"],
        "CS lower bound (MPa)": r["CS_lower_MPa"],
        "CS point prediction (MPa)": r["CS_point_MPa"],
        "Conformal margin (MPa)": r["Margin_MPa"],
        "CO2 (kg/m3)": r["CO2_kg_m3"],
        "Cost (USD/m3)": r["Cost_USD_m3"],
        "Energy (MJ/m3)": r["Energy_MJ_m3"],
        "CO2 per MPa": r["CO2_per_MPa"],
        "Cost per MPa": r["Cost_per_MPa"],
        "TOPSIS closeness": r["TOPSIS_closeness"],
        "NN distance": r["NN_distance"],
        "Variables near a search bound": r["n_near_bounds_scored"],
        "Passes reliability screen": bool(r["conservative_pass"]),
    }


def build_scenario_table(front, rec_idx, grades, tag):
    specs = [("TOPSIS opt.", rec_idx),
             ("Max CS", int(front["CS_lower_MPa"].idxmax())),
             ("Min CO2/CS", int(front["CO2_per_MPa"].idxmin())),
             ("Min cost/CS", int(front["Cost_per_MPa"].idxmin()))]

    label_by_idx, order = {}, []
    for label, idx in specs:
        if idx in label_by_idx:
            label_by_idx[idx] += " + " + label
        else:
            label_by_idx[idx] = label
            order.append(idx)

    records, notes = {}, []
    for idx in order:
        records[label_by_idx[idx]] = mix_record(front.loc[idx])

    for t in grades:
        gidx, rule = grade_row(front, t)
        records[f"~{t} MPa"] = mix_record(front.loc[gidx])
        notes.append(f"{tag} {t} MPa selected by {rule}")

    out = pd.DataFrame(records)
    for param, nd in ROUND.items():
        if param in out.index:
            out.loc[param] = pd.to_numeric(out.loc[param], errors="coerce").round(nd)
    return out, notes


tab_A, notes_A = build_scenario_table(A, rec_idx_ambient, GRADES_AMB, "ambient")
tab_H, notes_H = build_scenario_table(H, rec_idx_heat, GRADES_HEAT, "heat")

table_y = pd.concat({"Ambient curing only": tab_A,
                     "Heat curing allowed": tab_H}, axis=1)

print("Grade rows are the highest TOPSIS closeness within "
      f"{GRADE_TOL:.1f} MPa of the target strength")
for n in notes_A + notes_H:
    print("  " + n)
print()
display(table_y)

# a quick check that the grade rows now move the way the band analysis says
print("\nGrade rows, slag fraction and cost against target strength")
for label, tab, grades in [("ambient", tab_A, GRADES_AMB), ("heat", tab_H, GRADES_HEAT)]:
    for t in grades:
        col = tab[f"~{t} MPa"]
        print(f"  {label:8s} {t:2d} MPa -> slag {float(col['Slag fraction of binder']):.3f} "
              f"| CO2 {float(col['CO2 (kg/m3)']):6.2f} | cost {float(col['Cost (USD/m3)']):6.2f} "
              f"| {col['Curing regime']}")

T1 = os.path.join(SAVE_DIR, "Table_Y_optimized_mixtures.csv")
table_y.to_csv(T1)
print("\nSaved:", T1)
files.download(T1)

In [ ]:
# =============================================================================
# Table Z — Fly ash to slag price sensitivity, both scenarios
# =============================================================================

import os
import numpy as np
import pandas as pd
from google.colab import files

for _n in ["price_ambient", "price_heat"]:
    assert _n in globals(), f"{_n} is missing. Re-run the snapshot lines."

# corr_slag_CS is dropped because it flips sign across ratios at three seeds and
# 800 generations, so it carries no signal
KEEP = ["n", "CS_max", "CO2_min", "Cost_min", "slag_frac_mean", "slag_frac_median",
        "pct_slag_gt25", "rec_CS", "rec_slag_frac", "rec_CO2", "rec_cost"]
HEAT_EXTRA = ["pct_heat", "rec_temp", "rec_regime"]

RENAME = {"n": "Front size", "CS_max": "Max CS lower (MPa)",
          "CO2_min": "Min CO2 (kg/m3)", "Cost_min": "Min cost (USD/m3)",
          "slag_frac_mean": "Front mean slag fraction",
          "slag_frac_median": "Front median slag fraction",
          "pct_slag_gt25": "Share with slag above 25% (%)",
          "pct_heat": "Share heat cured (%)",
          "rec_CS": "Rec. CS lower (MPa)", "rec_slag_frac": "Rec. slag fraction",
          "rec_CO2": "Rec. CO2 (kg/m3)", "rec_cost": "Rec. cost (USD/m3)",
          "rec_temp": "Rec. curing temp (C)", "rec_regime": "Rec. regime"}


def shape(price_df, extra):
    cols = KEEP + [c for c in extra if c in price_df.columns]
    out = price_df.set_index(["FA_slag_ratio", "FA_price"])[cols].copy()
    return out.rename(columns=RENAME)


pa = shape(price_ambient, [])
ph = shape(price_heat, HEAT_EXTRA)

price_table = pd.concat({"Ambient curing only": pa,
                         "Heat curing allowed": ph}, axis=1).round(3)
price_table.index.names = ["FA/slag price ratio", "FA price (USD/kg)"]

display(price_table)

print("\nFront mean slag fraction against price ratio")
for label, t in [("ambient", pa), ("heat", ph)]:
    v = t["Front mean slag fraction"].to_numpy(float)
    print(f"  {label:8s} {v.min():.3f} to {v.max():.3f}, "
          f"{'responds' if v.max() - v.min() > 0.05 else 'flat'}")

print("Recommended slag fraction against price ratio")
for label, t in [("ambient", pa), ("heat", ph)]:
    v = t["Rec. slag fraction"].to_numpy(float)
    print(f"  {label:8s} {v.min():.3f} to {v.max():.3f}, "
          f"{'responds' if v.max() - v.min() > 0.05 else 'insensitive'}")

T2 = os.path.join(SAVE_DIR, "Table_Z_price_sensitivity.csv")
price_table.to_csv(T2)
print("\nSaved:", T2)
files.download(T2)

# **OPC Benchmark**

In [ ]:
# =============================================================================
# Portland cement benchmark — mix design, impacts, comparison table and figure
# Self-contained. Nothing from earlier cells is required.
# =============================================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
from matplotlib.lines import Line2D

SAVE_DIR = "/content"
os.makedirs(SAVE_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# 1. Reference OPC mixtures, kg/m3
#    Common to all four: 25 mm nominal maximum aggregate, 75-100 mm slump,
#    non air entrained with 1.5% entrapped air, fine aggregate fineness modulus
#    2.6, coarse aggregate dry rodded unit weight 1602 kg/m3. Mixing water of
#    193 kg/m3 and a coarse aggregate bulk volume fraction of 0.69 are the
#    ACI 211.1 table values for those conditions, and fine aggregate closes the
#    mixture by absolute volume using specific gravities of 3.15, 2.68 and 2.64
#    for cement, coarse and fine aggregate.
#
#    30 and 40 MPa follow ACI 211.1 directly at the nominal grade, with water to
#    cementitious ratios of 0.54 and 0.42 read from its strength table. The
#    40 MPa mixture reproduces the one published by Alsalman et al. 2021.
#    ACI 211.1 does not tabulate strengths above 45 MPa, so the 50 and 70 MPa
#    mixtures extend the same relationship and apply a high range water reducer,
#    with water reductions of 12 and 20 percent inside the documented range.
# -----------------------------------------------------------------------------

opc_mixes = pd.DataFrame({
    "Grade (MPa)":              [30, 40, 50, 70],
    "Cement (kg/m3)":           [357.0, 460.0, 500.0, 550.0],
    "Water (kg/m3)":            [193.0, 193.0, 170.0, 154.0],
    "Coarse aggregate (kg/m3)": [1105.0, 1105.0, 1105.0, 1105.0],
    "Fine aggregate (kg/m3)":   [703.0, 622.0, 632.0, 625.0],
    "Superplasticizer (kg/m3)": [0.0, 0.0, 5.0, 8.3],
    "Design basis":             ["ACI 211.1", "ACI 211.1",
                                 "ACI 211.1 extended, HRWR", "ACI 211.1 extended, HRWR"],
}).set_index("Grade (MPa)")

opc_mixes["w/cm"] = (opc_mixes["Water (kg/m3)"] / opc_mixes["Cement (kg/m3)"]).round(3)

# -----------------------------------------------------------------------------
# 2. Factors. Emissions follow Torres et al. 2023 Table 2, who take them from
#    Alsalman et al. 2021. Energy follows Alsalman directly. Unit costs are US
#    2025 market values, with cement from USGS.
# -----------------------------------------------------------------------------

EF_CO2 = {"cement": 0.84, "aggregate": 0.0048, "sp": 1.88, "water": 0.0}
EF_ENERGY = {"cement": 4.53, "aggregate": 0.083, "sp": 29.1, "water": 0.0}
EF_COST = {"cement": 0.16, "coarse": 0.014, "fine": 0.010, "sp": 1.00, "water": 0.001}


def opc_impacts(row):
    cem = row["Cement (kg/m3)"]
    coarse = row["Coarse aggregate (kg/m3)"]
    fine = row["Fine aggregate (kg/m3)"]
    sp = row["Superplasticizer (kg/m3)"]
    water = row["Water (kg/m3)"]
    agg = coarse + fine

    co2 = EF_CO2["cement"] * cem + EF_CO2["aggregate"] * agg + EF_CO2["sp"] * sp
    energy = EF_ENERGY["cement"] * cem + EF_ENERGY["aggregate"] * agg + EF_ENERGY["sp"] * sp
    cost = (EF_COST["cement"] * cem + EF_COST["coarse"] * coarse + EF_COST["fine"] * fine
            + EF_COST["sp"] * sp + EF_COST["water"] * water)
    return pd.Series({"CO2 (kg/m3)": co2, "Cost (USD/m3)": cost, "Energy (MJ/m3)": energy})


opc = opc_mixes.join(opc_mixes.apply(opc_impacts, axis=1))

# the 40 MPa mixture is published together with its impacts, so it doubles as a
# check on the factor set and on the arithmetic
chk = opc.loc[40]
print("Check against Alsalman et al. 2021, 40 MPa OPCC")
print(f"  CO2    computed {chk['CO2 (kg/m3)']:7.1f} | published   395 kg/m3")
print(f"  Energy computed {chk['Energy (MJ/m3)']:7.1f} | published  2227 MJ/m3")
print()

# -----------------------------------------------------------------------------
# 3. Optimized AAC grade references, taken from the Pareto front tables.
#    Strength is the conformal lower bound, not a point prediction.
# -----------------------------------------------------------------------------

aac = pd.DataFrame([
    {"Grade (MPa)": 30, "Scenario": "Ambient curing only", "CS lower (MPa)": 30.58,
     "CO2 (kg/m3)": 39.88, "Cost (USD/m3)": 61.59, "Energy (MJ/m3)": 552.8,
     "Slag fraction": 0.016, "Curing regime": "Ambient"},
    {"Grade (MPa)": 40, "Scenario": "Ambient curing only", "CS lower (MPa)": 40.92,
     "CO2 (kg/m3)": 43.70, "Cost (USD/m3)": 56.90, "Energy (MJ/m3)": 618.5,
     "Slag fraction": 0.280, "Curing regime": "Ambient"},
    {"Grade (MPa)": 50, "Scenario": "Ambient curing only", "CS lower (MPa)": 50.67,
     "CO2 (kg/m3)": 47.43, "Cost (USD/m3)": 62.51, "Energy (MJ/m3)": 680.3,
     "Slag fraction": 0.400, "Curing regime": "Ambient"},
    {"Grade (MPa)": 30, "Scenario": "Heat curing allowed", "CS lower (MPa)": 30.99,
     "CO2 (kg/m3)": 42.17, "Cost (USD/m3)": 58.50, "Energy (MJ/m3)": 592.1,
     "Slag fraction": 0.179, "Curing regime": "Ambient"},
    {"Grade (MPa)": 50, "Scenario": "Heat curing allowed", "CS lower (MPa)": 50.75,
     "CO2 (kg/m3)": 47.34, "Cost (USD/m3)": 62.62, "Energy (MJ/m3)": 678.8,
     "Slag fraction": 0.395, "Curing regime": "Ambient"},
    {"Grade (MPa)": 70, "Scenario": "Heat curing allowed", "CS lower (MPa)": 69.03,
     "CO2 (kg/m3)": 56.59, "Cost (USD/m3)": 86.60, "Energy (MJ/m3)": 726.3,
     "Slag fraction": 0.052, "Curing regime": "Heat"},
])

# -----------------------------------------------------------------------------
# 4. Comparison table
# -----------------------------------------------------------------------------

METRICS = ["CO2 (kg/m3)", "Cost (USD/m3)", "Energy (MJ/m3)"]

rows = []
for _, r in aac.iterrows():
    g = r["Grade (MPa)"]
    ref = opc.loc[g]
    rec = {"Grade (MPa)": g, "Scenario": r["Scenario"],
           "Curing regime": r["Curing regime"], "CS lower (MPa)": r["CS lower (MPa)"],
           "Slag fraction": r["Slag fraction"]}
    for m in METRICS:
        rec[f"OPC {m}"] = ref[m]
        rec[f"AAC {m}"] = r[m]
        rec[f"Reduction {m.split(' ')[0]} (%)"] = 100 * (ref[m] - r[m]) / ref[m]
    rows.append(rec)

comparison = pd.DataFrame(rows).sort_values(["Grade (MPa)", "Scenario"]).reset_index(drop=True)

print("Reference OPC mixtures")
display(opc.round(2))
print("\nOPC benchmark against the optimized mixtures")
display(comparison.round(2))

print("\nReduction relative to OPC concrete at matched grade")
for _, r in comparison.iterrows():
    print(f"  {int(r['Grade (MPa)']):2d} MPa, {r['Scenario']:20s} -> "
          f"CO2 {r['Reduction CO2 (%)']:5.1f}% | cost {r['Reduction Cost (%)']:5.1f}% "
          f"| energy {r['Reduction Energy (%)']:5.1f}%")

# -----------------------------------------------------------------------------
# 5. Figure
# -----------------------------------------------------------------------------

GRADES = [30, 40, 50, 70]

# bar fills are light, and the reduction labels use a darker shade of the same
# hue so they stay legible against white
SERIES = [("Portland cement concrete", "#9E9E9E", "#5A5A5A"),
          ("Ambient curing only", "#7A9BE8", "#12439C"),
          ("Heat curing allowed", "#E8736B", "#A81C13")]

PANELS = [("CO2 (kg/m3)", "CO$_2$ (kg/m$^3$)", "(a) Carbon emissions"),
          ("Cost (USD/m3)", "Cost (USD/m$^3$)", "(b) Material cost"),
          ("Energy (MJ/m3)", "Energy (MJ/m$^3$)", "(c) Embodied energy")]


def value_for(series, metric, grade):
    """Height of one bar. Returns nan where that scenario has no mixture at
    this grade, which leaves a visible gap rather than a misleading zero."""
    if series == "Portland cement concrete":
        return float(opc.loc[grade, metric])
    sub = aac[(aac["Scenario"] == series) & (aac["Grade (MPa)"] == grade)]
    return float(sub[metric].iloc[0]) if len(sub) else np.nan


x = np.arange(len(GRADES))
width = 0.26

fig, axes = plt.subplots(1, 3, figsize=(18.0, 6.6))

for ax, (metric, ylabel, title) in zip(axes, PANELS):
    top = np.nanmax([value_for(s, metric, g) for s, _, _ in SERIES for g in GRADES])
    ax.set_ylim(0, top * 1.30)

    for k, (series, fill, label_colour) in enumerate(SERIES):
        vals = [value_for(series, metric, g) for g in GRADES]
        pos = x + (k - 1) * width
        ax.bar(pos, vals, width, color=fill, edgecolor="black", linewidth=0.9,
               zorder=3)

        # percentage reduction printed vertically above each AAC bar
        if series != "Portland cement concrete":
            for xi, g, v in zip(pos, GRADES, vals):
                if np.isnan(v):
                    continue
                ref = float(opc.loc[g, metric])
                ax.text(xi, v + 0.015 * top, f"-{100 * (ref - v) / ref:.0f}%",
                        ha="center", va="bottom", rotation=90, fontsize=14,
                        color=label_colour, zorder=5)

    ax.set_xticks(x)
    ax.set_xticklabels([f"{g} MPa" for g in GRADES], fontsize=16)
    ax.set_ylabel(ylabel, fontsize=16, labelpad=9)
    ax.set_title(title, fontsize=18, fontweight="bold", pad=10)
    ax.tick_params(axis="y", labelsize=13)

    # horizontal reference lines only, with a lighter minor set between them.
    # no vertical grid, since the categories are already separated by position
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.grid(axis="y", which="major", color="#BFBFBF", linestyle="-",
            linewidth=0.6, alpha=0.8, zorder=0)
    ax.grid(axis="y", which="minor", color="#DCDCDC", linestyle="-",
            linewidth=0.5, alpha=0.8, zorder=0)
    ax.grid(axis="x", which="both", visible=False)
    ax.set_axisbelow(True)

    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.05)

handles = [Line2D([0], [0], marker="s", color="w", markerfacecolor=f,
                  markeredgecolor="k", markersize=13, label=s)
           for s, f, _ in SERIES]
fig.legend(handles=handles, loc="upper center", ncol=3, fontsize=15,
           frameon=False, bbox_to_anchor=(0.5, 1.04))

fig.tight_layout(w_pad=2.6, rect=[0, 0, 1, 0.95])

FIG = os.path.join(SAVE_DIR, "OPC_benchmark_comparison.png")
fig.savefig(FIG, dpi=600, facecolor="white", bbox_inches="tight", pad_inches=0.05)
print("\nSaved:", FIG)
plt.show()

TAB = os.path.join(SAVE_DIR, "Table_OPC_benchmark.csv")
comparison.to_csv(TAB, index=False)
MIX = os.path.join(SAVE_DIR, "Table_OPC_mix_designs.csv")
opc.to_csv(MIX)
print("Saved:", TAB)
print("Saved:", MIX)

try:
    from google.colab import files
    files.download(FIG)
    files.download(TAB)
    files.download(MIX)
except Exception:
    pass